South Bay Indoor Air Quality Study — Main Analysis Script
 Figures 1–6
 =============================================================================
 HOW TO USE:
   1. Update file paths in the CONFIG section below
   2. Run all cells top-to-bottom
   3. To change date range, edit PLOT_START / PLOT_END in CONFIG
   4. To add/remove sensors, edit AG_LOCATIONS or QUANTAQ_FILES in CONFIG

In [ ]:
# ── Cell 2: Imports ───────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# @title
# ── Cell 3: CONFIG — edit all settings here ───────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────
# DATE RANGE (change here to update all figures)
PLOT_START = '2026-04-03'
PLOT_END   = '2026-05-14'

# FILE PATHS — update to your Google Drive paths
BASE = '/content/drive/MyDrive/Colab Notebooks/capstone_code/Extended data set'

AG_FILE        = f'{BASE}/export (21).csv'
QUANTAQ_FILES  = {
    'QuantAQ ML':    f'{BASE}/MOD-00370-6c7754515cd148d0a3bf3eeedba87d5c.csv',
    'QuantAQ Coast':      f'{BASE}/MOD-00367-1c28104a02aa4539b2f96d90bc6ad6a5.csv',
    'QuantAQ IB City Hall': f'{BASE}/MOD-00369-b950720ea2c44a7fa7afde050f5ed26d.csv',
    'QuantAQ TR Estuary': f'{BASE}/MOD-00365-097157fe8eb6491c87812cb84da2e94b.csv',
    'QuantAQ SY':         f'{BASE}/MOD-00366-5b0e9b2bc27641b78c4662cc05817f6b.csv',
    'QuantAQ SIO':      f'{BASE}/MOD-00364-a02bf9e1bc9041548768478c40eff5aa.csv',
    'QuantAQ CABR':     f'{BASE}/MOD-00371-6da146653d0740789e0af15319122820.csv',
}
H2S_TRAILER_FILE = f'{BASE}/H2S - trailer-data-2026-06-23 10_42_56.csv'
H2S_COAST_FILE   = f'{BASE}/H2S - coast-data-2026-06-23 10_43_35.csv'
RIVER_FILE       = f'{BASE}/BulkExport-11013101-20260623234323.csv'
MLH_FILE         = f'{BASE}/mlh_solar_extended.csv'

# SENSOR NAMES — full descriptive names used throughout
def dms(d, m, s, direction):
    v = d + m/60 + s/3600
    return -v if direction in ('S','W') else v

AG_LOCATIONS = {
    'AG-3 (SY)':             (dms(32,34,20.73,'N'), dms(117,1,10.80,'W')),
    'AG-6 (IB Classroom)':   (dms(32,34,30.00,'N'), dms(117,6,45.00,'W')),
    'AG-7 (IB)':             (dms(32,34,40.91,'N'), dms(117,6,27.72,'W')),
    'AG-9 (SY)':             (dms(32,33, 0.58,'N'), dms(117,2,51.40,'W')),
    'AG-11 (IB Classroom)':  (dms(32,34,42.69,'N'), dms(117,7,22.10,'W')),
    'AG-12 (IB)':            (dms(32,34,50.62,'N'), dms(117,7,56.40,'W')),
    'AG-13 (TR Estuary)':    (dms(32,34,29.29,'N'), dms(117,7,34.68,'W')),
    'AG-15 (IB)':            (dms(32,34,39.31,'N'), dms(117,5,44.80,'W')),
    'AG-17 (IB City Hall)':  (dms(32,34,34.51,'N'), dms(117,6,57.44,'W')),
    'AG-19 (ML)':    (dms(32,33,55.23,'N'), dms(117,5,25.32,'W')),
}
ag_locs = list(AG_LOCATIONS.keys())

# TRAILER LOCATION (reference for distance ordering)
TRAILER_COORDS = AG_LOCATIONS['AG-19 (ML)']

# EVENT THRESHOLDS
# H2S: 1-hour rolling mean >= 30 ppb (CARB 1-hour ambient standard)
# evaluated using LOCAL PDT day boundaries (UTC-7) to avoid nocturnal spike attribution errors at UTC midnight
H2S_HOURLY_THRESH = 30    # ppb — max hourly average per local day

# PM1: dual criterion — both must be met:
#   1. 1-hour rolling mean exceeds study-period top quartile
#   2. Daily mean PM2.5 >= 9 µg/m³
# Top quartile computed dynamically in Cell 8 from study period data
PM25_DAILY_THRESH = 9.0   # µg/m³ — daily mean PM2.5

# AG-13 data availability
AG13_START = '2026-04-13'   # AG-13 data not available before this date

# CASE STUDY DAYS — representative days for three-panel figures
CASE_STUDY_1   = '2026-04-05'
CASE_STUDY_2   = '2026-04-08'
CASE_STUDY_3   = '2026-04-18'
CASE_STUDY_4   = '2026-04-024'
CASE_STUDY_5   = '2026-04-30'
#CASE_STUDY_6   = '2026-05-11'
CASE_STUDY_CLEAN = '2026-04-13'   # Clean reference day

# FIGURE STYLE
FONT_FAMILY  = 'sans-serif'
FONT_SIZE    = 18
FIG_DPI      = 300
SAVE_DPI     = 300

# COLOR PALETTE
C_H2S  = 'red'
C_PM1  = 'green'
C_RH   = 'blue'
C_MLH  = 'grey'
C_RIV  = 'teal'
C_TEMP = 'darkorange'
C_SOL  = 'royalblue'

# Shading colors for event days
SHADE_H2S    = 'red'
SHADE_PM1    = 'green'
SHADE_BOTH   = 'purple'
SHADE_CLEAN  = 'skyblue'
SHADE_ALPHA  = 0.15

plt.rcParams.update({
    'font.family':    'sans-serif',
    'font.size':       14,
    'axes.linewidth':  0.8,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'xtick.direction':   'out',
    'ytick.direction':   'out',
    'figure.dpi':        FIG_DPI,
    'figure.facecolor':  'white',
    'axes.facecolor':    'white',
    'grid.alpha':        0.2,
    'grid.linewidth':    0.4,
})

print("CONFIG loaded.")
print(f"  Date range : {PLOT_START} → {PLOT_END}")
print(f"  AG sensors : {len(ag_locs)}")


In [ ]:
# ── QuantAQ Sensor Locations (approximate, placeholder coordinates) ───────────
# These coordinates are approximate and should be verified for accuracy.
QUANTAQ_LOCATIONS = {
    'QuantAQ ML':      (dms(32,33,55.23,'N'), dms(117,5,25.32,'W')), # Near AG-19 (Mobile Lab)
    'QuantAQ Coast':        (dms(32,34,0.00,'N'),  dms(117,8,0.00,'W') ),  # Approximate coastal location
    'QuantAQ IB City Hall': (dms(32,34,34.51,'N'), dms(117,6,57.44,'W')), # Near AG-17 (IB City Hall)
    'QuantAQ TR Estuary':   (dms(32,34,29.29,'N'), dms(117,7,34.68,'W')), # Near AG-13 (TR Estuary)
    'QuantAQ SY':           (dms(32,34,20.73,'N'), dms(117,1,10.80,'W')), # Near AG-3 (SY)
}

print("QuantAQ locations defined.")

In [ ]:
# @title
# ── Cell 4: DATA LOADING HELPERS ──────────────────────────────────────────────

AG_MEASURE_COLS = [
    'PM1 (μg/m³)',
    'PM2.5 (μg/m³) corrected',
    'PM10 (μg/m³)',
    'CO2 (ppm) corrected',
    'NOX index',
    '0.3μm particle count',
]
AG_VAR_MAP = {
    'PM1 (μg/m³)':             'pm1',
    'PM2.5 (μg/m³) corrected': 'pm25',
    'PM10 (μg/m³)':            'pm10',
    'CO2 (ppm) corrected':     'co2',
    'NOX index':               'nox',
    '0.3μm particle count':    'pm03_count',
}

def build_ag_dataset(filepath, locations, start, end):
    """Load AirGradient export, rename locations, return xr.Dataset (time × location) in PDT."""
    df = pd.read_csv(filepath)
    df['UTC Date/Time'] = pd.to_datetime(df['UTC Date/Time'])
    df['UTC Date/Time'] = df['UTC Date/Time'] - pd.Timedelta(hours=7)  # ← shift to PDT here
    df = df.rename(columns={'UTC Date/Time': 'PDT Date/Time'})
    short_to_full = {name.split(' ')[0]: name for name in locations}
    df['Location Name'] = df['Location Name'].map(short_to_full).fillna(df['Location Name'])
    missing = set(locations) - set(df['Location Name'].unique())
    if missing:
        print(f"  Warning — not found in AG data: {missing}")
    df = df[df['Location Name'].isin(locations) &
            df['PDT Date/Time'].between(start, end)]
    pivot = df.pivot_table(
        index='PDT Date/Time', columns='Location Name', values=AG_MEASURE_COLS
    ).loc[:, (AG_MEASURE_COLS, locations)]
    time = pivot.index.tz_localize(None).rename('time')
    return xr.Dataset(
        data_vars={AG_VAR_MAP[col]: (('time','location'), pivot[col].values)
                   for col in AG_MEASURE_COLS},
        coords={
            'time':     time,
            'location': np.array(locations),
            'hour':     ('time', time.hour.values),
        }
    )

def build_quantaq_dataset(file_dict, start, end):
    """Load QuantAQ CSVs, resample to 5-min, return xr.Dataset (time × location) in PDT."""
    QAQ_COLS = ['pm1','pm25','rh','temp','ws','wd']
    QAQ_MAP  = {'pm1':'pm1','pm25':'pm25','rh':'rh','temp':'temp',
                'ws':'wind_speed','wd':'wind_dir'}
    dfs = {}
    for loc, path in file_dict.items():
        df = pd.read_csv(path)
        df['timestamp'] = pd.to_datetime(df['timestamp']) - pd.Timedelta(hours=7)  # ← shift to PDT
        df = df[(df['timestamp'].dt.date >= pd.to_datetime(start).date()) &
                (df['timestamp'].dt.date <= pd.to_datetime(end).date())]
        df = df.sort_values('timestamp').set_index('timestamp')
        if 'pm1' not in df.columns:
            df['pm1'] = df['pm1_env']
        if 'pm25' not in df.columns:
            df['pm25'] = df['pm25_env']

        # Add missing 'ws' and 'wd' columns with NaN values if they don't exist
        if 'ws' not in df.columns:
            df['ws'] = np.nan
        if 'wd' not in df.columns:
            df['wd'] = np.nan

        df = df[QAQ_COLS]
        df.index = df.index.tz_localize(None).rename('time')
        df_r = df.drop(columns=['wd']).resample('5min').mean()
        wd_r = df['wd']
        df_r['wd'] = np.rad2deg(np.arctan2(
            np.sin(np.deg2rad(wd_r)).resample('5min').mean(),
            np.cos(np.deg2rad(wd_r)).resample('5min').mean()
        )) % 360
        dfs[loc] = df_r
    locs = list(file_dict.keys())
    combined = pd.concat(dfs, axis=1)
    time = combined.index
    return xr.Dataset(
        data_vars={QAQ_MAP[col]: (('time','location'),
            np.stack([combined[loc][col].values for loc in locs], axis=1))
            for col in QAQ_COLS},
        coords={'time': time, 'location': np.array(locs),
                'hour': ('time', time.hour.values)}
    )


def load_h2s(filepath, col_name):
    """Load H2S CSV, strip units, resample to 5-min."""
    df = pd.read_csv(filepath)
    df['measured_at'] = pd.to_datetime(df['measured_at'])
    df[col_name] = df[col_name].astype(str).str.replace(' ppb','').astype(float)
    return df.set_index('measured_at')[[col_name]].resample('5min').mean()

def load_river(filepath):
    """Load river stage CSV (UTC-8/PST) → resample to 5-min → convert to PDT."""
    df = pd.read_csv(filepath, skiprows=5, header=None)
    df.columns = ['datetime_utc', 'river_stage_m']
    df = df[pd.to_datetime(df['datetime_utc'], errors='coerce').notna()]
    df['datetime_utc'] = pd.to_datetime(df['datetime_utc'])
    df['river_stage_m'] = pd.to_numeric(df['river_stage_m'], errors='coerce')
    df = (df.set_index('datetime_utc')
            .resample('5min').mean()
            .ffill())
    # Timestamps are PST (UTC-8) — convert to PDT (UTC-7) by adding 1 hour
    df.index = df.index + pd.Timedelta(hours=1)
    df['river_stage_lag6h'] = df['river_stage_m'].shift(periods=int(6*60/5))
    print(f"River loaded: {len(df)} rows, index sample: {df.index[:2].tolist()}")
    return df[['river_stage_m', 'river_stage_lag6h']]

def load_mlh(filepath):
    """Load MLH/solar CSV (UTC) → resample to 5-min → convert to PDT."""
    df = pd.read_csv(filepath)
    df['datetime_utc'] = pd.to_datetime(df['datetime_utc'])
    for c in ['mlh_m', 'shortwave_radiation_wm2']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df = (df[['datetime_utc', 'mlh_m', 'shortwave_radiation_wm2']]
            .set_index('datetime_utc')
            .rename(columns={'mlh_m': 'mlh', 'shortwave_radiation_wm2': 'shortwave'})
            .resample('5min').mean()
            .ffill())
    # ERA5 timestamps are UTC — convert to PDT (UTC-7)
    df.index = df.index - pd.Timedelta(hours=7)
    print(f"MLH loaded: {len(df)} rows, index sample: {df.index[:2].tolist()}")
    return df

In [ ]:
# ── Cell 5: LOAD ALL DATA ─────────────────────────────────────────────────────
print("Loading AG data...")
ds_ag = build_ag_dataset(AG_FILE, ag_locs, PLOT_START, PLOT_END)

print("Loading QuantAQ data...")
ds_aq = build_quantaq_dataset(QUANTAQ_FILES, PLOT_START, PLOT_END)

print("Loading H2S data...")
h2s_trailer = load_h2s(H2S_TRAILER_FILE, 'trailer').rename(columns={'trailer': 'h2s'})
h2s_coast   = load_h2s(H2S_COAST_FILE,   'coast').rename(columns={'coast': 'h2s_coast'})

print("Loading river stage and MLH data...")

river = load_river(RIVER_FILE)
mlh   = load_mlh(MLH_FILE)

print("Done loading.\n")
print("AG  dataset:", ds_ag)
print("QuantAQ dataset:", ds_aq)

In [ ]:
def drop_full_dropouts(ds):
    """Drop locations where every variable is 100% NaN across the whole time range (sensor produced zero data)."""
    dropped = [
        loc for loc in ds.location.values
        if all(ds[v].sel(location=loc).isnull().all().item() for v in ds.data_vars)
    ]
    if dropped:
        print(f"  Dropped full dropouts: {dropped}")
    kept = [loc for loc in ds.location.values if loc not in dropped]
    return ds.sel(location=kept)


def drop_sparse_locations(ds, threshold=0.5, exclude_from_dropping=None):
    if exclude_from_dropping is None:
        exclude_from_dropping = []

    candidate_locations = [loc for loc in ds.location.values if loc not in exclude_from_dropping]

    kept_candidates = [loc for loc in candidate_locations
                       if max(ds[v].sel(location=loc).isnull().mean().item()
                              for v in ds.data_vars) < threshold]

    final_kept_locations = sorted(list(set(exclude_from_dropping) | set(kept_candidates)))

    all_original_locations = set(ds.location.values)
    actually_dropped = all_original_locations - set(final_kept_locations)

    if actually_dropped:
        print(f"  Dropped sparse: {actually_dropped}")

    return ds.sel(location=final_kept_locations)


ds_ag_clean = ds_ag.pipe(drop_full_dropouts).pipe(drop_sparse_locations)
ds_aq_clean = ds_aq.pipe(drop_full_dropouts).pipe(drop_sparse_locations, exclude_from_dropping=['QuantAQ CABR'])

In [ ]:
# ── PM1 AMBIENT CALIBRATION CORRECTION ───────────────────────────────────────
# Network-mean SLR derived from overnight ambient OPS co-location (Aug 19–20, 2026)
# Reference aerosol: indoor ambient room air (density = 1.35 g/cm³, coastal CA)
# Equation: OPS_PM1 = 0.2566 × AG_raw_PM1 + 1.2904
# Applied as: PM1_corrected = 0.2566 × PM1_raw + 1.2904
# Slope CV across 14 sensors = 9.3% → single network-mean equation defensible
# (Malings et al. 2019; Duvall et al. 2021)
#
# Note: PM2.5 uses the EPA Barkjohn (2021) correction already applied by
# AirGradient firmware (PM2.5 (μg/m³) corrected column). No additional
# PM2.5 correction is applied here.

#density=1.35
#PM1_CAL_SLOPE     = 0.2566
#PM1_CAL_INTERCEPT = 1.2904

#density=1
PM1_CAL_SLOPE     = 0.1910
PM1_CAL_INTERCEPT = 1.1234

# Apply to ds_ag_clean in-place (overwrites pm1 with corrected values)
# Clip at 0 to avoid physically impossible negative concentrations
APPLY_PM1_CORRECTION = True  # set False to use raw firmware PM1

if APPLY_PM1_CORRECTION:
    ds_ag_clean['pm1'] = (
        ds_ag_clean['pm1'] * PM1_CAL_SLOPE + PM1_CAL_INTERCEPT
    ).clip(min=0)
    print("PM1 correction APPLIED")
else:
    print("PM1 correction NOT applied — using raw firmware values")

print(f"PM1 correction applied: PM1_corrected = {PM1_CAL_SLOPE} × PM1_raw + {PM1_CAL_INTERCEPT}")
print(f"Network-mean PM1 after correction: {float(ds_ag_clean['pm1'].mean()):.3f} µg/m³")
print(f"Note: PM2.5 already corrected by AirGradient firmware (EPA Barkjohn 2021)")

In [ ]:
# Recalibrate cooking filter thresholds to match the corrected PM1 scale
PM1_CAL_SLOPE     = 0.1910
PM1_CAL_INTERCEPT = 1.1234

COOK_CFG_RAW = {
    'AG-3':  ( 5.0, 407, 50, False),
    'AG-7':  ( 8.0, 408, 60, False),
    'AG-9':  (12.0, 432, 60, False),
    'AG-12': ( 4.0, 416, 60, True ),
    'AG-13': ( 2.0, 410, 50, True ),
    'AG-15': (10.0, 434, 70, False),
}

# Rebuild COOK_CFG with thresholds transformed to corrected-scale units
COOK_CFG = {
    sensor: (round(pm1_thresh * PM1_CAL_SLOPE + PM1_CAL_INTERCEPT, 3), co2_bl, co2_thresh, co2_req)
    for sensor, (pm1_thresh, co2_bl, co2_thresh, co2_req) in COOK_CFG_RAW.items()
}

print("Recalibrated COOK_CFG (corrected-scale thresholds):")
for k, v in COOK_CFG.items():
    print(f"  {k}: pm1_thresh = {v[0]} (was {COOK_CFG_RAW[k][0]} in raw scale)")

COOKING FILTER

In [ ]:
# @title
# ── COOKING FILTER (PM1 + PM2.5, PM10-ratio layer removed) ──────────────────
# Combined replacement for the original PM1 ("Cell 7: COOKING FILTER") and
# PM2.5 ("PM2.5 COOKING FILTER") cells. Run this AFTER the PM1 calibration
# correction cell (which defines PM1_CAL_SLOPE / PM1_CAL_INTERCEPT and
# overwrites ds_ag_clean['pm1'] with corrected values).
#
# Changes from the original two cells:
#   1. PM10 ratio layer (former L4) removed from BOTH the PM1 and PM2.5
#      filters — indoor/outdoor decoupling ratio is renumbered L4.
#   2. PM1 thresholds (COOK_CFG) are rescaled through the calibration
#      equation so they trigger correctly against corrected-scale PM1 data.
#   3. PM2.5 thresholds (COOK_CFG_PM25) are still derived as ~1.5x the
#      ORIGINAL raw PM1 thresholds (COOK_CFG_RAW), since PM2.5 uses a
#      separate firmware correction and was never compressed by the
#      PM1-specific calibration.

COOKING_HOURS_UTC = set(list(range(13,17)) + list(range(18,21)) + list(range(0,5)))

COOKING_HOURS_LOCAL_BY_SENSOR = {
    'AG-3 (SY)':            {11,12,15,16,17,18},   # 11am-1pm, 3-7pm
    'AG-7 (IB)':            {9,10,17,18},           # 9-11am, 5-7pm
    'AG-9 (SY)':            {7,8,17,18},            # 7-9am, 5-7pm
    'AG-11 (IB Classroom)': {13,14},                # 1-3pm
    'AG-12 (IB)':           {17,18},                # 5-7pm
    'AG-15 (IB)':           {7,8,11,12,17,18},      # 7-9am, 11am-1pm, 5-7pm
    'AG-17 (IB City Hall)': {11,12},                # 11am-1pm
    'AG-19 (ML)':           set(),                  # no cooking
}

# ── Original raw-scale PM1 thresholds (pre-calibration) — kept as the
#    reference basis for deriving PM2.5 thresholds below ──────────────────
COOK_CFG_RAW = {
    # sensor_short: (pm1_thresh, co2_baseline, co2_thresh, co2_required)
    'AG-3':  ( 5.0, 407, 50, False),
    'AG-7':  ( 8.0, 408, 60, False),
    'AG-9':  (12.0, 432, 60, False),
    'AG-12': ( 4.0, 416, 60, True ),
    'AG-13': ( 2.0, 410, 50, True ),
    'AG-15': (10.0, 434, 70, False),
}

# ── PM1 thresholds rescaled to the corrected PM1 scale ───────────────────
# Requires PM1_CAL_SLOPE / PM1_CAL_INTERCEPT already defined by the
# calibration cell (currently 0.1910 / 1.1234).
COOK_CFG = {
    sensor: (round(pm1_thresh * PM1_CAL_SLOPE + PM1_CAL_INTERCEPT, 3), co2_bl, co2_thresh, co2_req)
    for sensor, (pm1_thresh, co2_bl, co2_thresh, co2_req) in COOK_CFG_RAW.items()
}

print("Recalibrated COOK_CFG (corrected-scale PM1 thresholds):")
for k, v in COOK_CFG.items():
    print(f"  {k}: pm1_thresh = {v[0]} (was {COOK_CFG_RAW[k][0]} raw)")

# ── PM2.5 thresholds — 1.5x the ORIGINAL raw PM1 thresholds, unaffected
#    by the PM1-specific calibration ──────────────────────────────────────
COOK_CFG_PM25 = {
    sensor: (round(pm1_thresh * 1.5, 3), co2_bl, co2_thresh, co2_req)
    for sensor, (pm1_thresh, co2_bl, co2_thresh, co2_req) in COOK_CFG_RAW.items()
}

print("\nCOOK_CFG_PM25 (1.5x raw PM1 thresholds, PM2.5 scale unaffected by calibration):")
for k, v in COOK_CFG_PM25.items():
    print(f"  {k}: pm25_thresh = {v[0]}")


def apply_cooking_filter(ds_ag, ds_aq_clean):
    """Return a copy of ds_ag with two filtered PM1 variables:
       pm1          → original raw values (unchanged)
       pm1_filtered → cooking events (High + Medium) set to NaN  [conservative]
       pm1_strict   → all flagged events (High + Medium + Low) set to NaN  [strict]
    """
    out_pm1 = (ds_aq_clean['pm1']
               .sel(location='QuantAQ ML')
               .to_pandas()
               .reindex(pd.DatetimeIndex(ds_ag.time.values), method='nearest',
                        tolerance=pd.Timedelta('10min')))

    pm1_conserv = ds_ag['pm1'].values.copy().astype(float)
    pm1_strict  = ds_ag['pm1'].values.copy().astype(float)

    for li, loc in enumerate(ds_ag.location.values):
        short = loc.split(' ')[0]
        if short not in COOK_CFG:
            continue
        pm1_thresh, co2_bl, co2_thresh, co2_req = COOK_CFG[short]

        co2_s = ds_ag['co2'].sel(location=loc).to_pandas()

        for ti, t in enumerate(ds_ag.time.values):
            pm1_val = pm1_conserv[ti, li]
            if np.isnan(pm1_val) or pm1_val == 0:
                continue
            t_pd       = pd.Timestamp(t)
            local_hour = t_pd.hour
            out_val    = out_pm1.get(t_pd, np.nan)
            co2_val    = co2_s.get(t_pd, np.nan)

            cooking_hrs = COOKING_HOURS_LOCAL_BY_SENSOR.get(loc, set())
            L1 = local_hour in cooking_hrs
            L2 = pm1_val >= pm1_thresh
            L3 = (not np.isnan(co2_val)) and (co2_val - co2_bl) >= co2_thresh
            L4 = (not np.isnan(out_val) and pm1_val >= pm1_thresh
                  and out_val < 10.0 and (pm1_val / (out_val + 0.5)) >= 2.0)

            if not L2:
                continue
            if co2_req and not L3:
                continue

            n = L2 + L3 + L4
            is_high_med = n >= 2 or (L2 and L1)
            is_low      = L2 and not is_high_med

            if is_high_med:
                pm1_conserv[ti, li] = np.nan
                pm1_strict[ti, li]  = np.nan
            elif is_low:
                pm1_strict[ti, li]  = np.nan

    ds_filt = ds_ag.copy()
    ds_filt['pm1'].values[:]          = ds_ag['pm1'].values
    ds_filt['pm1_filtered']           = ds_filt['pm1'].copy()
    ds_filt['pm1_filtered'].values[:] = pm1_conserv
    ds_filt['pm1_strict']             = ds_filt['pm1'].copy()
    ds_filt['pm1_strict'].values[:]   = pm1_strict

    return ds_filt


def apply_cooking_filter_pm25(ds_ag, ds_aq_clean):
    """Return a copy of ds_ag with two filtered PM2.5 variables:
       pm25          → original raw values (unchanged)
       pm25_filtered → cooking events (High + Medium) set to NaN  [conservative]
       pm25_strict   → all flagged events (High + Medium + Low) set to NaN  [strict]
    """
    out_pm25 = (ds_aq_clean['pm25']
               .sel(location='QuantAQ ML')
               .to_pandas()
               .reindex(pd.DatetimeIndex(ds_ag.time.values), method='nearest',
                        tolerance=pd.Timedelta('10min')))

    pm25_conserv = ds_ag['pm25'].values.copy().astype(float)
    pm25_strict  = ds_ag['pm25'].values.copy().astype(float)

    for li, loc in enumerate(ds_ag.location.values):
        short = loc.split(' ')[0]
        if short not in COOK_CFG_PM25:
            continue
        pm25_thresh, co2_bl, co2_thresh, co2_req = COOK_CFG_PM25[short]

        co2_s = ds_ag['co2'].sel(location=loc).to_pandas()

        for ti, t in enumerate(ds_ag.time.values):
            pm25_val = pm25_conserv[ti, li]
            if np.isnan(pm25_val) or pm25_val == 0:
                continue
            t_pd       = pd.Timestamp(t)
            local_hour = t_pd.hour
            out_val    = out_pm25.get(t_pd, np.nan)
            co2_val    = co2_s.get(t_pd, np.nan)

            cooking_hrs = COOKING_HOURS_LOCAL_BY_SENSOR.get(loc, set())
            L1 = local_hour in cooking_hrs
            L2 = pm25_val >= pm25_thresh
            L3 = (not np.isnan(co2_val)) and (co2_val - co2_bl) >= co2_thresh
            L4 = (not np.isnan(out_val) and pm25_val >= pm25_thresh
                  and out_val < 15.0 and (pm25_val / (out_val + 0.5)) >= 1.5)

            if not L2:
                continue
            if co2_req and not L3:
                continue

            n = L2 + L3 + L4
            is_high_med = n >= 2 or (L2 and L1)
            is_low      = L2 and not is_high_med

            if is_high_med:
                pm25_conserv[ti, li] = np.nan
                pm25_strict[ti, li]  = np.nan
            elif is_low:
                pm25_strict[ti, li]  = np.nan

    ds_filt_pm25 = ds_ag.copy()
    ds_filt_pm25['pm25'].values[:]          = ds_ag['pm25'].values
    ds_filt_pm25['pm25_filtered']           = ds_filt_pm25['pm25'].copy()
    ds_filt_pm25['pm25_filtered'].values[:] = pm25_conserv
    ds_filt_pm25['pm25_strict']             = ds_filt_pm25['pm25'].copy()
    ds_filt_pm25['pm25_strict'].values[:]   = pm25_strict

    return ds_filt_pm25


print("\nApplying PM1 cooking filter...")
ds_ag_filt = apply_cooking_filter(ds_ag_clean, ds_aq_clean)
print("Done.")

print("Applying PM2.5 cooking filter...")
ds_ag_filt_pm25 = apply_cooking_filter_pm25(ds_ag_clean, ds_aq_clean)
print("Done.")

In [ ]:
# @title
# ── GRIGOREV PEAK CHARACTERIZATION (Validation of Cooking Filter) ─────────────
# Validates flagged cooking events against Grigorev et al. (2023, PNAS) shape
# criteria: peak prominence > 5 µg/m³ AND peak width < 6 hours.
#
# Inputs:
#   ds_ag_clean  — raw pre-filter xarray dataset (pm1 untouched)
#   ds_ag_filt   — post-filter xarray dataset (pm1_filtered has NaNs where flagged)
#
# Outputs:
#   event_log_df   — one row per detected peak, with Grigorev validation metadata
#   summary_table  — per-sensor summary comparing event timing to L1 cooking windows
#
# Logic:
#   A timestep is considered "flagged by cooking filter" if:
#       pm1_filtered is NaN  AND  raw pm1 is NOT NaN
#   (i.e. the filter set it to NaN, not a pre-existing gap)

import numpy as np
import pandas as pd
from scipy.signal import find_peaks

# ── CONFIG ────────────────────────────────────────────────────────────────────
PROMINENCE_THRESHOLD = 5.0    # µg/m³  — Grigorev et al. (2023)
MAX_WIDTH_HOURS      = 6.0    # hours  — Grigorev et al. (2023)
FREQ_MINUTES         = 5      # data resolution
MAX_WIDTH_SAMPLES    = int((MAX_WIDTH_HOURS * 60) / FREQ_MINUTES)  # 72 samples

# Cooking hours in LOCAL time (matches Cell 7 definition)
COOKING_HOURS_LOCAL_BY_SENSOR = {
    'AG-3 (SY)':            {11, 12, 15, 16, 17, 18},
    'AG-7 (IB)':            {9, 10, 17, 18},
    'AG-9 (SY)':            {7, 8, 17, 18},
    'AG-11 (IB Classroom)': {13, 14},
    'AG-12 (IB)':           {17, 18},
    'AG-15 (IB)':           {7, 8, 11, 12, 17, 18},
    'AG-17 (IB City Hall)': {11, 12},
    'AG-19 (ML)':          set(),
}

# Only sensors that were actually filtered in Cell 7
FILTERED_SENSORS = {'AG-3', 'AG-7', 'AG-9', 'AG-12', 'AG-13', 'AG-15'}

# ── EXTRACT TIMESTAMPS ────────────────────────────────────────────────────────
times_pdt = pd.DatetimeIndex(ds_ag_clean.time.values)

# ── MAIN LOOP: PER SENSOR ────────────────────────────────────────────────────
event_records = []

for li, loc in enumerate(ds_ag_clean.location.values):
    short = loc.split(' ')[0]
    if short not in FILTERED_SENSORS:
        continue

    # Raw PM1 (pre-filter) and filtered PM1 (post-filter)
    raw_pm1      = ds_ag_clean['pm1'].values[:, li].astype(float)
    filtered_pm1 = ds_ag_filt['pm1_filtered'].values[:, li].astype(float)

    # Boolean flag array: True where cooking filter set a value to NaN
    cooking_flagged = np.isnan(filtered_pm1) & ~np.isnan(raw_pm1)

    # Fill NaNs in raw with 0 for peak finder (NaN breaks scipy)
    pm1_for_peaks = np.where(np.isnan(raw_pm1), 0.0, raw_pm1)

    # ── Find peaks in raw PM1 ─────────────────────────────────────────────
    peak_indices, properties = find_peaks(
        pm1_for_peaks,
        prominence=PROMINENCE_THRESHOLD,
        width=1,           # minimum width; we apply max width filter below
        rel_height=0.5     # width measured at half prominence
    )

    prominences  = properties['prominences']
    left_bases   = properties['left_bases'].astype(int)
    right_bases  = properties['right_bases'].astype(int)

    for k, idx in enumerate(peak_indices):
        prominence   = prominences[k]
        width_samp   = right_bases[k] - left_bases[k]
        width_hours  = (width_samp * FREQ_MINUTES) / 60.0

        passes_grigorev = (prominence >= PROMINENCE_THRESHOLD) and \
                          (width_hours <= MAX_WIDTH_HOURS)

        # Is the peak timestep inside the cooking flag?
        inside_flag = bool(cooking_flagged[idx])

        # Local hour of peak (UTC - 7)
        t_pdt      = times_pdt[idx]
        local_hour = t_pdt.hour
        local_min  = t_pdt.minute

        # Is it within L1 cooking hours for this sensor?
        l1_hrs     = COOKING_HOURS_LOCAL_BY_SENSOR.get(loc, set())
        in_l1      = local_hour in l1_hrs

        # Duration of the contiguous flagged run containing this peak
        if inside_flag:
            run_start = idx
            run_end   = idx
            while run_start > 0 and cooking_flagged[run_start - 1]:
                run_start -= 1
            while run_end < len(cooking_flagged) - 1 and cooking_flagged[run_end + 1]:
                run_end += 1
            run_duration_min = (run_end - run_start + 1) * FREQ_MINUTES
        else:
            run_duration_min = np.nan

        event_records.append({
            'sensor'                  : short,
            'location'                : loc,
            'peak_time_pdt'           : t_pdt,
            'peak_hour_local'         : local_hour + local_min / 60.0,
            'peak_hour_local_str'     : f"{local_hour:02d}:{local_min:02d}",
            'peak_pm1_ugm3'           : round(float(raw_pm1[idx]), 2),
            'prominence_ugm3'         : round(float(prominence), 2),
            'width_hours'             : round(float(width_hours), 2),
            'passes_grigorev'         : passes_grigorev,
            'inside_cooking_flag'     : inside_flag,
            'in_l1_window'            : in_l1,
            'flag_run_duration_min'   : run_duration_min,
            'validated_event'         : passes_grigorev and inside_flag,
        })

event_log_df = pd.DataFrame(event_records)

# ── PRINT DETECTION SUMMARY ───────────────────────────────────────────────────
total        = len(event_log_df)
pass_grigo   = event_log_df['passes_grigorev'].sum()
inside_flag  = event_log_df['inside_cooking_flag'].sum()
validated    = event_log_df['validated_event'].sum()
flagged_only = (event_log_df['inside_cooking_flag'] & ~event_log_df['passes_grigorev']).sum()
grigo_only   = (event_log_df['passes_grigorev'] & ~event_log_df['inside_cooking_flag']).sum()

print(f"── Grigorev Peak Detection Summary ──────────────────────────────────")
print(f"  Total peaks found (prominence > {PROMINENCE_THRESHOLD} µg/m³): {total}")
print(f"  Pass Grigorev width criterion (< {MAX_WIDTH_HOURS}h):           {pass_grigo}")
print(f"  Inside cooking filter flag:                                      {inside_flag}")
print(f"  Validated by BOTH (cooking filter ∩ Grigorev):                  {validated}")
print(f"  Flagged by cooking filter but NOT Grigorev (low-amplitude):      {flagged_only}")
print(f"  Detected by Grigorev but NOT cooking filter (missed events):     {grigo_only}")

# ── SUMMARY TABLE ─────────────────────────────────────────────────────────────
# Built from validated events (pass Grigorev AND inside cooking flag)

validated_df = event_log_df[event_log_df['validated_event']].copy()

summary_rows = []
for loc, grp in validated_df.groupby('location'):
    short      = loc.split(' ')[0]
    l1_hrs     = COOKING_HOURS_LOCAL_BY_SENSOR.get(loc, set())
    l1_str     = ', '.join(
        f"{h:02d}:00–{h+1:02d}:59"
        for h in sorted(l1_hrs)
    ) if l1_hrs else 'None'

    n_events       = len(grp)
    median_hour    = grp['peak_hour_local'].median()
    med_h          = int(median_hour)
    med_m          = int((median_hour % 1) * 60)
    pct_in_l1      = 100.0 * grp['in_l1_window'].mean()
    med_peak_pm1   = grp['peak_pm1_ugm3'].median()
    med_prom       = grp['prominence_ugm3'].median()
    med_dur        = grp['flag_run_duration_min'].median()

    # Modal 2-hour window
    hour_bins = pd.cut(
        grp['peak_hour_local'],
        bins=list(range(0, 25, 2)),
        right=False,
        labels=[f"{h:02d}:00–{h+1:02d}:59" for h in range(0, 24, 2)]
    )
    modal_window = hour_bins.mode().iloc[0] if len(hour_bins) > 0 else 'N/A'

    summary_rows.append({
        'Sensor'                      : short,
        'Location'                    : loc,
        'L1 Survey Window(s)'         : l1_str,
        'N Validated Events'          : n_events,
        'Median Peak Hour (local)'    : f"{med_h:02d}:{med_m:02d}",
        'Modal 2h Window'             : modal_window,
        '% Events in L1 Window'       : f"{pct_in_l1:.1f}%",
        'Median Peak PM₁ (µg/m³)'    : med_peak_pm1,
        'Median Prominence (µg/m³)'   : med_prom,
        'Median Event Duration (min)' : med_dur,
    })

summary_table = pd.DataFrame(summary_rows).set_index('Sensor')

print(f"\n── Per-Sensor Cooking Event Validation Summary ───────────────────────")
print(summary_table.to_string())

# ── OPTIONAL: EXPORT EVENT LOG ────────────────────────────────────────────────
# Uncomment to save the full event log as CSV for SI table
# event_log_df.to_csv('cooking_event_log_grigorev.csv', index=False)
# print("\nEvent log saved to cooking_event_log_grigorev.csv")

In [ ]:
# @title
# ── Cell 8: EVENT DAY CLASSIFICATION ─────────────────────────────────────────
# Classify each day based on daily mean H2S and daily mean outdoor PM1

PDT_OFFSET = pd.Timedelta(hours=7)

def resample_local_daily(series_utc, agg='mean'):
    """Resample a UTC series to local PDT daily values."""
    local = series_utc.copy()
    if agg == 'mean':
        return local.resample('D').mean()
    elif agg == 'max':
        return local.resample('D').max()

# Outdoor PM1 and PM2.5 (local daily)
out_pm1_series = (ds_aq_clean['pm1']
                  .sel(location='QuantAQ ML')
                  .to_pandas())
out_pm1_series.index = pd.DatetimeIndex(out_pm1_series.index).tz_localize(None)

out_pm25_series = (ds_aq_clean['pm25']
                   .sel(location='QuantAQ ML')
                   .to_pandas())
out_pm25_series.index = pd.DatetimeIndex(out_pm25_series.index).tz_localize(None)

trailer_pm1_daily  = resample_local_daily(out_pm1_series,  'mean')
trailer_pm25_daily = resample_local_daily(out_pm25_series, 'mean')

# H2S: count hours per day exceeding threshold
h2s_local = h2s_trailer['h2s'].copy()
h2s_hourly = h2s_local.resample('h').mean()
h2s_hours_exceeding = (h2s_hourly >= H2S_HOURLY_THRESH).resample('D').sum()

# PM1: hourly mean, then count hours per day exceeding Q3
pm1_hourly = out_pm1_series.resample('h').mean()
study_mask_h = (pm1_hourly.index >= pd.Timestamp(PLOT_START)) & \
               (pm1_hourly.index <= pd.Timestamp(PLOT_END))
PM1_Q3_THRESH = float(pm1_hourly[study_mask_h].quantile(0.75))
pm1_hours_exceeding = (pm1_hourly >= PM1_Q3_THRESH).resample('D').sum()

HOURS_THRESH = 3

print(f"H2S threshold:    >= {H2S_HOURLY_THRESH} ppb for >= {HOURS_THRESH} hours/day")
print(f"PM1 Q3 threshold: {PM1_Q3_THRESH:.2f} µg/m³ for >= {HOURS_THRESH} hours/day")
print(f"PM2.5 daily mean: {PM25_DAILY_THRESH} µg/m³")

# Display top days for reference
# Calculate h2s_daily_mean here
h2s_daily_mean = resample_local_daily(h2s_trailer['h2s'], 'mean')
print("\nTop 5 H2S days (daily mean, local PDT):")
display(h2s_daily_mean.nlargest(5).rename('H2S daily mean (ppb)'))
print("\nTop 5 PM1 days (daily mean, local PDT):")
display(trailer_pm1_daily.nlargest(5).rename('PM1 daily mean (µg/m³)'))



def classify_day(date):
    """
    Event classification using hour-count threshold.
    H2S event : >= 8 hours in the day where hourly mean H2S >= 30 ppb
    PM1 event : >= 8 hours in the day where hourly mean PM1 >= study Q3
                AND daily mean PM2.5 >= 9 µg/m³
    Clean     : neither threshold met
    """
    d_str = str(pd.Timestamp(date).tz_localize(None).date())

    h_hours = float(h2s_hours_exceeding.get(d_str, 0))
    p_hours = float(pm1_hours_exceeding.get(d_str, 0))
    p25     = float(trailer_pm25_daily.get(d_str, 0))

    h2s_event = h_hours >= HOURS_THRESH
    pm1_event = (p_hours >= HOURS_THRESH) and (p25 >= PM25_DAILY_THRESH)

    if   h2s_event and pm1_event: return 'both'
    elif h2s_event:               return 'h2s'
    elif pm1_event:               return 'pm1'
    else:                         return 'clean'

def get_day_event(date):
    """Returns title-case category for analysis."""
    cat = classify_day(date)
    mapping = {
        'clean': 'Clean',
        'h2s':   'High H₂S',
        'pm1':   'High PM₁',
        'both':  'High H₂S + PM₁',
        'other':  None,
    }
    return mapping.get(cat, None)

# Shading helper used in all figures
def shade_days(ax, start, end):
    """Shade event days on a time-series axis."""
    d = pd.Timestamp(start)
    while d <= pd.Timestamp(end):
        cat = classify_day(d)
        color = {'h2s':SHADE_H2S,'pm1':SHADE_PM1,
                 'both':SHADE_BOTH,'clean':SHADE_CLEAN}.get(cat)
        if color:
            ax.axvspan(d, d + pd.Timedelta(days=1),
                       color=color, alpha=SHADE_ALPHA, zorder=0, lw=0)
        d += pd.Timedelta(days=1)

# Shared legend patches
SHADE_LEGEND = [
    mpatches.Patch(color=SHADE_H2S,   alpha=0.8, label='High H₂S day'),
    mpatches.Patch(color=SHADE_PM1,   alpha=0.8, label='High PM₁ day'),
    mpatches.Patch(color=SHADE_BOTH,  alpha=0.8, label='High H₂S + PM₁'),
    mpatches.Patch(color=SHADE_CLEAN, alpha=0.8, label='Clean reference'),
]

# Distance from trailer for sensor ordering
def dist_km(loc):
    c = AG_LOCATIONS.get(loc, None)
    if c is None: return 999
    return ((c[0]-TRAILER_COORDS[0])**2 + (c[1]-TRAILER_COORDS[1])**2)**0.5

ag_by_dist = sorted(
    [l for l in ds_ag_filt.location.values],
    key=dist_km
)

# ── Clean days and infiltration factors ───────────────────────────────────────
CLEAN_DAYS_LIST = [
    str(pd.Timestamp(d).date())
    for d in pd.date_range(PLOT_START, PLOT_END, freq='D')
    if classify_day(d) == 'clean'
]
print(f"\nClean reference days ({len(CLEAN_DAYS_LIST)}): {CLEAN_DAYS_LIST}")

# Print classification summary
print(f"\nEvent day summary:")
all_days = [str(pd.Timestamp(d).date()) for d in pd.date_range(PLOT_START, PLOT_END, freq='D')]
for cat in ['clean','h2s','both','pm1']:
    days = [d for d in all_days if classify_day(d)==cat]
    label = {'clean':'Clean','h2s':'High H₂S only','both':'High H₂S + PM₁','pm1':'High PM₁ only'}[cat]
    print(f"  {label}: {len(days)} days")

OUT_PM1_THRESH  = 9.0

# Background per sensor = mean pm1_strict on clean reference days
SENSOR_BACKGROUND = {}
for loc in ds_ag_filt.location.values:
    ind = ds_ag_filt['pm1_strict'].sel(location=loc).to_pandas()
    ind.index = pd.DatetimeIndex(ind.index).tz_localize(None)
    ind_dates = ind.index.date.astype(str)
    clean_vals = ind[np.isin(ind_dates, CLEAN_DAYS_LIST)].dropna()
    SENSOR_BACKGROUND[loc] = float(clean_vals.mean()) if len(clean_vals) > 10 else 0.0

# Background for pm2.5
SENSOR_BACKGROUND_PM25 = {}
for loc in ds_ag_filt_pm25.location.values:
    ind25 = ds_ag_filt_pm25['pm25_strict'].sel(location=loc).to_pandas()
    ind25.index = pd.DatetimeIndex(ind25.index).tz_localize(None)
    ind25_dates = ind25.index.date.astype(str)
    clean_vals25 = ind25[np.isin(ind25_dates, CLEAN_DAYS_LIST)].dropna()
    SENSOR_BACKGROUND_PM25[loc] = float(clean_vals25.mean()) if len(clean_vals25) > 10 else 0.0

# Outdoor PM1/PM25 at 5-min for regression
out_5min = (ds_aq_clean['pm1']
            .sel(location='QuantAQ ML')
            .to_pandas())
out_5min.index = pd.DatetimeIndex(out_5min.index).tz_localize(None)
out_5min = out_5min.resample('5min').mean()

out_5min_pm25 = (ds_aq_clean['pm25']
                 .sel(location='QuantAQ ML')
                 .to_pandas())
out_5min_pm25.index = pd.DatetimeIndex(out_5min_pm25.index).tz_localize(None)
out_5min_pm25 = out_5min_pm25.resample('5min').mean()

# Infiltration factors PM1
INFILTRATION_FACTORS = {}
print("\nComputing infiltration factors (PM1):")
print(f"{'Sensor':<30} {'F':>7} {'SE':>7} {'R²':>7} {'n':>6}")
print('-'*55)
for loc in ds_ag_filt.location.values:
    bg  = SENSOR_BACKGROUND[loc]
    ind = ds_ag_filt['pm1_strict'].sel(location=loc).to_pandas()
    ind.index = pd.DatetimeIndex(ind.index).tz_localize(None)
    out_aligned = out_5min.reindex(ind.index, method='nearest',
                                   tolerance=pd.Timedelta('6min'))
    od_mask = (out_aligned > OUT_PM1_THRESH) & ind.notna() & out_aligned.notna()
    y = (ind[od_mask] - bg).values
    x = out_aligned[od_mask].values
    valid = np.isfinite(x) & np.isfinite(y) & (y >= 0)
    if valid.sum() < 30:
        INFILTRATION_FACTORS[loc] = 0.0
        print(f"  {loc:<30}: insufficient data (n={valid.sum()})")
        continue
    slope, _, r, _, se = stats.linregress(x[valid], y[valid])
    INFILTRATION_FACTORS[loc] = max(0.0, round(float(slope), 3))
    print(f"  {loc:<30} {slope:>7.3f} {se:>7.3f} {r**2:>7.3f} {valid.sum():>6}")

print()
print("Background concentrations (clean day mean):")
for loc, bg in SENSOR_BACKGROUND.items():
    print(f"  {loc:<30}: {bg:.3f} µg/m³")

# Infiltration factors PM2.5
OUT_PM25_THRESH = 9.0
INFILTRATION_FACTORS_PM25 = {}
print("\nComputing PM2.5-based infiltration factors:")
print(f"{'Sensor':<30} {'F_pm25':>8} {'SE':>7} {'R²':>7} {'n':>6}")
print('-'*55)
for loc in ds_ag_filt_pm25.location.values:
    bg25 = SENSOR_BACKGROUND_PM25.get(loc, 0.0)
    ind25 = ds_ag_filt_pm25['pm25_strict'].sel(location=loc).to_pandas()
    ind25.index = pd.DatetimeIndex(ind25.index).tz_localize(None)
    out25_aligned = out_5min_pm25.reindex(ind25.index, method='nearest',
                                          tolerance=pd.Timedelta('6min'))
    od_mask25 = (out25_aligned > OUT_PM25_THRESH) & ind25.notna() & out25_aligned.notna()
    y25 = (ind25[od_mask25] - bg25).values
    x25 = out25_aligned[od_mask25].values
    valid25 = np.isfinite(x25) & np.isfinite(y25) & (y25 >= 0)
    if valid25.sum() < 30:
        INFILTRATION_FACTORS_PM25[loc] = 0.0
        print(f"  {loc:<30}: insufficient data (n={valid25.sum()})")
        continue
    slope25, _, r25, _, se25 = stats.linregress(x25[valid25], y25[valid25])
    INFILTRATION_FACTORS_PM25[loc] = max(0.0, round(float(slope25), 3))
    print(f"  {loc:<30} {slope25:>8.3f} {se25:>7.3f} {r25**2:>7.3f} {valid25.sum():>6}")

print()
print("PM1 vs PM2.5 infiltration factor comparison:")
print(f"{'Sensor':<30} {'F_pm1':>8} {'F_pm25':>8} {'ratio':>8}")
print('-'*55)
for loc in ds_ag_filt.location.values:
    f1  = INFILTRATION_FACTORS.get(loc, 0.0)
    f25 = INFILTRATION_FACTORS_PM25.get(loc, 0.0)
    ratio = f25/f1 if f1 > 0 else float('nan')
    print(f"  {loc:<30} {f1:>8.3f} {f25:>8.3f} {ratio:>8.3f}")

# ── PM1 DECOMPOSITION ─────────────────────────────────────────────────────────
def compute_decomposition(ds_ag_filt, ds_aq_clean):
    out_pm1_series = (ds_aq_clean['pm1']
                      .sel(location='QuantAQ ML')
                      .to_pandas())
    out_pm1_series.index = pd.DatetimeIndex(out_pm1_series.index).tz_localize(None)
    out_pm1_series = out_pm1_series.resample('5min').mean()

    out_pm25_series = (ds_aq_clean['pm25']
                   .sel(location='QuantAQ ML')
                   .to_pandas())
    out_pm25_series.index = pd.DatetimeIndex(out_pm25_series.index).tz_localize(None)
    out_pm25_series = out_pm25_series.resample('5min').mean()

    records = []
    for loc in ds_ag_filt.location.values:
        F  = INFILTRATION_FACTORS.get(loc, 0.0)
        bg = SENSOR_BACKGROUND.get(loc, 0.0)

        pm1_raw = ds_ag_filt['pm1'].sel(location=loc).to_pandas()
        pm1_raw.index = pd.DatetimeIndex(pm1_raw.index).tz_localize(None)

        out_aligned = out_pm1_series.reindex(pm1_raw.index,
                                             method='nearest',
                                             tolerance=pd.Timedelta('6min'))

        df = pd.DataFrame({
            'dt':      pm1_raw.index,
            'sensor':  loc,
            'pm1_raw': pm1_raw.values,
            'out_pm1': out_aligned.values,
        })
        df['C_tr'] = (F * df['out_pm1']).clip(lower=0)

        F_pm25_direct = INFILTRATION_FACTORS_PM25.get(loc, 0.0)
        F_pm1_val     = INFILTRATION_FACTORS.get(loc, 0.0)
        PM1_PM25_RATIO = 0.703

        if F_pm25_direct > 0:
            F_pm25 = F_pm25_direct
        else:
            F_pm25 = F_pm1_val / PM1_PM25_RATIO
            print(f"  {loc}: using F_pm1-scaled F_pm25 = {F_pm25:.3f}")

        out_pm25_aligned = out_pm25_series.reindex(pm1_raw.index,
                                                   method='nearest',
                                                   tolerance=pd.Timedelta('6min'))
        df['out_pm25']  = out_pm25_aligned.values
        df['C_tr_pm25'] = (F_pm25 * df['out_pm25']).clip(lower=0)
        df['C_background']  = bg
        df['C_indoor_gen']  = (df['pm1_raw'] - df['C_tr'] - bg).clip(lower=0)
        records.append(df)

    decomp_df = pd.concat(records, ignore_index=True)
    decomp_df['date']       = pd.to_datetime(decomp_df['dt']).dt.date.astype(str)
    decomp_df['hour_local'] = pd.to_datetime(decomp_df['dt']).dt.hour
    return decomp_df

print("\nComputing PM1 decomposition...")
decomp_df = compute_decomposition(ds_ag_filt, ds_aq_clean)
print(f"Decomposition complete: {len(decomp_df):,} rows across {decomp_df.sensor.nunique()} sensors")
print()
print("Mean TR-attributable PM1 (C_tr) by sensor:")
for loc, grp in decomp_df.groupby('sensor'):
    print(f"  {loc:<30}: {grp.C_tr.mean():.3f} µg/m³  (F={INFILTRATION_FACTORS.get(loc,0):.3f})")

print("\nAG sensors ordered by distance from PML:")
for l in ag_by_dist:
    print(f"  {l}")

# H2S SCATTERPLOT FIGURE

In [ ]:
# ── Figure 7: H₂S as a TR Tracer — Co-occurrence with Outdoor PM₁ ────────────

from statsmodels.nonparametric.smoothers_lowess import lowess
from scipy.stats import spearmanr, mannwhitneyu

FIG7_START = '2026-04-01'
FIG7_END   = '2026-04-28'

FIG7_SITES = [
    'QuantAQ IB City Hall',
    'QuantAQ ML',
    'QuantAQ TR Estuary',
    'QuantAQ Coast',
]
FIG7_COLORS = {
    'QuantAQ IB City Hall': 'green',
    'QuantAQ ML':          'purple',
    'QuantAQ TR Estuary':   'red',
    'QuantAQ Coast':        'blue',
}
FIG7_MARKERS = {
    'QuantAQ IB City Hall': 'o',
    'QuantAQ ML':          's',
    'QuantAQ TR Estuary':   '^',
    'QuantAQ Coast':        'D',
}
FIG7_LABELS = {
    'QuantAQ IB City Hall': 'IB City Hall',
    'QuantAQ ML':          'ML',
    'QuantAQ TR Estuary':   'TR Estuary',
    'QuantAQ Coast':        'Coast',
}

# ── Use same H2S threshold as Cell 8 ─────────────────────────────────────────
# High H2S day = >= HOURS_THRESH hours where hourly mean H2S >= H2S_HOURLY_THRESH
# h2s_hours_exceeding is already computed in Cell 8

h2s_fig7 = h2s_trailer['h2s'].copy().clip(lower=0)
h2s_fig7 = h2s_fig7[(h2s_fig7.index >= FIG7_START) & (h2s_fig7.index <= FIG7_END)]
h2s_daily_fig7 = h2s_fig7.resample('D').mean()

# Recompute hours exceeding for the Fig7 date window
h2s_hourly_fig7 = h2s_fig7.resample('h').mean()
hours_exc_fig7  = (h2s_hourly_fig7 >= H2S_HOURLY_THRESH).resample('D').sum()

high_days_fig7  = set(hours_exc_fig7[hours_exc_fig7 >= HOURS_THRESH].index.date)
n_high_fig7     = len(high_days_fig7)
n_low_fig7      = len(h2s_daily_fig7) - n_high_fig7

print(f"H2S threshold: >= {H2S_HOURLY_THRESH} ppb for >= {HOURS_THRESH} hours/day")
print(f"High H2S days: {n_high_fig7},  Low H2S days: {n_low_fig7}")
print(f"High H2S dates: {sorted(high_days_fig7)}")

# ── Build per-site daily and hourly PM1 series ────────────────────────────────
daily_stats_fig7  = {}
hourly_stats_fig7 = {}

for loc in FIG7_SITES:
    pm1_series = (ds_aq_clean['pm1']
                  .sel(location=loc)
                  .to_pandas())
    pm1_series.index = pd.DatetimeIndex(pm1_series.index).tz_localize(None)
    pm1_local = pm1_series[
        (pm1_series.index >= FIG7_START) & (pm1_series.index <= FIG7_END)
    ]

    # daily
    pm1_daily = pm1_local.resample('D').mean()
    d = pd.DataFrame({'h2s': h2s_daily_fig7, 'pm1': pm1_daily}).dropna()
    d = d[d['h2s'] > 1.0]
    d['is_high'] = [dt in high_days_fig7 for dt in d.index.date]
    r, p = spearmanr(d['h2s'], d['pm1'])
    daily_stats_fig7[loc] = {'df': d, 'r': r, 'p': p}

    # hourly box
    pm1_hourly = pm1_local.resample('1h').mean().dropna().reset_index()
    pm1_hourly.columns = ['time', 'pm1']
    pm1_hourly['date'] = pm1_hourly['time'].dt.date
    hi = pm1_hourly[pm1_hourly['date'].isin(high_days_fig7)]['pm1'].values
    lo = pm1_hourly[~pm1_hourly['date'].isin(high_days_fig7)]['pm1'].values
    _, p_mw = mannwhitneyu(hi, lo, alternative='greater')
    hourly_stats_fig7[loc] = {'hi': hi, 'lo': lo, 'p_mw': p_mw}

# ── Plot ──────────────────────────────────────────────────────────────────────
fig7, (ax_a, ax_b) = plt.subplots(
    1, 2, figsize=(13, 6),
    gridspec_kw={'wspace': 0.32, 'left': 0.08, 'right': 0.97,
                 'top': 0.88, 'bottom': 0.12}
)

# ─── Panel A: daily scatter ───────────────────────────────────────────────────
for loc in FIG7_SITES:
    dd  = daily_stats_fig7[loc]
    df  = dd['df']
    c   = FIG7_COLORS[loc]
    mk  = FIG7_MARKERS[loc]
    lbl = FIG7_LABELS[loc]
    ax_a.scatter(df.loc[~df['is_high'], 'h2s'], df.loc[~df['is_high'], 'pm1'],
                 marker=mk, facecolors='none', edgecolors=c,
                 s=55, linewidths=1.4, alpha=0.85, zorder=3)
    ax_a.scatter(df.loc[df['is_high'], 'h2s'], df.loc[df['is_high'], 'pm1'],
                 marker=mk, color=c, s=65, alpha=0.95, zorder=4)

ax_a.set_xlabel('Daily mean H₂S (ppb)', fontsize=14)
ax_a.set_ylabel('Daily mean outdoor PM₁ (µg m⁻³)', fontsize=14)
ax_a.set_title('(A)  Outdoor PM₁ vs. H₂S', fontsize=14, loc='left')

# Vertical line at H2S_HOURLY_THRESH for reference
ax_a.axvline(H2S_HOURLY_THRESH, color='#555', linestyle='--',
             linewidth=1.0, alpha=0.5,
             label=f'{H2S_HOURLY_THRESH} ppb CARB standard')

site_handles = [
    plt.Line2D([0], [0], marker=FIG7_MARKERS[loc], color=FIG7_COLORS[loc],
               linestyle='None', markersize=8, label=FIG7_LABELS[loc])
    for loc in FIG7_SITES
]
site_handles += [
    plt.Line2D([0], [0], marker='o', color='gray', linestyle='None',
               markersize=7, markerfacecolor='gray',
               label=f'High H₂S day  (≥{HOURS_THRESH}h ≥{H2S_HOURLY_THRESH} ppb, filled)'),
    plt.Line2D([0], [0], marker='o', color='gray', linestyle='None',
               markersize=7, markerfacecolor='none', markeredgecolor='gray',
               label='Low H₂S day  (open)'),
]
ax_a.legend(handles=site_handles, fontsize=10, loc='upper left', framealpha=0.9)
ax_a.grid(True, alpha=0.2, linewidth=0.4)

# ─── Panel B: grouped box plots ───────────────────────────────────────────────
n_sites    = len(FIG7_SITES)
group_gap  = 3.2
box_offset = 0.55
pos_lo   = np.array([i * group_gap + 1.0              for i in range(n_sites)])
pos_hi   = np.array([i * group_gap + 1.0 + box_offset for i in range(n_sites)])
tick_ctr = (pos_lo + pos_hi) / 2

for i, loc in enumerate(FIG7_SITES):
    hs = hourly_stats_fig7[loc]
    c  = FIG7_COLORS[loc]
    for pos, dat, alpha in [(pos_lo[i], hs['lo'], 0.35),
                             (pos_hi[i], hs['hi'], 0.85)]:
        if len(dat) == 0:
            continue
        bp = ax_b.boxplot(
            [dat], positions=[pos], widths=0.42, patch_artist=True,
            medianprops=dict(color='black', linewidth=2.0),
            flierprops=dict(marker='o', markersize=2.0, alpha=0.2, color='gray'),
            whiskerprops=dict(linewidth=1.2, color=c),
            capprops=dict(linewidth=1.2, color=c),
        )
        bp['boxes'][0].set_facecolor(c)
        bp['boxes'][0].set_alpha(alpha)
        bp['boxes'][0].set_edgecolor(c)

for i, loc in enumerate(FIG7_SITES):
    hs   = hourly_stats_fig7[loc]
    p_mw = hs['p_mw']
    sig  = '***' if p_mw < 0.001 else ('**' if p_mw < 0.01 else ('*' if p_mw < 0.05 else 'n.s.'))
    y_br = np.percentile(np.concatenate([hs['hi'], hs['lo']]), 93) * 1.02
    ax_b.plot([pos_lo[i], pos_hi[i]], [y_br, y_br], color='#444', lw=0.9)
    ax_b.text(tick_ctr[i], y_br * 1.01, sig, ha='center', va='bottom', fontsize=12)

ax_b.set_xticks(tick_ctr)
ax_b.set_xticklabels([FIG7_LABELS[loc] for loc in FIG7_SITES], fontsize=12)
ax_b.set_ylabel('Hourly outdoor PM₁ (µg m⁻³)', fontsize=14)
ax_b.set_title('(B) Outdoor PM₁ by H₂S day type — all QuantAQ sites',
               fontsize=14, loc='left')
ax_b.set_xlim(pos_lo[0] - 0.7, pos_hi[-1] + 0.7)
ax_b.grid(True, axis='y', alpha=0.2, linewidth=0.4)

lo_patch = mpatches.Patch(facecolor='gray', alpha=0.35,
                           label=f'Low H₂S days  (n={n_low_fig7})')
hi_patch = mpatches.Patch(facecolor='gray', alpha=0.85,
                           label=f'High H₂S days  (n={n_high_fig7}, '
                                 f'≥{HOURS_THRESH}h ≥{H2S_HOURLY_THRESH} ppb)')
ax_b.legend(handles=[lo_patch, hi_patch], fontsize=11, loc='upper right', framealpha=0.9)

# ── Statistics ────────────────────────────────────────────────────────────────
print("── Panel A: Daily Spearman correlations ─────────────────────────────────")
for loc in FIG7_SITES:
    dd    = daily_stats_fig7[loc]
    p_str = '<0.001' if dd['p'] < 0.001 else f'{dd["p"]:.3f}'
    print(f"  {FIG7_LABELS[loc]:<20} r_s={dd['r']:+.3f}  p={p_str}  n={len(dd['df'])} days")

print(f"\n── Panel B: Mann–Whitney U ───────────────────────────────────────────────")
print(f"  High H₂S: n={n_high_fig7} days  |  Low H₂S: n={n_low_fig7} days")
print()
for loc in FIG7_SITES:
    hs    = hourly_stats_fig7[loc]
    p_str = '<0.001' if hs['p_mw'] < 0.001 else f'{hs["p_mw"]:.3f}'
    sig   = '***' if hs['p_mw'] < 0.001 else ('**' if hs['p_mw'] < 0.01 else ('*' if hs['p_mw'] < 0.05 else 'n.s.'))
    print(f"  {FIG7_LABELS[loc]:<20} median High={np.median(hs['hi']):.2f}  "
          f"median Low={np.median(hs['lo']):.2f}  p={p_str} {sig}")

plt.savefig('figure7b_h2s.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()
print("Figure 7 saved.")

# SI FIG - METEO TIME SERIES

In [ ]:
# condensed time series including pm1 and meteo
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Cell 9: FIGURE 1 — Time series stacked plots ──────────────────────────────
# Rows: H2S | Outdoor PM1 | Relative Humidity | MLH | River stage (6h lag) |
#        Temperature + Shortwave radiation (twin axis)

trailer_loc = 'QuantAQ ML'
t_idx = pd.DatetimeIndex(ds_aq_clean.time.values)

out_pm1 = ds_aq_clean['pm1'].sel(location=trailer_loc).to_pandas()
out_rh   = ds_aq_clean['rh'].sel(location=trailer_loc).to_pandas()
out_temp = ds_aq_clean['temp'].sel(location=trailer_loc).to_pandas()
h2s_ts   = h2s_trailer['h2s']

# Align all series to a common 5-min index
common_idx = pd.date_range(PLOT_START, PLOT_END, freq='5min')
def align(series):
    return series.reindex(common_idx, method='nearest', tolerance=pd.Timedelta('6min'))

h2s_a    = align(h2s_ts)
pm1_a    = align(out_pm1)
rh_a     = align(out_rh)
temp_a   = align(out_temp)
mlh_a    = align(mlh['mlh'])
river_a  = align(river['river_stage_lag6h'])
solar_a  = align(mlh['shortwave'])


NROWS = 6
fig1, axes = plt.subplots(NROWS, 1, figsize=(16, 14), sharex=True)
fig1.subplots_adjust(left=0.08, right=0.93, top=0.95, bottom=0.06, hspace=0.06)

rows = [
    (axes[0], h2s_a,   C_H2S,  'H₂S (ppb)',     None),
    (axes[1], pm1_a,   C_PM1,  'PM₁ (µg m⁻³)',  None),
    (axes[2], rh_a,    C_RH,   'RH (%)', None),
    (axes[3], mlh_a,   C_MLH,  'MLH (m)',        None),
    (axes[4], river_a, C_RIV,  'River Stage (m)', None),
]
for ax, data, color, ylabel, _ in rows:
    ax.plot(data.index, data.values, color=color, lw=1.5, alpha=0.85)
    ax.set_ylabel(ylabel, fontsize=16)
    ax.grid(True, axis='y')
    shade_days(ax, PLOT_START, PLOT_END)

# Row 6: Temperature + Solar radiation (twin axis)
ax6  = axes[5]
ax6b = ax6.twinx()
ax6.plot(temp_a.index,  temp_a.values,  color=C_TEMP, lw=1.2, alpha=0.85, label='Temp (°C)')
ax6b.plot(solar_a.index, solar_a.values, color=C_SOL, lw=1.2, alpha=0.65,
          linestyle='--', label='Shortwave (W m⁻²)')
ax6.set_ylabel('Temperature (°C)', fontsize=16, color=C_TEMP)
ax6b.set_ylabel('Shortwave (W m⁻²)', fontsize=16, color=C_SOL)
ax6.tick_params(axis='y', colors=C_TEMP)
ax6b.tick_params(axis='y', colors=C_SOL)
ax6b.spines['right'].set_visible(True)
ax6b.spines['right'].set_color(C_SOL)
ax6b.spines['top'].set_visible(False)
shade_days(ax6, PLOT_START, PLOT_END)

# Format x-axis
axes[-1].xaxis.set_major_locator(mdates.DayLocator(interval=2))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %-d'))
plt.setp(axes[-1].xaxis.get_majorticklabels(), rotation=45)
axes[-1].set_xlabel('Date', fontsize=18) # Updated xlabel

# Panel labels
for i, ax in enumerate(axes):
    ax.text(0.005, 0.93, f'({chr(97+i)})', transform=ax.transAxes,
            fontsize=18,  va='top')

# Legend
fig1.legend(handles=SHADE_LEGEND, fontsize=18, ncol=4,
            loc='upper center', bbox_to_anchor=(0.5, 0.995),
            frameon=True, framealpha=0.9, edgecolor='#ccc')
fig1.savefig('figure1_timeseries.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()

# INTRO TIME SERIES PLOT

# SI FIG - FULL TIME SERIES HEATMAP

In [ ]:
# @title
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Figure 3: Single Heatmap for Full Period with Event Day Boxes ─────

# 1. Define all QuantAQ sensors, ordered by distance from ML (hotspot)
QUANTAQ_LOCS_RAW = list(QUANTAQ_FILES.keys())

# Calculate distances for QuantAQ sensors from TRAILER_COORDS
quantaq_distances = []
for loc_name in QUANTAQ_LOCS_RAW:
    if loc_name in QUANTAQ_LOCATIONS:
        coords = QUANTAQ_LOCATIONS[loc_name]
        # Assuming TRAILER_COORDS is available in the global scope from Cell 8
        d = ((coords[0]-TRAILER_COORDS[0])**2 + (coords[1]-TRAILER_COORDS[1])**2)**0.5
        quantaq_distances.append((d, loc_name))
    else:
        quantaq_distances.append((float('inf'), loc_name)) # Put at end if no coords

quantaq_distances.sort()
ALL_QUANTAQ_LOCS_ORDERED = [loc for dist, loc in quantaq_distances]

# 2. Define all sensors for the heatmap (QuantAQ + AG ordered by distance from PML)
# AG-6 is excluded as it generally has no data for the full study period
# AG-13 is excluded as requested by the user
AG_SENSORS_FOR_HM = [l for l in ag_by_dist if 'AG-6' not in l and 'AG-13' not in l]
ALL_SENSORS_FOR_HM = ALL_QUANTAQ_LOCS_ORDERED + AG_SENSORS_FOR_HM

# 3. Prepare data for the entire period
frames_full_period = {}
# Create a master 5-minute time grid for the specific day to force alignment
master_index = pd.date_range(
    start=pd.to_datetime(PLOT_START),
    end=pd.to_datetime(PLOT_END),
    freq='5min'
)

# This list will keep track of sensors that actually made it into frames_full_period
actual_sensors_in_heatmap = []

for s in ALL_SENSORS_FOR_HM:
    if s in ALL_QUANTAQ_LOCS_ORDERED: # QuantAQ outdoor sensors
        if s in ds_aq_clean.location.values:
            ser = ds_aq_clean['pm1'].sel(location=s).to_pandas()
            ser.index = pd.DatetimeIndex(ser.index).tz_localize(None) # Make timezone-naive
            frames_full_period[s] = ser.resample('5min').mean().ffill().bfill().reindex(master_index, method='nearest', tolerance=pd.Timedelta('10min'))
            actual_sensors_in_heatmap.append(s) # Add to list if successfully processed
    else: # AG indoor/outdoor sensors (now includes AG-19)
        if s in ds_ag_filt.location.values:
            ser = ds_ag_filt['pm1_filtered'].sel(location=s).to_pandas()
            ser.index = pd.DatetimeIndex(ser.index).tz_localize(None)
            frames_full_period[s] = ser.resample('5min').mean().reindex(master_index, method='nearest', tolerance=pd.Timedelta('10min'))
            actual_sensors_in_heatmap.append(s) # Add to list if successfully processed

full_hm_df = pd.DataFrame(frames_full_period)

# Now, ensure the plotting order uses only the sensors that were actually included
# Re-sort actual_sensors_in_heatmap based on the original desired order in ALL_SENSORS_FOR_HM
# This is important to maintain the distance-based ordering.
final_heatmap_order = [s for s in ALL_SENSORS_FOR_HM if s in actual_sensors_in_heatmap]

# Ensure the data is sorted by sensor order for consistent plotting
full_hm_df = full_hm_df[final_heatmap_order]

# 4. Setup the single heatmap plot
VMIN, VMAX = 0, 15
norm_hm = mcolors.Normalize(vmin=VMIN, vmax=VMAX)
CMAP_HM = 'viridis'

fig3, ax = plt.subplots(figsize=(24, 10)) # Adjust figure size as needed

sns.heatmap(full_hm_df.T, ax=ax, cmap=CMAP_HM, norm=norm_hm,
            xticklabels=False, yticklabels=True, # Set xticklabels to False
            cbar=True, cbar_kws={'label': 'PM₁ (µg m⁻³)'},
            linewidths=0)

# 5. Add colored boxes for event days
all_dates = pd.date_range(PLOT_START, PLOT_END, freq='D')
SHADE_COLORS = {
    'h2s': SHADE_H2S,
    'pm1': SHADE_PM1,
    'both': SHADE_BOTH,
    'clean': SHADE_CLEAN
}

for d in all_dates:
    cat = classify_day(d) # classify_day is defined in Cell 8
    color = SHADE_COLORS.get(cat)
    if color:
        # Determine if a border should be added and its color/width
        lw = 0
        edgecolor = None
        if cat in ['both', 'clean']:
            lw = 1.5  # Line width for the border
            edgecolor = 'black' # Color of the border

        ax.axvspan(pd.to_datetime(d), pd.to_datetime(d) + pd.Timedelta(days=1),
                   color=color, alpha=SHADE_ALPHA, zorder=1, # Increased zorder for visibility
                   lw=lw, edgecolor=edgecolor)

# 6. X-axis formatting (explicitly setting positions and labels)
# Create a date range for the desired tick locations (every day)
start_date_for_ticks = pd.to_datetime(PLOT_START)
end_date_for_ticks = pd.to_datetime(PLOT_END)
tick_dates_for_display = pd.date_range(start=start_date_for_ticks, end=end_date_for_ticks, freq='D')

tick_positions = []
tick_labels = []

for display_dt in tick_dates_for_display:
    # Align target_dt to the start time of master_index (PLOT_START + 7h)
    target_dt = display_dt + pd.Timedelta(hours=7)

    # Find the integer position (column index) in the heatmap for this target_dt
    # Use searchsorted for efficient lookup in a sorted index
    idx = master_index.searchsorted(target_dt, side='left')

    # Ensure the found index is valid and corresponds closely to the target date
    # Adding a check for proximity to avoid misaligned ticks if target_dt falls between 5-min intervals
    if 0 <= idx < len(master_index) and np.abs(master_index[idx] - target_dt) < pd.Timedelta('1D'):
        tick_positions.append(idx)
        tick_labels.append(display_dt.strftime('%b %d')) # Label with just Month Day

ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels, rotation=45, ha='right', fontsize=16)
ax.tick_params(axis='x', which='major', pad=10)

# Y-axis labels
ax.set_ylabel('Sensors', fontsize=18) # Increased font size
ax.tick_params(axis='y', labelsize=16) # Increased font size

# Divider line between QuantAQ and AG sensors
qaq_count = len(ALL_QUANTAQ_LOCS_ORDERED)
ax.axhline(qaq_count, color='white', lw=1.5)

ax.set_xlabel('Date', fontsize=20) # Updated x-axis label

# Adjust colorbar label font size
ax.collections[0].colorbar.set_label('PM₁ (µg m⁻³)', fontsize=20) # Increased font size

# Ensure tight layout and save
fig3.tight_layout() # Removed rect to let tight_layout handle automatically
fig3.savefig('figure3_single_heatmap_full_period.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()

# THREE DAY HEAT MAP

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Cell 12: FIGURE 4 — Boxplots by day-level event type ───────────────────
# Day classified by daily mean H2S and PM1.
# Split daytime (06-20 UTC) / nighttime (20-06 UTC).
# Whiskers: 9th / 91st percentile. Notches: 95% CI.

# Build long DataFrame: sensor, pm1, event_type, is_daytime

EVENT_ORDER = ['Clean', 'High H₂S + PM₁']
EVENT_COLS  = {'Clean':'blue',
               'High H₂S + PM₁':'purple'}

records = []
for loc in ds_ag_filt.location.values:
    pm1_da  = ds_ag_filt['pm1_filtered'].sel(location=loc)
    for t in ds_ag_filt.time.values:
        val = float(pm1_da.sel(time=t).values)
        if np.isnan(val): continue
        t_pd  = pd.Timestamp(t)
        day   = str(t_pd.date())
        etype = get_day_event(day)
        if etype is None: continue
        records.append({
            'sensor':     loc,
            'pm1':        val,
            'event_type': etype,
            'is_daytime': int(6 <= (t_pd.hour -7) %24 < 20),
        })

bp_df = pd.DataFrame(records)

# Filter ag_by_dist specifically for Figure 4
ag_by_dist_f4 = [s for s in ag_by_dist if s not in ['AG-9 (SY)', 'AG-3 (SY)']]

# Define sensor colors
num_ag_sensors = len(ag_by_dist_f4)
SENSOR_COLORS = {loc: plt.cm.tab10(i / num_ag_sensors) for i, loc in enumerate(ag_by_dist_f4)}

def custom_boxplot(ax, data, positions, colors, labels):
    """Draw boxplot with 9th/91st whiskers and notched 95% CI."""
    for pos, (et, col) in zip(positions, zip(labels, colors)):
        d = data[data['event_type'] == et]['pm1'].dropna().values
        if len(d) < 10: continue
        q9, q25, med, q75, q91 = np.percentile(d, [9, 25, 50, 75, 91])
        iqr = q75 - q25
        bw  = 0.35

        # Box
        rect = mpatches.Rectangle((pos - bw/2, q25), bw, iqr,
                                   facecolor=col, alpha=0.30,
                                   edgecolor=col, lw=1.2, zorder=3)

        # Whiskers (9th / 91st)
        ax.plot([pos, pos], [q9,  q25], color=col, lw=1.1, alpha=0.8, zorder=3)
        ax.plot([pos, pos], [q75, q91], color=col, lw=1.1, alpha=0.8, zorder=3)
        for cap_y in [q9, q91]:
            ax.plot([pos - bw*0.2, pos + bw*0.2], [cap_y, cap_y],
                    color=col, lw=1.1, alpha=0.8, zorder=3)

fig4, axes4 = plt.subplots(2, 1, figsize=(14, 7), sharey=True)
fig4.subplots_adjust(left=0.09, right=0.97, top=0.91, bottom=0.10, wspace=0.06)

# colors_f4 = [EVENT_COLS[e] for e in EVENT_ORDER] # No longer used for boxplot colors

for ai, (ax, period, tlabel) in enumerate(zip(
    axes4, ['Daytime (06:00–19:59)', 'Nighttime (20:00–05:59)'], [1, 0]
)):
    sub = bp_df[bp_df['is_daytime'] == tlabel]
    n_events = len(EVENT_ORDER)
    n_sensors = len(ag_by_dist_f4) # Use filtered list
    GW = n_sensors + 1.5   # group width

    for gi, et in enumerate(EVENT_ORDER):
        x_base = gi * GW
        if gi > 0:
            ax.axvline(x_base - 0.75, color='#cccccc', lw=0.7, zorder=0)
        for si, loc in enumerate(ag_by_dist_f4): # Use filtered list
            x = x_base + si
            d = sub[(sub['sensor'] == loc) & (sub['event_type'] == et)]['pm1'].dropna().values
            if len(d) < 10: continue
            q9, q25, med, q75, q91 = np.percentile(d, [9, 25, 50, 75, 91])
            iqr = q75 - q25
            ci  = 1.57 * iqr / np.sqrt(len(d))
            bw  = 0.52
            col = SENSOR_COLORS[loc] # Color by sensor
            rect = mpatches.Rectangle((x - bw/2, q25), bw, iqr,
                                       facecolor=col, alpha=0.25,
                                       edgecolor=col, lw=1.0, zorder=3)
            ax.add_patch(rect)
            for ny in [med - ci, med + ci]:
                ax.plot([x - bw*0.3, x - bw/2], [ny, ny], color=col, lw=0.8, zorder=4)
                ax.plot([x + bw*0.3, x + bw/2], [ny, ny], color=col, lw=0.8, zorder=4)
            ax.plot([x - bw*0.3, x + bw*0.3], [med, med], color=col, lw=2.0, zorder=5)
            ax.plot([x, x], [q9,  q25], color=col, lw=1.0, alpha=0.8, zorder=3)
            ax.plot([x, x], [q75, q91], color=col, lw=1.0, alpha=0.8, zorder=3)
            for cy in [q9, q91]:
                ax.plot([x - bw*0.2, x + bw*0.2], [cy, cy], color=col, lw=1.0, alpha=0.8)

    tick_x = [gi * GW + n_sensors / 2 - 0.5 for gi in range(n_events)]
    ax.set_xticks(tick_x)
    ax.set_xticklabels(EVENT_ORDER, fontsize=18)
    ax.set_xlim(-0.8, n_events * GW - 0.6)
    ax.set_ylim(0, 5) # Capped y-axis at 30
    ax.set_yticks([0, 1, 2, 3, 4, 5]) # Adjusted y-ticks accordingly
    if ai == 0: # Daytime
        ax.set_title(period, fontsize=16, pad=4)
    else: # Nighttime
        ax.set_title(period, fontsize=16, pad=-5) # Adjusted pad for nighttime title
    ax.grid(True, axis='y', alpha=0.2)
    ax.text(0.005, 0.97, f'({"ab"[ai]})', transform=ax.transAxes,
            fontsize=16, va='center')

# Add common y-label
fig4.supylabel('Indoor PM₁ (µg m⁻³)', fontsize=18, x=0.04)

# Sensor legend
sensor_handles = [mpatches.Patch(color=SENSOR_COLORS[l], alpha=0.7, label=l)
                  for l in ag_by_dist_f4] # Use filtered list

# Add the legend to fig4
legend = fig4.legend(handles=sensor_handles, fontsize=16, ncol=1,
            loc='center left', bbox_to_anchor=(1, 0.5),
            frameon=True, framealpha=0.9, edgecolor='#ccc', title='AG Sensors')
legend.get_title().set_fontsize(18) # Set legend title font size


fig4.savefig('figure4_boxplots.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()
print("Figure 4 saved.")

### BOXPLOT AVERAGE DAILY MAX


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# --- CONFIGURATION (re-use from Figure 4 where appropriate) ---
# EVENT_ORDER and EVENT_COLS are the same as Figure 4 (defined in WgnBQdqkOSVi)

# Sensors to include (consistent with Figure 4, excluding 'AG-9 (SY)' and 'AG-3 (SY)')
ag_by_dist_f4b = [s for s in ag_by_dist if s not in ['AG-9 (SY)', 'AG-3 (SY)', 'AG-6 (IB Classroom)']]

# Define sensor colors (consistent with Figure 4)
num_ag_sensors_f4b = len(ag_by_dist_f4b)
SENSOR_COLORS_F4B = {loc: plt.cm.tab10(i / num_ag_sensors_f4b) for i, loc in enumerate(ag_by_dist_f4b)}

EVENT_ORDER_f4b = ['Clean', 'High H₂S + PM₁', 'All']


# Short display labels
def short_label(loc):
    """Convert full location name to compact label for heatmap y-axis."""
    replacements = {
        'AG-19 (ML)':    'R1 (0.716 km)',
        'AG-15 (IB)':            'R2 (2.068 km)',
        'AG-7 (IB)':             'R3 (2.530 km)',
        'AG-17 (IB City Hall)':  'I1 (2.899 km)',
        'AG-11 (IB Classroom)':  'I2 (3.557 km)',
        'AG-12 (IB)':            'R4 (4.429 km)',
        'AG-13 (TR Estuary)':    'I3 (3.607 km)',
        'AG-6 (IB Classroom)':   'AG-6 IB Classroom (2.478 km)',
    }
    return replacements.get(loc, loc)

# --- DATA PREPARATION: Daily Maximum PM1 ---

records_daily_max = []

# Iterate through each unique date in the filtered AG data
for date_val in pd.to_datetime(ds_ag_filt.time.values).normalize().unique():
    date_str = str(date_val.date())
    event_type = get_day_event(date_str) # Classify the entire day
    if event_type is None: # Skip if the day cannot be classified
        continue

    # Convert date_val to PDT start of day for accurate time slicing
    pdt_start_of_day = pd.Timestamp(date_str).tz_localize('America/Los_Angeles') # Use 'America/Los_Angeles' as defined implicitly by PDT_OFFSET
    pdt_end_of_day   = pdt_start_of_day + pd.Timedelta(days=1) - pd.Timedelta(minutes=5) # End of day inclusive

    for loc in ds_ag_filt.location.values:
        # Select data for the specific sensor and day
        pm1_data_for_day = ds_ag_filt['pm1_strict'].sel(
            location=loc,
            time=slice(date_val, date_val + pd.Timedelta(days=1) - pd.Timedelta(minutes=5)) # slice for the whole day in UTC
        ).to_pandas()

        if pm1_data_for_day.empty or pm1_data_for_day.isnull().all():
            continue


        # Filter for daytime (06:00-19:59 PDT) and nighttime (20:00-05:59 PDT)
        daytime_pm1_values = pm1_data_for_day[(pm1_data_for_day.index.hour >= 6) & (pm1_data_for_day.index.hour < 20)]
        nighttime_pm1_values = pm1_data_for_day[~((pm1_data_for_day.index.hour >= 6) & (pm1_data_for_day.index.hour < 20))]

        # Calculate daily maximum for daytime and nighttime separately
        daytime_max_pm1 = daytime_pm1_values.max()
        nighttime_max_pm1 = nighttime_pm1_values.max()

        if pd.notna(daytime_max_pm1):
            records_daily_max.append({
                'sensor': loc, 'pm1': daytime_max_pm1,
                'event_type': event_type, 'is_daytime': 1, 'date': date_str
            })
            # Also add to 'All' bucket
            records_daily_max.append({
                'sensor': loc, 'pm1': daytime_max_pm1,
                'event_type': 'All', 'is_daytime': 1, 'date': date_str
            })
        if pd.notna(nighttime_max_pm1):
            records_daily_max.append({
                'sensor': loc, 'pm1': nighttime_max_pm1,
                'event_type': event_type, 'is_daytime': 0, 'date': date_str
            })
            records_daily_max.append({
                'sensor': loc, 'pm1': nighttime_max_pm1,
                'event_type': 'All', 'is_daytime': 0, 'date': date_str
            })

bp_df_daily_max = pd.DataFrame(records_daily_max)

# --- PLOTTING: Figure 4b ---

fig_f4b, axes_f4b = plt.subplots(2, 1, figsize=(14, 7), sharey=True)
fig_f4b.subplots_adjust(left=0.09, right=0.97, top=0.91, bottom=0.10, wspace=0.06)

# Re-using custom_boxplot function if defined globally, otherwise re-define here for self-containment
# Assuming custom_boxplot (from WgnBQdqkOSVi) is available, otherwise it needs to be copied
# For simplicity, using the original boxplot plotting logic adapted below.

for ai, (ax, period_label, tlabel) in enumerate(zip(
    axes_f4b, ['Daytime (06:00–19:59)', 'Nighttime (20:00–05:59)'], [1, 0]
)):
    sub = bp_df_daily_max[bp_df_daily_max['is_daytime'] == tlabel]

    n_events = len(EVENT_ORDER_f4b)
    n_sensors = len(ag_by_dist_f4b)
    GW = n_sensors + 1.5 # group width

    for gi, et in enumerate(EVENT_ORDER_f4b):
        x_base = gi * GW
        if gi > 0:
            ax.axvline(x_base - 0.75, color='#cccccc', lw=0.7, zorder=0)
        for si, loc in enumerate(ag_by_dist_f4b):
            x = x_base + si
            d = sub[(sub['sensor'] == loc) & (sub['event_type'] == et)]['pm1'].dropna().values
            # Removed: if len(d) < 10: continue

            # Only attempt to plot if there is *some* data
            if len(d) > 0:
                q9, q25, med, q75, q91 = np.percentile(d, [9, 25, 50, 75, 91])
                iqr = q75 - q25
                # Handle cases where IQR might be zero or very small
                if iqr == 0:
                    ci = 0 # No CI if IQR is zero
                else:
                    # 95% CI notch half-width for median
                    # This is approximately 1.57 * IQR / sqrt(N)
                    ci = 1.57 * iqr / np.sqrt(len(d)) # 95% CI notch half-width
                bw = 0.52
                col = SENSOR_COLORS_F4B[loc]

                # Box (IQR)
                rect = mpatches.Rectangle((x - bw/2, q25), bw, iqr,
                                           facecolor=col, alpha=0.25,
                                           edgecolor=col, lw=1.0, zorder=3)
                ax.add_patch(rect)

                # Median (no notches)
                ax.plot([x - bw*0.3, x + bw*0.3], [med, med], color=col, lw=2.0, zorder=5)

                # Whiskers (9th/91st percentile)
                ax.plot([x, x], [q9, q25], color=col, lw=1.0, alpha=0.8, zorder=3)
                ax.plot([x, x], [q75, q91], color=col, lw=1.0, alpha=0.8, zorder=3)
                for cy in [q9, q91]:
                    ax.plot([x - bw*0.2, x + bw*0.2], [cy, cy], color=col, lw=1.0, alpha=0.8)

    tick_x = [gi * GW + n_sensors / 2 - 0.5 for gi in range(n_events)]
    ax.set_xticks(tick_x)
    if ai == 1: # Only set x-tick labels for the bottom subplot (nighttime)
        ax.set_xticklabels(EVENT_ORDER_f4b, fontsize=18)
    else:
        ax.set_xticklabels([]) # Remove labels for the top subplot
    ax.set_xlim(-0.8, n_events * GW - 0.6)
    ax.set_ylim(0, 12) # Adjusted y-axis for potentially higher max values
    ax.set_yticks([0, 2, 4, 6, 8, 10, 12])

    ax.set_title(period_label, fontsize=16, pad=4 if ai == 0 else -5)
    ax.grid(True, axis='y', alpha=0.2)
    ax.text(0.005, 0.97, f'({("ab")[ai]})', transform=ax.transAxes,
            fontsize=16, va='center')

# Add common y-label
fig_f4b.supylabel('Daily Max Indoor PM₁ (µg m⁻³)', fontsize=18, x=0.04)

# Sensor legend
sensor_handles_f4b = [mpatches.Patch(color=SENSOR_COLORS_F4B[l], alpha=0.7, label=short_label(l))
                      for l in ag_by_dist_f4b]

legend_f4b = fig_f4b.legend(handles=sensor_handles_f4b, fontsize=16, ncol=1,
                            loc='center left', bbox_to_anchor=(1, 0.5),
                            frameon=True, framealpha=0.9, edgecolor='#ccc', title='AG Sensors')
legend_f4b.get_title().set_fontsize(18)

fig_f4b.savefig('figure4b_boxplots_daily_max.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()
print("Figure 4b saved.")

# combined boxplot figure

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import string
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG ─────────────────────────────
INCLUDE_ALL_COLUMN = False
EVENT_ORDER_COMBINED = ['Clean', 'High H₂S + PM₁'] + (['All'] if INCLUDE_ALL_COLUMN else [])

MET_ROW_CONFIG = [
    ('MLH (m)',          met_dfs['MLH (m)'],          '#2A4E7A'),
    ('Wind Speed (m/s)', met_dfs['Wind Speed (m/s)'], '#2C6E49'),
    ('River Stage (m)',  met_dfs['River Stage (m)'],  '#B5651D'),
]

n_events = len(EVENT_ORDER_COMBINED)
n_sensors = len(ag_by_dist_f4c)
GW = n_sensors + 1.5

MET_BOX_WIDTH = 2  # wider met boxplots (was 1.2)

fig, axes = plt.subplots(
    4, 2,
    figsize=(18, 16),  # bumped height slightly to give the taller PM1 row room to breathe
    gridspec_kw={'hspace': 0.12, 'wspace': 0.08, 'height_ratios': [1.5, 1, 1, 1]}
)
panel_labels = iter(string.ascii_lowercase)

# ── ROW 1: PM1 by sensor ─ shared y-max across day+night ────────
pm1_all_vals = bp_df_daily_max['pm1'].dropna().values
pm1_ymax = np.percentile(pm1_all_vals, 99.5) * 1.1

for ai, (period_label, tlabel) in enumerate(zip(['Daytime (06:00–19:59)', 'Nighttime (20:00–05:59)'], [1, 0])):
    ax = axes[0, ai]
    sub = bp_df_daily_max[bp_df_daily_max['is_daytime'] == tlabel]

    for gi, et in enumerate(EVENT_ORDER_COMBINED):
        x_base = gi * GW
        if gi > 0:
            ax.axvline(x_base - 0.75, color='#cccccc', lw=0.7, zorder=0)
        for si, loc in enumerate(ag_by_dist_f4c):
            x = x_base + si
            d = sub[(sub['sensor'] == loc) & (sub['event_type'] == et)]['pm1'].dropna().values
            if len(d) > 0:
                q9, q25, med, q75, q91 = np.percentile(d, [9, 25, 50, 75, 91])
                iqr = q75 - q25
                bw = 0.52
                col = SENSOR_COLORS_F4C[loc]
                rect = mpatches.Rectangle((x - bw/2, q25), bw, iqr, facecolor=col, alpha=0.25,
                                           edgecolor=col, lw=1.0, zorder=3)
                ax.add_patch(rect)
                ax.plot([x - bw*0.3, x + bw*0.3], [med, med], color=col, lw=2.0, zorder=5)
                ax.plot([x, x], [q9, q25], color=col, lw=1.0, alpha=0.8, zorder=3)
                ax.plot([x, x], [q75, q91], color=col, lw=1.0, alpha=0.8, zorder=3)
                for cy in [q9, q91]:
                    ax.plot([x - bw*0.2, x + bw*0.2], [cy, cy], color=col, lw=1.0, alpha=0.8)

    ax.set_ylim(0, pm1_ymax)
    ax.set_xlim(-0.8, n_events * GW - 0.6)
    ax.set_xticks([])
    ax.grid(True, axis='y', alpha=0.2)
    ax.set_title(period_label, fontsize=16)
    if ai == 1:
        ax.set_yticklabels([])
    else:
        ax.set_ylabel('Period Max\nIndoor PM₁ (µg m⁻³)', fontsize=16)
    ax.text(0.015, 0.93, f'({next(panel_labels)})', transform=ax.transAxes, fontsize=16, fontweight='bold')

# ── ROWS 2–4: Met variables ─ shared y-range per row, wider boxes ───────
for ri, (var_name, df, col) in enumerate(MET_ROW_CONFIG, start=1):
    row_vals = df['value'].dropna().values
    pad = (row_vals.max() - row_vals.min()) * 0.08
    row_ymin = row_vals.min() - pad
    row_ymax = row_vals.max() + pad

    for ai, (period_label, tlabel) in enumerate(zip(['Daytime', 'Nighttime'], [1, 0])):
        ax = axes[ri, ai]
        sub = df[df['is_daytime'] == tlabel]

        for gi, et in enumerate(EVENT_ORDER_COMBINED):
            x_center = gi * GW + n_sensors / 2 - 0.5
            d = sub[sub['event_type'] == et]['value'].dropna().values
            if len(d) == 0:
                continue
            q9, q25, med, q75, q91 = np.percentile(d, [9, 25, 50, 75, 91])
            iqr = q75 - q25
            bw = MET_BOX_WIDTH
            rect = mpatches.Rectangle((x_center - bw/2, q25), bw, iqr, facecolor=col, alpha=0.3,
                                       edgecolor=col, lw=1.2, zorder=3)
            ax.add_patch(rect)
            ax.plot([x_center - bw*0.3, x_center + bw*0.3], [med, med], color=col, lw=2.2, zorder=5)
            ax.plot([x_center, x_center], [q9, q25], color=col, lw=1.2, alpha=0.8, zorder=3)
            ax.plot([x_center, x_center], [q75, q91], color=col, lw=1.2, alpha=0.8, zorder=3)
            for cy in [q9, q91]:
                ax.plot([x_center - bw*0.2, x_center + bw*0.2], [cy, cy], color=col, lw=1.2, alpha=0.8)

        ax.set_ylim(row_ymin, row_ymax)
        ax.set_xlim(-0.8, n_events * GW - 0.6)
        ax.grid(True, axis='y', alpha=0.2)
        if ai == 1:
            ax.set_yticklabels([])
        else:
            ax.set_ylabel(var_name, fontsize=16)
        ax.text(0.015, 0.93, f'({next(panel_labels)})', transform=ax.transAxes, fontsize=16, fontweight='bold')

        if ri < len(MET_ROW_CONFIG):
            ax.set_xticks([])
        else:
            # bottom row: keep category labels, remove numeric ticks
            tick_x = [gi * GW + n_sensors / 2 - 0.5 for gi in range(n_events)]
            ax.set_xticks(tick_x)
            ax.set_xticklabels(EVENT_ORDER_COMBINED, fontsize=13)

# ── Shared sensor legend ─────────────────────────────
# Apply short_label to sensor names in the legend
sensor_handles = [mpatches.Patch(color=SENSOR_COLORS_F4C[l], alpha=0.7, label=short_label(l))
                   for l in ag_by_dist_f4c]
legend = fig.legend(handles=sensor_handles, fontsize=18, ncol=1,
                     loc='center left', bbox_to_anchor=(1.0, 0.78),
                     frameon=True, framealpha=0.9, edgecolor='#ccc', title='AG Sensors (PM₁ row only)')
legend.get_title().set_fontsize(18)

fig.tight_layout()
outname = 'figure_combined_SI.png' if INCLUDE_ALL_COLUMN else 'figure_combined_main.png'
fig.savefig(outname, dpi=SAVE_DPI, bbox_inches='tight')
plt.show()
print(f"Saved: {outname}")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import matplotlib.dates as mdates
import matplotlib.ticker as ticker

# Short display labels
def short_label(loc):
    """Convert full location name to compact label for heatmap y-axis."""
    replacements = {
        'AG-7 (IB)':             'R3',
        'AG-17 (IB City Hall)':  'I1',
        'AG-11 (IB Classroom)':  'I2',
    }
    return replacements.get(loc, loc)


# Define the selected AG sensors, outdoor sensor, and dates
ag_sensors_to_plot = ['AG-7 (IB)', 'AG-11 (IB Classroom)', 'AG-17 (IB City Hall)']
outdoor_sensor = 'QuantAQ ML'
dates_to_plot = {
    '2026-04-08': ('April 8',  'Top H₂S+PM₁'),
    #'2026-04-10': ('April 10', 'Top H₂S+PM₁'),
    #'2026-04-18': ('April 18', 'Top H₂S+PM₁'),
    '2026-04-24': ('April 24', 'Top H₂S+PM₁'),
    '2026-04-13': ('April 13', 'Clean reference'),
}

# Create a 3x3 subplot grid
fig, axes = plt.subplots(len(dates_to_plot), len(ag_sensors_to_plot),
                         figsize=(18, 15), sharey=True) # Removed sharex=True

for i, date_str in enumerate(dates_to_plot):
    # Get date_label and event_label for the current row
    date_label, event_label = dates_to_plot[date_str]

    start_of_day = pd.to_datetime(date_str)
    end_of_day   = start_of_day + pd.Timedelta(days=1) - pd.Timedelta(minutes=5)

    for j, ag_sensor in enumerate(ag_sensors_to_plot):
        ax = axes[i, j]

        # Filter indoor AG data for the specific sensor and date
        ag_pm1_day_data = ds_ag_filt['pm1_filtered'].sel(
            location=ag_sensor,
            time=slice(start_of_day, end_of_day)
        ).to_pandas()

        # Filter outdoor QuantAQ data for the specific sensor and date
        aq_pm1_day_data = ds_aq_clean['pm1'].sel(
            location=outdoor_sensor,
            time=slice(start_of_day, end_of_day)
        ).to_pandas()

        # Align outdoor PM1 to indoor timestamps if necessary
        # Using .reindex with method='nearest' and tolerance for robust alignment
        if not ag_pm1_day_data.empty and not aq_pm1_day_data.empty:
            outdoor_aligned = aq_pm1_day_data.reindex(
                ag_pm1_day_data.index,
                method='nearest',
                tolerance=pd.Timedelta('5min')
            )
        else:
            outdoor_aligned = pd.Series(index=ag_pm1_day_data.index, dtype=float)

        # Plot indoor AG data
        ax.plot(ag_pm1_day_data.index, ag_pm1_day_data.values, label='AirGradient',
                color='blue', linestyle='-', alpha=1, linewidth=2)

        # Plot outdoor QuantAQ data
        ax.plot(outdoor_aligned.index, outdoor_aligned.values, label='QuantAQ ML',
                color='red', linestyle='--', alpha=1, linewidth=2)

        # Titles and labels
        if i == 0: # Top row for column titles (AG sensor names)
            ax.set_title(f'{short_label(ag_sensor)}', fontsize=20)

        if j == 0: # Leftmost column for row titles (dates)
            ax.set_ylabel(f'{date_label}\n({event_label})', fontsize=20) # Changed to just show date, not 'PM1'

        # Set x-axis limits to cover the specific day
        ax.set_xlim(start_of_day, start_of_day + pd.Timedelta(days=1))

        # Set major ticks to every 6 hours, minor ticks to every hour
        ax.xaxis.set_major_locator(mdates.HourLocator(interval=6))
        ax.xaxis.set_minor_locator(mdates.HourLocator(interval=1))
        def pdt_formatter(x, pos):
            dt = mdates.num2date(x)
            local = dt
            return local.strftime('%H:%M')

        ax.xaxis.set_major_formatter(ticker.FuncFormatter(pdt_formatter))

        ax.grid(True, linestyle='--', alpha=0.6)
        # Only apply x-axis tick labels to the last row (i.e., third row when len(dates_to_plot) is 3)
        if i == len(dates_to_plot) - 1:
            ax.tick_params(axis='x', rotation=45, labelsize=16)
        else:
            ax.tick_params(axis='x', labelbottom=False)

        ax.tick_params(axis='y', labelsize=16)
        #ax.set_ylim(0, 30) # Set a consistent y-limit for better comparison

        # Add panel label (a, b, c, ...)
        panel_label = chr(97 + i * len(ag_sensors_to_plot) + j)
        ax.text(0.005, 0.93, f'({panel_label})', transform=ax.transAxes, fontsize=18, va='top', ha='left')

# Add a single legend outside the subplots
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper right', bbox_to_anchor=(0.95, 0.99),
           ncol=2, fontsize=20, frameon=True, edgecolor='black')

# Add super x and y labels
fig.supxlabel('Time', fontsize=20)
fig.supylabel('PM₁ (µg m⁻³)', fontsize=20)

plt.tight_layout(rect=[0.02, 0.03, 1, 0.95]) # Adjust layout to make room for suptitle and legend and new super labels

fig.savefig('specific_sensors.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()

In [ ]:
# @title
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
from matplotlib.lines import Line2D

# ── Cell 13: FIGURE 5 ─ Full-period correlation matrix ─
# Spearman. Indoor AG sensors (rows) vs outdoor QuantAQ (cols).
# AG sensors ordered closest → farthest from trailer.

def make_corr_matrix(y_ag_ds, y_quantaq_ds, x_quantaq_ds, y_sensor_order, x_locs, label=''):
    """
    Build Spearman correlation matrix between two sets of PM1 sensors.
    y_ag_ds: xarray Dataset for AirGradient sensors on y-axis (e.g., ds_ag_filt).
    y_quantaq_ds: xarray Dataset for QuantAQ sensors on y-axis (e.g., ds_aq_clean).
    x_quantaq_ds: xarray Dataset for QuantAQ sensors on x-axis (e.g., ds_aq_clean).
    y_sensor_order: list of sensor names for the y-axis.
    x_locs: list of sensor names for the x-axis.
    Returns DataFrame (y_sensors × x_sensors).
    """
    # Build DataFrame for y-axis sensors
    y_dfs = []
    for loc in y_sensor_order:
        if loc in y_ag_ds.location.values:
            # Use pm1_strict for AG sensors (pm1_filtered is also an option but pm1_strict is cleaner)
            y_dfs.append(y_ag_ds['pm1_filtered'].sel(location=loc).to_pandas().rename(loc))
        elif loc in y_quantaq_ds.location.values:
            # Use pm1 for QuantAQ sensors
            y_dfs.append(y_quantaq_ds['pm1'].sel(location=loc).to_pandas().rename(loc))
        else:
            print(f"Warning: Sensor {loc} not found in either AG or QuantAQ datasets for y-axis.")
    y_df = pd.concat(y_dfs, axis=1)

    # Build DataFrame for x-axis sensors (assuming all are QuantAQ)
    x_dfs = []
    for loc in x_locs:
        if loc in x_quantaq_ds.location.values:
            x_dfs.append(x_quantaq_ds['pm1'].sel(location=loc).to_pandas().rename(loc))
        else:
            print(f"Warning: Sensor {loc} not found in x_quantaq_ds for x-axis.")
    x_df = pd.concat(x_dfs, axis=1)

    # Ensure indices are DatetimeIndex and timezone-naive
    if not isinstance(y_df.index, pd.DatetimeIndex):
        y_df.index = pd.to_datetime(y_df.index)
    if y_df.index.tz is not None:
        y_df.index = y_df.index.tz_localize(None)

    if not isinstance(x_df.index, pd.DatetimeIndex):
        x_df.index = pd.to_datetime(x_df.index)
    if x_df.index.tz is not None:
        x_df.index = x_df.index.tz_localize(None)

    # Resample both to 5-min (QuantAQ is 1-min with non-round timestamps)
    y_df = y_df.resample('5min').mean()
    x_df = x_df.resample('5min').mean()

    # Align on common index
    common = y_df.index.intersection(x_df.index)
    y_df   = y_df.loc[common]
    x_df   = x_df.loc[common]

    corr = pd.DataFrame(index=y_sensor_order, columns=x_locs, dtype=float)
    for y_s in y_sensor_order:
        if y_s not in y_df.columns: continue
        for x_s in x_locs:
            if x_s not in x_df.columns: continue
            mask = y_df[y_s].notna() & x_df[x_s].notna()
            if mask.sum() < 30: continue
            r, _ = stats.spearmanr(y_df[y_s][mask], x_df[x_s][mask])
            corr.loc[y_s, x_s] = r
    return corr.astype(float)

def correlation_to_stars(corr_df):
    """
    Converts a DataFrame of correlation coefficients to a DataFrame of star ratings.
    '***': strong correlation (abs(r) >= 0.7)
    '**': moderate correlation (abs(r) >= 0.5)
    '*': weak correlation (abs(r) >= 0.3)
    '': very weak or no correlation (abs(r) < 0.3)
    """
    star_df = corr_df.applymap(lambda r:
        '***' if abs(r) >= 0.7 else
        '**'  if abs(r) >= 0.5 else
        '*'   if abs(r) >= 0.3 else
        ''
    )
    return star_df

# X-axis: only 'QuantAQ ML'
QAQ_X_SENSORS_F5 = ['QuantAQ ML']

# Y-axis: QuantAQ sensors (excluding ML and SY) and indoor AG sensors
# The variable AG_ORDER_F5 was previously set to exclude 'QuantAQ SY' and 'QuantAQ ML'
QAQ_Y_SENSORS_F5 = [loc for loc in list(QUANTAQ_FILES.keys()) if loc != 'QuantAQ ML' and loc != 'QuantAQ SY']

# Indoor AG sensors, excluding AG-6 (sparse data), AG-3 (SY) and AG-9 (SY)
INDOOR_AG_SENSORS_F5 = [loc for loc in ag_by_dist if 'AG-6' not in loc and 'AG-3 (SY)' not in loc and 'AG-9 (SY)' not in loc]

# Combine all sensor locations for sorting
ALL_SENSOR_LOCATIONS = {**AG_LOCATIONS, **QUANTAQ_LOCATIONS}

# Define a unified distance function that works for both AG and QuantAQ sensors
def dist_to_hotspot(loc_name):
    coords = ALL_SENSOR_LOCATIONS.get(loc_name)
    if coords is None:
        # Assign a very large distance if coordinates are not found, effectively placing it last
        return float('inf')
    # TRAILER_COORDS is defined in Cell 8 as AG_LOCATIONS['AG-19 (ML)']
    return ((coords[0]-TRAILER_COORDS[0])**2 + (coords[1]-TRAILER_COORDS[1])**2)**0.5

# Create a combined list of all Y-axis sensors and sort them by distance from the hotspot
# Removed QAQ_Y_SENSORS_F5 to exclude QuantAQ sensors from the Y-axis
ALL_Y_SENSORS_UNSORTED = INDOOR_AG_SENSORS_F5
Y_SENSORS_COMBINED = sorted(ALL_Y_SENSORS_UNSORTED, key=dist_to_hotspot)

# Adjust figure size based on the number of y-axis sensors
num_y_sensors = len(Y_SENSORS_COMBINED)
fig_height = max(7, num_y_sensors * 0.7) # Minimum height of 7, approximately 0.7 inches per sensor

# Update the correlation matrix call
corr_full = make_corr_matrix(ds_ag_filt, ds_aq_clean, ds_aq_clean, Y_SENSORS_COMBINED, QAQ_X_SENSORS_F5)
star_annotations = correlation_to_stars(corr_full) # Generate star annotations

fig5, ax5 = plt.subplots(figsize=(11, fig_height))
sns.heatmap(corr_full, ax=ax5, annot=star_annotations, fmt='', # Use star_annotations for annot
            cmap='YlOrRd', vmin=0, vmax=1,
            linewidths=0.5, linecolor='#eeeeee',
            cbar_kws={'label': 'Spearman ρ', 'shrink': 0.85},
            annot_kws={'color': 'black', 'fontsize': 16}) # Added annot_kws for color and size
ax5.set_xlabel('Outdoor QuantAQ ML', fontsize=16)
ax5.set_ylabel('Indoor AG Sensors', fontsize=16) # Updated y-axis label
ax5.tick_params(axis='x', rotation=0, labelsize=12)
ax5.tick_params(axis='y', rotation=0,  labelsize=12)
ax5.invert_yaxis() # Flip the y-axis

fig5.tight_layout()
fig5.savefig('figure5_correlation_full.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()

In [ ]:
# @title
# Identify 'High H₂S + PM₁' days
all_study_dates = pd.date_range(PLOT_START, PLOT_END, freq='D')
high_h2s_pm1_dates = [
    d.strftime('%Y-%m-%d') for d in all_study_dates if classify_day(d) == 'both'
]

print(f"Identified {len(high_h2s_pm1_dates)} 'High H₂S + PM₁' days:")
for day in high_h2s_pm1_dates:
    print(f"  - {day}")

# Filter AirGradient data for identified event days
ds_ag_filt_event = ds_ag_filt.where(
    ds_ag_filt.time.dt.strftime('%Y-%m-%d').isin(high_h2s_pm1_dates),
    drop=True
)

# Filter QuantAQ data for identified event days
ds_aq_clean_event = ds_aq_clean.where(
    ds_aq_clean.time.dt.strftime('%Y-%m-%d').isin(high_h2s_pm1_dates),
    drop=True
)

print(f"Filtered AG data for {len(ds_ag_filt_event.time)} time points.")
print(f"Filtered QuantAQ data for {len(ds_aq_clean_event.time)} time points.")

In [ ]:
# ─── Build aligned sensor DataFrame from ds_ag_filt ───────────────────────────
# Indoor PM1 (strict filtered)
indoor_strict = pd.DataFrame({
    loc: ds_ag_filt['pm1_filtered'].sel(location=loc).to_pandas()
    for loc in ds_ag_filt.location.values
})
indoor_strict.index = pd.DatetimeIndex(indoor_strict.index).tz_localize(None)

# Outdoor PM1 (QuantAQ Trailer, resampled to 5-min)
outdoor = (ds_aq_clean['pm1']
           .sel(location='QuantAQ ML')
           .to_pandas())
outdoor.index = pd.DatetimeIndex(outdoor.index).tz_localize(None)
outdoor = outdoor.resample('5min').mean()

# Align
common_idx   = indoor_strict.index.intersection(outdoor.index)
indoor_strict = indoor_strict.loc[common_idx]
outdoor       = outdoor.loc[common_idx]

# Date and event labels
dates        = pd.Series(common_idx).dt.date.astype(str).values
event_labels = np.array([get_day_event(d) for d in dates])   # from Cell 8

# Sensors to analyse (exclude AG-6 no data, AG-19 outdoor ref)
STAT_SENSORS = [l for l in ds_ag_filt.location.values
                if 'AG-6' not in l]

# ─── ANALYSIS 1: Mann-Whitney U ──────────────────────────────────────────────
# One-sided test: indoor PM1 during High PM1 event days > clean reference days
print("=" * 75)
print("ANALYSIS 1 ─ Mann-Whitney U: H₂S + PM₁ event days vs clean reference")
print("One-sided test: event > clean  |  Filtered indoor PM₁")
print("=" * 75)
print(f"{'Sensor':<30} {'n_clean':>8} {'n_event':>8} "
      f"{'med_clean':>10} {'med_event':>10} {'p-value':>12} {'sig':>5}")
print("-" * 80)

mw_records = []
for loc in STAT_SENSORS:
    clean_vals = indoor_strict.loc[event_labels == 'Clean', loc].dropna()
    event_vals = indoor_strict.loc[event_labels == 'High H₂S + PM₁', loc].dropna()
    # Debugging print statement to see counts
    # print(f"  {loc:<30}: Clean values: {len(clean_vals)}, Event values: {len(event_vals)}")
    if len(clean_vals) < 10 or len(event_vals) < 10:
        # print(f"  Skipping {loc} due to insufficient data.")
        continue
    u, p = scipy_stats.mannwhitneyu(event_vals, clean_vals, alternative='greater')
    sig  = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    mw_records.append({
        'sensor':     loc,
        'n_clean':    len(clean_vals),
        'n_event':    len(event_vals),
        'med_clean':  round(float(clean_vals.median()), 3),
        'med_event':  round(float(event_vals.median()), 3),
        'U_stat':     round(float(u), 0),
        'p_value':    float(p),
        'sig':        sig,
    })
    print(f"{loc:<30} {len(clean_vals):>8} {len(event_vals):>8} "
          f"{clean_vals.median():>10.3f} {event_vals.median():>10.3f} "
          f"{p:>12.3e} {sig:>5}")

mw_df = pd.DataFrame(mw_records)

# Also test High H2S vs clean ─ expect no significant difference
print()
print("High H₂S event days vs clean (expect no difference):")
print(f"{'Sensor':<30} {'med_clean':>10} {'med_h2s':>10} {'p-value':>12} {'sig':>5}")
print("-" * 60)
for loc in STAT_SENSORS:
    clean_vals = indoor_strict.loc[event_labels == 'Clean',    loc].dropna()
    h2s_vals   = indoor_strict.loc[event_labels == 'High H₂S', loc].dropna()
    if len(clean_vals) < 10 or len(h2s_vals) < 10:
        continue
    _, p = scipy_stats.mannwhitneyu(h2s_vals, clean_vals, alternative='greater')
    sig  = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f"{loc:<30} {clean_vals.median():>10.3f} {h2s_vals.median():>10.3f} "
          f"{h2s_vals.median():>10.3f} {p:>12.3e} {sig:>5}") # Changed h2s_vals.median() to print for med_h2s column

# ─── ANALYSIS 1b: Paired T-test ──────────────────────────────────────────────
# Paired samples t-test: indoor PM1 during High H₂S + PM₁ event days vs clean reference days
print("\n" + "=" * 75)
print("ANALYSIS 1b ─ Paired T-test: H₂S + PM₁ event days vs clean reference")
print("Paired samples t-test: event ≠ clean  |  Filtered indoor PM₁")
print("=" * 75)
print(f"{'Sensor':<30} {'n_pairs':>8} {'mean_clean':>10} {'mean_event':>10} {'p-value':>12} {'sig':>5}")
print("-" * 80)

paired_t_test_records = []
for loc in STAT_SENSORS:
    # Get all 5-min data for clean and event days for the current sensor
    all_clean_data_for_sensor = indoor_strict.loc[event_labels == 'Clean', loc].dropna()
    all_event_data_for_sensor = indoor_strict.loc[event_labels == 'High H₂S + PM₁', loc].dropna()

    # Group by time of day and calculate the mean for each 5-min interval
    clean_daily_profile = all_clean_data_for_sensor.groupby(all_clean_data_for_sensor.index.time).mean()
    event_daily_profile = all_event_data_for_sensor.groupby(all_event_data_for_sensor.index.time).mean()

    # Align these daily profiles on their time-of-day index
    paired_profiles_df = pd.DataFrame({
        'clean': clean_daily_profile,
        'event': event_daily_profile
    }).dropna()

    if len(paired_profiles_df) < 2: # Paired t-test requires at least 2 pairs
        continue

    # Perform paired samples t-test
    # Alternative='two-sided' to test for significant difference in either direction
    t_stat, p_val = scipy_stats.ttest_rel(paired_profiles_df['event'], paired_profiles_df['clean'], alternative='two-sided')

    # Two-sided significance
    sig  = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'

    paired_t_test_records.append({
        'sensor':     loc,
        'n_pairs':    len(paired_profiles_df),
        'mean_clean': round(float(paired_profiles_df['clean'].mean()), 3),
        'mean_event': round(float(paired_profiles_df['event'].mean()), 3),
        't_stat':     round(float(t_stat), 3),
        'p_value':    float(p_val),
        'sig':        sig,
    })
    print(f"{loc:<30} {len(paired_profiles_df):>8} "
          f"{paired_profiles_df['clean'].mean():>10.3f} {paired_profiles_df['event'].mean():>10.3f} "
          f"{p_val:>12.3e} {sig:>5}")

paired_t_test_df = pd.DataFrame(paired_t_test_records)


# ─── ANALYSIS 2: Spearman lag correlation ────────────────────────────────────
# Outdoor PM1 lagged by 0-24h vs indoor PM1
# Identifies the time delay between outdoor pollution and indoor infiltration
print()
print("=" * 75)
print("ANALYSIS 2 ─ Spearman ρ: outdoor PM₁ (lagged) vs indoor PM₁ (filtered)")
print("Lag = time outdoor PM₁ leads indoor PM₁ (hours)")
print("=" * 75)

LAGS_H     = [0, 1, 2, 4, 6, 8, 12, 24]
LAGS_STEPS = [int(l * 60 / 5) for l in LAGS_H]   # convert to 5-min steps

# Header
print(f"{'Sensor':<30}", end='')
for l in LAGS_H:
    print(f"  {l:>3}h", end='')
print("  | peak_ρ  lag  sig")
print("-" * 90)

lag_records = []
for loc in STAT_SENSORS:
    ind = indoor_strict[loc].dropna()
    print(f"{loc:<30}", end='')
    best_r = 0; best_lag = 0; best_p = 1
    row = {'sensor': loc}

    for lag_h, lag_steps in zip(LAGS_H, LAGS_STEPS):
        out_lagged = outdoor.shift(lag_steps)
        both = pd.concat([ind, out_lagged], axis=1).dropna()
        both.columns = ['indoor', 'outdoor']
        if len(both) < 30:
            print(f"   N/A", end='')
            row[f'lag_{lag_h}h'] = np.nan
            continue
        r, p = scipy_stats.spearmanr(both['indoor'], both['outdoor'])
        print(f"  {r:>4.2f}", end='')
        row[f'lag_{lag_h}h'] = round(r, 3)
        if r > best_r:
            best_r = r; best_lag = lag_h; best_p = p

    sig = '***' if best_p < 0.001 else '**' if best_p < 0.01 else '*' if p < 0.05 else 'ns'
    row.update({'peak_r': round(best_r, 3), 'peak_lag_h': best_lag,
                'peak_p': best_p, 'sig': sig})
    lag_records.append(row)
    print(f"  | {best_r:>5.3f}  {best_lag:>3}h  {sig}")

lag_df = pd.DataFrame(lag_records)

print()
print("=" * 75)
print("SUMMARY")
print("=" * 75)
display(lag_df[['sensor','lag_0h','lag_1h','lag_2h','lag_4h',
                'lag_6h','lag_8h','lag_12h','lag_24h',
                'peak_r','peak_lag_h','sig']])


# TOXICITY POTENTIAL FIGURE

In [ ]:
WHO_PM25_TP    = 15.0   # µg/m³ — WHO 24-h PM2.5 guideline
DT_HOURS_TP    = 5 / 60 # hours per 5-min interval

# ── Dynamically determine SENSOR_ORDER_TP and sort by distance from ML ──────
# Places sensors by their proximity to ML (closest first), as requested.

# Identify all sensors that have 'All' event type and 'Full day' period data
sensors_with_all_data = tp_df[
    (tp_df['event_type'] == 'All') &
    (tp_df['period'] == 'Full day')
]['sensor'].unique().tolist()

# Filter ag_by_dist to include only sensors present in tp_df, excluding AG-3 and AG-9
# ag_by_dist is already sorted by distance from ML (from Cell 8)
SENSOR_ORDER_TP = [s for s in ag_by_dist if s in sensors_with_all_data and s not in ['AG-3 (SY)', 'AG-9 (SY)']]

# EVENT CATEGORIES AND PLOTTING STYLES
EVENT_CATS_TP = ['Clean', 'High H₂S + PM₁', 'All']
EVENT_COLS_TP = {
    'Clean':           'blue',
    'High H₂S + PM₁':  'purple',
    'All':             'red', # New color for 'All'
}
EVENT_MARKS_TP = {
    'Clean':           '^',
    'High H₂S + PM₁':  '^',
    'All':             's', # Changed marker for 'All' to square for distinctiveness
}
OFFSETS_TP = {'Clean': -0.22, 'High H₂S + PM₁': 0, 'All': 0.22} # New offset for 'All'

# Short display labels
def short_label(loc):
    """Convert full location name to compact label for heatmap y-axis."""
    replacements = {
        'AG-19 (ML)':    'R1 (0.716 km)',
        'AG-15 (IB)':            'R2 (2.068 km)',
        'AG-7 (IB)':             'R3 (2.530 km)',
        'AG-17 (IB City Hall)':  'I1 (2.899 km)',
        'AG-11 (IB Classroom)':  'I2 (3.557 km)',
        'AG-12 (IB)':            'R4 (4.429 km)',
        'AG-13 (TR Estuary)':    'I3 (3.607 km)',
        'AG-6 (IB Classroom)':   'AG-6 IB Classroom (2.478 km)',
    }
    return replacements.get(loc, loc)

# ── Compute daily mean Tp per sensor per event type ─────────────
# Step 1: add event label and C_tr_pm25 to decomp_df
# decomp_df already has C_tr (PM1-based) from compute_decomposition()
# For PM2.5 we multiply by outdoor pm25/pm1 ratio or use outdoor pm25 directly

# Get outdoor PM25 from QuantAQ Trailer
out_pm25_ser = (
    ds_aq_clean['pm25']
    .sel(location='QuantAQ ML')
    .to_pandas()
)
out_pm25_ser.index = pd.DatetimeIndex(out_pm25_ser.index).tz_localize(None)
out_pm25_ser = out_pm25_ser.resample('5min').mean()

# Build a working copy of decomp_df with event labels and C_tr_pm25
tp_work = decomp_df.copy()
tp_work['event_type'] = tp_work['date'].apply(get_day_event)
tp_work = tp_work[tp_work['event_type'].notna()].copy()

# C_tr_pm25 = F × outdoor_pm25
# Use INFILTRATION_FACTORS computed in Cell 8
tp_work['dt_aligned'] = pd.to_datetime(tp_work['dt']).dt.floor('5min')
out_pm25_dict = out_pm25_ser.to_dict()
tp_work['out_pm25'] = tp_work['dt_aligned'].map(out_pm25_dict)

# Fall back to out_pm1 * 1.2 if pm25 not available at a given timestamp
tp_work['out_pm25'] = tp_work['out_pm25'].fillna(tp_work['out_pm1'] * 1.2)
# Use INFILTRATION_FACTORS_PM25 for C_tr_pm25 calculation
tp_work['C_tr_pm25'] = tp_work.apply(
    lambda r: max(0, INFILTRATION_FACTORS_PM25.get(r['sensor'], 0) * r['out_pm25'])
    if pd.notna(r['out_pm25']) else np.nan, axis=1)

# Step 2: daily mean C_tr_pm25 per sensor per day for Clean, High H₂S, High H₂S + PM₁
daily_ctr = (
    tp_work
    .groupby(['sensor', 'date', 'event_type'])['C_tr_pm25']
    .mean()
    .reset_index()
)
daily_ctr['tp_daily'] = daily_ctr['C_tr_pm25'] / WHO_PM25_TP

# Step 3: mean and SD of daily Tp across days within each event type (Clean, High H₂S + PM₁)
tp_event_data = (
    daily_ctr
    .groupby(['sensor', 'event_type'])['tp_daily']
    .agg(['mean', 'std', 'count'])
    .reset_index()
)
tp_event_data.columns = ['sensor', 'event_type', 'tp_mean', 'tp_std', 'n_days']
tp_event_data['tp_std'] = tp_event_data['tp_std'].fillna(0)

# Filter tp_event_data for the desired event types and sensors
tp_event_data = tp_event_data[
    tp_event_data['event_type'].isin(['Clean', 'High H₂S + PM₁']) &
    tp_event_data['sensor'].isin(SENSOR_ORDER_TP)
].copy()

# Get 'All' study period data from tp_df (already calculated for full study period, full day)
# Filter tp_df for 'All' event type, 'Full day' period, and sensors in SENSOR_ORDER_TP
tp_all_data = tp_df[
    (tp_df['event_type'] == 'All') &
    (tp_df['period'] == 'Full day') &
    (tp_df['sensor'].isin(SENSOR_ORDER_TP))
][
    ['sensor', 'event_type', 'tp_mean', 'tp_std', 'n_days']
].copy()

# Set the event_type to 'All' for these rows to match the new category name
tp_all_data['event_type'] = 'All'

# Combine these two dataframes
tp_plot_data = pd.concat([tp_event_data, tp_all_data], ignore_index=True)


# Print summary for reference
print("Daily mean Tp by sensor and selected event types:")
pivot = tp_plot_data.pivot_table(
    index='sensor', columns='event_type', values='tp_mean')
available_et_plot = [e for e in EVENT_CATS_TP if e in pivot.columns]
display(pivot[available_et_plot].round(4))

# Print Tp values and standard deviations
print("\nTp Values (Mean ± Std Dev):")
for sensor in SENSOR_ORDER_TP:
    print(f"  Sensor: {short_label(sensor)}")
    for et in EVENT_CATS_TP:
        row = tp_plot_data[
            (tp_plot_data['sensor'] == sensor) &
            (tp_plot_data['event_type'] == et)
        ]
        if not row.empty:
            mean_val = row['tp_mean'].values[0]
            std_val = row['tp_std'].values[0]
            print(f"    {et}: {mean_val:.4f} \u00B1 {std_val:.4f}")

# ── Build figure ─────────────────────────────
fig_tp, ax_tp = plt.subplots(figsize=(11, 6))
fig_tp.subplots_adjust(left=0.22, right=0.96, top=0.88, bottom=0.14)

y_pos = np.arange(len(SENSOR_ORDER_TP))

for et in EVENT_CATS_TP:
    col  = EVENT_COLS_TP[et]
    mark = EVENT_MARKS_TP[et]
    off  = OFFSETS_TP[et]

    for yi, sensor in enumerate(SENSOR_ORDER_TP):
        row = tp_plot_data[
            (tp_plot_data['sensor']     == sensor) &
            (tp_plot_data['event_type'] == et)
        ]
        if len(row) == 0:
            continue

        m = float(row['tp_mean'].values[0])
        s = float(row['tp_std'].values[0])
        y = y_pos[yi] + off

        # Stem from zero to mean
        ax_tp.plot([0, m], [y, y], color=col, lw=1.5, alpha=0.7, zorder=2)

        # ±1 SD bar across days (shows day-to-day variability within category)
        ax_tp.plot([max(0, m - s), m + s], [y, y],
                   color=col, lw=3.0, alpha=0.35,
                   solid_capstyle='round', zorder=3)

        # Lollipop head
        ax_tp.scatter(m, y, color=col, s=65, marker=mark,
                      zorder=5, edgecolors='white', lw=0.7)

# Horizontal grid lines
for yi in y_pos:
    ax_tp.axhline(yi, color='#eeeeee', lw=0.6, zorder=0)

# Y axis
ax_tp.set_yticks(y_pos)
ax_tp.set_yticklabels([short_label(s) for s in SENSOR_ORDER_TP], fontsize=14)
ax_tp.set_ylim(-0.6, len(SENSOR_ORDER_TP) - 0.4)
ax_tp.set_ylabel('Sensor Location', fontsize=16)

#x axis
ax_tp.set_xlabel('Toxicity Potential', fontsize=16)

# Legend - updated to use the new EVENT_CATS_TP
legend_labels = {
    'Clean': 'Clean',
    'High H₂S + PM₁': 'High H₂S + PM₁',
    'All': 'Entire Study Period'
}

leg_handles = [
    plt.scatter([], [], color=EVENT_COLS_TP[et], s=65,
                marker=EVENT_MARKS_TP[et], edgecolors='white',
                lw=0.5, label=legend_labels[et])
    for et in EVENT_CATS_TP
]
leg_handles.append(mpatches.Patch(
    facecolor='#888888', alpha=0.35,
    label='±1 SD across days in category'))
ax_tp.legend(handles=leg_handles, fontsize=14,
             loc='center right', frameon=True,
             framealpha=0.9, edgecolor='#ccc')


fig_tp.savefig('figure_tp_daily.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()
print("Tp figure saved.")

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

# Helper function to resample a UTC series to local PDT daily values
# This is a modified version of the one in Cell 8 to handle xarray objects and already-aligned series
def resample_local_daily_wrapper(data_series_or_xarray, agg='mean', is_utc=True):
    if isinstance(data_series_or_xarray, xr.DataArray):
        series_utc = data_series_or_xarray.to_pandas()
    else:
        series_utc = data_series_or_xarray.copy()

    if is_utc:
        # Ensure index is datetime and then apply PDT offset for local day boundaries
        if series_utc.index.tz is not None:
            series_utc.index = series_utc.index.tz_localize(None) # Remove timezone for simplicity in offset
        local = series_utc.copy()
        local.index = local.index - PDT_OFFSET   # shift to local time (PDT = UTC-7)
    else: # Already in local time (PDT-aligned)
        local = series_utc.copy()
        if local.index.tz is not None:
            local.index = local.index.tz_localize(None) # Remove timezone for consistency

    if agg == 'mean':
        return local.resample('D').mean()
    elif agg == 'max':
        return local.resample('D').max()
    elif agg == 'min':
        return local.resample('D').min()
    else:
        raise ValueError("Aggregation method not supported.")

# Function for circular mean of wind direction
def circular_mean_daily(series_utc, is_utc=True):
    if is_utc:
        if series_utc.index.tz is not None:
            series_utc.index = series_utc.index.tz_localize(None)
        local = series_utc.copy()
        local.index = local.index - PDT_OFFSET
    else:
        local = series_utc.copy()
        if local.index.tz is not None:
            local.index = local.index.tz_localize(None)

    daily_sin = np.sin(np.deg2rad(local)).resample('D').mean()
    daily_cos = np.cos(np.deg2rad(local)).resample('D').mean()
    return np.rad2deg(np.arctan2(daily_sin, daily_cos)) % 360

print("Helper functions for daily resampling loaded.")

In [ ]:
# Retrieve relevant data for calculation
# H2S data (from Cell 8, already UTC, use for daily mean)
h2s_trailer_series_utc = h2s_trailer['h2s']

# Outdoor PM1 data (from Cell 5)
out_pm1_trailer_raw_utc = ds_aq_clean['pm1'].sel(location='QuantAQ ML')

# Outdoor Wind data (from Cell 5)
wind_dir_trailer_raw_utc = ds_aq_clean['wind_dir'].sel(location='QuantAQ ML')
wind_speed_trailer_raw_utc = ds_aq_clean['wind_speed'].sel(location='QuantAQ ML')

# Outdoor RH data (from Cell 5)
rh_trailer_raw_utc = ds_aq_clean['rh'].sel(location='QuantAQ ML')

# MLH data (from Cell 5)
mlh_raw_utc = mlh['mlh']

# River stage data (from Cell 5)
river_lag6h_raw_utc = river['river_stage_lag6h']

# AG PM1 data (from Cell 7)
ag_pm1_strict_utc = ds_ag_filt['pm1_strict']

# PDT Offset (from Cell 8)
PDT_OFFSET = pd.Timedelta(hours=7)

print("Raw data series collected.")

In [ ]:
# Calculate daily statistics for each variable in PDT

# Calculate 1-hour rolling mean for outdoor PM1
pm1_1h_rolling_mean = out_pm1_series.rolling('1H', min_periods=1).mean()
# Calculate daily maximum of the 1-hour rolling mean
pm1_1h_daily_max = resample_local_daily_wrapper(pm1_1h_rolling_mean, agg='max', is_utc=False)

# 1. Max H2S hourly average at trailer (PDT) - already available from Cell 8
daily_max_h2s_hourly_avg = h2s_daily_mean.rename('Avg H2S Daily Mean (ppb)')

# New: Max H2S 5-min spike at trailer (PDT)
daily_max_h2s_5min_spike = resample_local_daily_wrapper(h2s_trailer_series_utc, agg='max', is_utc=False).rename('Max H2S 5-min Spike (ppb)')

# 2. Average daily H2S at trailer (PDT)
daily_avg_h2s = resample_local_daily_wrapper(h2s_trailer_series_utc, agg='mean', is_utc=False).rename('Avg H2S (ppb)')

# 3. Max PM1 1-hour rolling mean at trailer (PDT) - already available from Cell 8
daily_max_pm1_1hr_roll_mean = pm1_1h_daily_max.rename('Max PM1 1hr Roll Mean ML (µg m⁻³)')

# New: Max PM1 5-min spike at trailer (PDT)
daily_max_pm1_5min_spike = resample_local_daily_wrapper(out_pm1_series, agg='max', is_utc=False).rename('Max PM1 5-min Spike ML (µg m⁻³)')

# 4. Average daily PM1 at trailer (PDT) - already available from Cell 8
daily_avg_pm1_trailer = trailer_pm1_daily.rename('Avg PM1 ML (µg m⁻³)')

# 5. Average daily PM1 at AirGradient locations (PDT)
daily_avg_ag_pm1 = pd.DataFrame()
for loc in ag_pm1_strict_utc.location.values:
    ag_loc_series = ag_pm1_strict_utc.sel(location=loc)
    daily_avg_ag_pm1[loc] = resample_local_daily_wrapper(ag_loc_series, agg='mean', is_utc=True)
daily_avg_ag_pm1 = daily_avg_ag_pm1.add_prefix('Avg PM1 ')

# 6. MLH (PDT) - originally UTC, convert to local then daily mean
daily_avg_mlh = resample_local_daily_wrapper(mlh_raw_utc, agg='mean', is_utc=True).rename('Avg MLH (m)')

# 7. River stage 6 hour lag (PDT) - originally UTC, convert to local then daily mean
daily_avg_river = resample_local_daily_wrapper(river_lag6h_raw_utc, agg='mean', is_utc=True).rename('Avg River Stage 6h Lag (m)')

# 8. Average daily wind direction at trailer (PDT) - circular mean
daily_avg_wind_dir = circular_mean_daily(wind_dir_trailer_raw_utc.to_pandas(), is_utc=True).rename('Avg Wind Dir ML (°)')

# 9. Average daily wind speed at trailer (PDT)
daily_avg_wind_speed = resample_local_daily_wrapper(wind_speed_trailer_raw_utc, agg='mean', is_utc=True).rename('Avg Wind Speed ML (m/s)')

# 10. Average daily RH at trailer (PDT)
daily_avg_rh = resample_local_daily_wrapper(rh_trailer_raw_utc, agg='mean', is_utc=True).rename('Avg RH ML (%)}')

print("All daily statistics calculated.")

# EVENT AND CLEAN DAY SUMMARY TABLE

In [ ]:
import pandas as pd
import numpy as np

# Compile the summary table

# Define EVENT_DAYS to be all study dates for the daily summary
all_study_dates = pd.date_range(PLOT_START, PLOT_END, freq='D')
EVENT_DAYS = all_study_dates.normalize() # Ensure dates are consistent

# EVENT_DAYS is already available from Cell 11 and contains UTC Timestamps
# Convert to date strings for consistent indexing with daily stats
event_dates_pdt = pd.to_datetime(EVENT_DAYS).date

summary_data = []
for date_obj in event_dates_pdt:
    date_str = str(date_obj)
    event_type = get_day_event(date_str) # from Cell 8

    row_data = {'Date': date_str, 'Event Type': event_type}

    # Add max H2S hourly avg
    row_data['Max H2S Hourly Avg (ppb)'] = daily_max_h2s_hourly_avg.get(date_str, np.nan)

    # Add max H2S 5-min spike
    row_data['Max H2S 5-min Spike (ppb)'] = daily_max_h2s_5min_spike.get(date_str, np.nan)

    # Add avg H2S
    row_data['Avg H2S (ppb)'] = daily_avg_h2s.get(date_str, np.nan)

    # Add max PM1 1hr roll mean trailer
    row_data['Max PM1 1hr Roll Mean ML (µg m⁻³)'] = daily_max_pm1_1hr_roll_mean.get(date_str, np.nan)

    # Add max PM1 5-min spike trailer
    row_data['Max PM1 5-min Spike ML (µg m⁻³)'] = daily_max_pm1_5min_spike.get(date_str, np.nan)

    # Add avg PM1 trailer
    row_data['Avg PM1 ML (µg m⁻³)'] = trailer_pm1_daily.get(date_str, np.nan)

    # Add avg PM1 for AG locations
    for col in daily_avg_ag_pm1.columns:
        row_data[col] = daily_avg_ag_pm1[col].get(date_str, np.nan)

    # Add MLH
    row_data['Avg MLH (m)'] = daily_avg_mlh.get(date_str, np.nan)

    # Add River Stage
    row_data['Avg River Stage 6h Lag (m)'] = daily_avg_river.get(date_str, np.nan)

    # Add Wind Direction
    row_data['Avg Wind Dir ML (°)'] = daily_avg_wind_dir.get(date_str, np.nan)

    # Add Wind Speed
    row_data['Avg Wind Speed ML (m/s)'] = daily_avg_wind_speed.get(date_str, np.nan)

    # Add RH
    row_data['Avg RH ML (%)'] = daily_avg_rh.get(date_str, np.nan)

    summary_data.append(row_data)

summary_df = pd.DataFrame(summary_data)

# Save to CSV
output_filename_summary = 'daily_summary_stats.csv'
summary_df.to_csv(output_filename_summary, index=False)

print("Summary table generated:")
display(summary_df)
print(f"Summary data saved to '{output_filename_summary}'.")

In [ ]:
num_hourly_spikes_over_30 = (daily_max_h2s_hourly_avg > H2S_HOURLY_THRESH).sum()
print(f"Number of hourly H2S spikes over {H2S_HOURLY_THRESH} ppb: {num_hourly_spikes_over_30}")

In [ ]:
# @title
"""
SI Figure S5 — Infiltration Factor Regression Plots
Per Wallace and Ott (2023):
- X axis: outdoor PM2.5 (QuantAQ PML)
- Y axis: indoor PM2.5 raw (pm25_strict) — NOT background-subtracted on axes
- Regression line: Y_indoor = F*X + C_background  (slope=F, intercept=bg)
- Forbidden Zone: region BELOW Y = F*X (through origin, no intercept)
  Points below this line imply negative indoor-generated PM2.5 —
  physically impossible if F is truly constant (Wallace & Ott 2023).
  Used as a DIAGNOSTIC, not a filter.
- Point categories:
    Blue circles  = used in regression (outdoor > threshold, not cooking, Y >= F*X)
    Red triangles = cooking-flagged (excluded from regression)
    Grey squares  = Forbidden Zone violations (Y < F*X, not cooking-flagged)
"""

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

OUT_PM25_THRESH = 9.0

# F values and backgrounds pulled directly from Cell 8 — do not hardcode
# INFILTRATION_FACTORS_PM25 and SENSOR_BACKGROUND_PM25 must be defined
# in Cell 8 before running this cell
SENSORS = list(INFILTRATION_FACTORS_PM25.keys())

BLUE = '#1a5fa8'
RED  = '#cc2222'
GREY = '#999999'
FZ_COL = '#f5ead5'

fig, axes = plt.subplots(4, 2, figsize=(14, 22))
fig.subplots_adjust(left=0.08, right=0.97, top=0.92,
                    bottom=0.08, hspace=0.45, wspace=0.28)

out_pm25_ser = (ds_aq_clean['pm25']
                .sel(location='QuantAQ ML')
                .to_pandas())
out_pm25_ser.index = pd.DatetimeIndex(out_pm25_ser.index).tz_localize(None)
out_pm25_ser = out_pm25_ser.resample('5min').mean()

summary = []

for ax, loc in zip(axes.flat, SENSORS):
    F  = INFILTRATION_FACTORS_PM25[loc]
    bg = SENSOR_BACKGROUND_PM25[loc]

    ind = ds_ag_filt_pm25['pm25_strict'].sel(location=loc).to_pandas()
    ind.index = pd.DatetimeIndex(ind.index).tz_localize(None)

    raw25 = ds_ag_filt_pm25['pm25'].sel(location=loc).to_pandas()
    raw25.index = pd.DatetimeIndex(raw25.index).tz_localize(None)

    out = out_pm25_ser.reindex(ind.index, method='nearest',
                               tolerance=pd.Timedelta('6min'))

    # Cooking flag = raw not NaN but strict is NaN
    ck_flag = raw25.notna() & ind.isna()

    df = pd.DataFrame({
        'y_strict': ind.values,
        'y_raw':    raw25.values,
        'x':        out.values,
        'ck':       ck_flag.values,
    }, index=ind.index).dropna(subset=['x'])

    x  = df['x'].values
    ys = df['y_strict'].values   # NaN where cooking-flagged
    yr = df['y_raw'].values
    ck = df['ck'].values

    # Outdoor-dominated, not cooking-flagged, strict not NaN
    od = (x > OUT_PM25_THRESH) & ~ck & np.isfinite(ys)

    # Forbidden Zone: od points where indoor < F*outdoor
    fz = od & (ys < F * x)

    # Regression points: od, not FZ, background-subtracted Y >= 0
    reg = od & ~fz & ((ys - bg) >= 0)

    # Cooking: raw available
    ck_od = ck & (x > 0) & np.isfinite(yr)

    n_od  = od.sum()
    n_fz  = fz.sum()
    n_reg = reg.sum()
    fz_pct = n_fz / max(n_od, 1) * 100

    # Regression
    if n_reg >= 30:
        y_adj = ys[reg] - bg
        sl, ic, r, _, se = stats.linregress(x[reg], y_adj)
        r2 = r**2
    else:
        sl, ic, r2, se = F, 0, np.nan, np.nan

    summary.append((loc, n_reg, n_fz, fz_pct, sl, se, r2))

    # Plot range
    xmax = min(np.nanpercentile(x[x > 0], 99) if (x > 0).any() else 30, 55)
    xmax = max(xmax, 20)
    ymax_vals = ys[np.isfinite(ys)]
    ymax = min(np.nanpercentile(ymax_vals, 97) if len(ymax_vals) else 20, 60)
    ymax = max(ymax, 12)
    xr = np.linspace(0, xmax, 300)

    # Forbidden Zone fill: between 0 and F*X
    ax.fill_between(xr, 0, F * xr, color=FZ_COL, alpha=0.85,
                    zorder=1, label='Forbidden Zone')
    ax.plot(xr, F * xr, color='#b08820', lw=1.5, ls='--',
            zorder=2, label='FZ boundary: Y = F×X')

    # Cooking points (show at raw pm25 y value)
    if ck_od.sum() > 0:
        ax.scatter(x[ck_od], yr[ck_od],
                   c=RED, marker='^', s=16, alpha=0.30,
                   zorder=3, label=f'Cooking-flagged (n={ck_od.sum():,})')

    # FZ violations
    if n_fz > 0:
        ax.scatter(x[fz], ys[fz],
                   c=GREY, marker='s', s=14, alpha=0.45,
                   zorder=4, label=f'FZ violation (n={n_fz:,})')

    # Regression points
    ax.scatter(x[reg], ys[reg],
               c=BLUE, marker='o', s=14, alpha=0.30,
               zorder=5, label=f'Regression (n={n_reg:,})')

    # Regression line (in raw indoor PM2.5 space: y = sl*x + ic + bg)
    y_fit = sl * xr + ic + bg
    ax.plot(xr, y_fit, color='#1a1a1a', lw=2.2, zorder=6,
            label=f'OLS: F={sl:.3f}, R²={r2:.3f}')

    # 1:1 line
    ax.plot([0, xmax], [0, xmax], color='#aaa', lw=0.8,
            ls=':', zorder=2, label='1:1 line')

    # FZ annotation
    fz_col = '#8B2020' if fz_pct > 20 else ('#996600' if fz_pct > 10 else '#6a6a40')
    ax.text(0.97, 0.05,
            f'FZ violations: {n_fz} ({fz_pct:.1f}%)',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=8.5, color=fz_col,
            bbox=dict(facecolor='white', edgecolor=FZ_COL,
                      boxstyle='round,pad=0.3', alpha=0.95))

    # Stats box
    ax.text(0.03, 0.97,
            f'F_pm25 = {sl:.3f} ± {se:.3f}\nR² = {r2:.3f}\n'
            f'C_bg = {bg:.3f} µg/m³\nn_reg = {n_reg:,}',
            transform=ax.transAxes, ha='left', va='top',
            fontsize=8.5, fontfamily='monospace',
            bbox=dict(facecolor='white', edgecolor='#ccc',
                      boxstyle='round,pad=0.4', alpha=0.95))

    ax.set_xlim(0, xmax); ax.set_ylim(0, ymax)
    ax.set_xlabel('Outdoor PM₂.₅ — QuantAQ PML (µg m⁻³)', fontsize=9.5)
    ax.set_ylabel('Indoor PM₂.₅ — pm25_strict (µg m⁻³)', fontsize=9.5)
    ax.set_title(loc, fontsize=11, fontweight='bold', pad=4)
    ax.grid(True, alpha=0.12, linewidth=0.4)
    ax.tick_params(labelsize=9)

    if loc == SENSORS[0]:
        ax.legend(fontsize=7.5, loc='upper left',
                  frameon=True, framealpha=0.92, edgecolor='#ccc',
                  handlelength=1.5)

if len(SENSORS) % 2 != 0:
    axes.flat[-1].set_visible(False)

fig.suptitle(
    'SI Figure S5. PM₂.₅ Infiltration Factor (F) Regression — Indoor vs Outdoor\n'
    'Yellow shading = Forbidden Zone (Wallace & Ott 2023): Y < F × X\n'
    'Points below this boundary imply negative indoor-generated PM₂.₅ '
    '— inconsistent with a constant infiltration factor',
    fontsize=10.5, fontweight='bold', y=0.995)

# Shared legend
leg = [
    mpatches.Patch(facecolor=FZ_COL, edgecolor='#b08820',
                   label='Forbidden Zone (Wallace & Ott 2023)'),
    mlines.Line2D([],[],color='#b08820',lw=1.5,ls='--',
                  label='FZ boundary: Y = F × X'),
    plt.scatter([],[],c=BLUE,marker='o',s=30,alpha=0.7,
                label='Regression points (outdoor > 9 µg/m³, Y ≥ F×X)'),
    plt.scatter([],[],c=RED,marker='^',s=35,alpha=0.7,
                label='Cooking-flagged (excluded)'),
    plt.scatter([],[],c=GREY,marker='s',s=30,alpha=0.7,
                label='Forbidden Zone violation (diagnostic only)'),
    mlines.Line2D([],[],color='#1a1a1a',lw=2.2,
                  label='OLS regression (slope = F_pm25)'),
    mlines.Line2D([],[],color='#aaa',lw=0.8,ls=':',
                  label='1:1 reference line'),
]
fig.legend(handles=leg, loc='lower center', ncol=4, fontsize=8.5,
           frameon=True, edgecolor='#ccc', framealpha=0.95,
           bbox_to_anchor=(0.5, 0.003))

plt.savefig('si_fig_regression.png', dpi=300, bbox_inches='tight', facecolor='white')

# Diagnostic summary — uses Cell 8 F values (sl in summary is re-estimated;
# use INFILTRATION_FACTORS_PM25 from Cell 8 for the canonical F)
print()
print("=== FORBIDDEN ZONE DIAGNOSTIC (Wallace & Ott 2023) ===")
print(f"{'Sensor':<28} {'n_reg':>7} {'n_FZ':>7} {'FZ%':>7}  {'F_Cell8':>9}  {'R2_fig':>7}")
print("-"*72)
for loc, n_reg, n_fz, fz_pct, sl, se, r2 in summary:
    f_c8  = INFILTRATION_FACTORS_PM25.get(loc, sl)
    flag  = "  <- HIGH" if fz_pct > 20 else ("  <- moderate" if fz_pct > 10 else "")
    print(f"  {loc:<26} {n_reg:>7,} {n_fz:>7,} {fz_pct:>7.1f}%"
          f"  {f_c8:>9.3f}  {r2:>7.3f}{flag}")

print()
print("Interpretation:")
print("  FZ% < 5%  -> constant F well-supported")
print("  FZ% 5-20% -> minor violations, F is a reasonable approximation")
print("  FZ% > 20% -> F may not be constant; consider stratification by season/window status")
print()
print("NOTE: F_Cell8 = INFILTRATION_FACTORS_PM25 from Cell 8 (canonical values).")
print("      R2_fig = R2 from regression within this figure script (for QC only).")
print("      High FZ% indicates window-to-window variability in effective infiltration.")

# CORRELATION MATRIX FOR PAPER:

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Sensors to include — exclude AG-3, AG-9, AG-6, AG-13 ─────────────────────
AG_ORDERED = [l for l in ag_by_dist
              if 'AG-9'  not in l
              and 'AG-6'  not in l
              and 'AG-3' not in l]

# ── Distance labels for y-axis ────────────────────────────────────────────────
LABEL_MAP = {
    'AG-19 (ML)':           'AG-19 ML (0.72 km)',
    'AG-15 (IB)':            'AG-15 IB (2.07 km)',
    'AG-7 (IB)':             'AG-7 IB (2.53 km)',
    'AG-17 (IB City Hall)':  'AG-17 City Hall (2.90 km)',
    'AG-11 (IB Classroom)':  'AG-11 Classroom (3.56 km)',
    'AG-12 (IB)':            'AG-12 IB (4.43 km)',
    'AG-6 (IB)':             'AG-6 IB (2.478 km)',
    'AG-13 (IB)':            'AG-13 IB (3.607 km)',
    'QuantAQ Coast':         'QuantAQ Coast (4.43 km)',
    'QuantAQ TR Estuary':   'QuantAQ TR Estuary (3.607 km)',
    'QuantAQ ML':           'QuantAQ ML (0.72 km)',
    'QuantAQ IB City Hall':  'QuantAQ City Hall (2.90 km)',
}

def short_label(loc):
    return LABEL_MAP.get(loc, loc)

# ── Sensor lists ──────────────────────────────────────────────────────────────
# Use AG_ORDERED (already excludes AG-3, AG-6, AG-9, AG-13) sorted by distance
ALL_AG_SENSORS_FOR_CORR_SORTED = sorted(
    [l for l in ag_locs
     if 'AG-9'  not in l
     and 'AG-3' not in l],
    key=dist_to_hotspot,
    reverse=False
)

OUTDOOR_QAQ_SENSORS_FOR_CORR = [
    'QuantAQ ML',
    'QuantAQ IB City Hall',
    'QuantAQ TR Estuary',
    'QuantAQ Coast',
]

# ── Compute correlation matrix ────────────────────────────────────────────────
corr_ag_vs_qaq_event_days = make_corr_matrix(
    ds_ag_filt_event,
    ds_aq_clean_event,
    ds_aq_clean_event,
    ALL_AG_SENSORS_FOR_CORR_SORTED,
    OUTDOOR_QAQ_SENSORS_FOR_CORR
)

# ── Rename index and columns with distance labels ─────────────────────────────
corr_ag_vs_qaq_event_days.index = [
    short_label(l) for l in corr_ag_vs_qaq_event_days.index
]
corr_ag_vs_qaq_event_days.columns = [
    short_label(c) for c in corr_ag_vs_qaq_event_days.columns
]

# ── Star annotations ──────────────────────────────────────────────────────────
star_annotations_ag_vs_qaq = correlation_to_stars(corr_ag_vs_qaq_event_days)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig_ag_vs_qaq, ax_ag_vs_qaq = plt.subplots(figsize=(12, 8))
sns.heatmap(
    corr_ag_vs_qaq_event_days,
    ax=ax_ag_vs_qaq,
    annot=star_annotations_ag_vs_qaq, fmt='',
    cmap='RdBu_r', vmin=-1, vmax=1,
    linewidths=0.5, linecolor='#eeeeee',
    cbar_kws={'label': 'Spearman ρ', 'shrink': 0.85},
    annot_kws={'color': 'black', 'fontsize': 16}
)
ax_ag_vs_qaq.set_xlabel('Outdoor QuantAQ Sensors', fontsize=16)
ax_ag_vs_qaq.set_ylabel('Indoor AG Sensors (Z-score normalized)', fontsize=16)
ax_ag_vs_qaq.set_title(
    'Spearman Correlation: All Indoor AG vs. Selected Outdoor QuantAQ PM₁\n'
    '(High H₂S + PM₁ Event Days, Z-score Normalized)',
    fontsize=18
)
ax_ag_vs_qaq.tick_params(axis='x', rotation=30, labelsize=11)
ax_ag_vs_qaq.tick_params(axis='y', rotation=0,  labelsize=11)
ax_ag_vs_qaq.invert_yaxis()

fig_ag_vs_qaq.tight_layout()
output_filename_ag_vs_qaq = 'correlation_all_ag_vs_qaq_event_days.png'
fig_ag_vs_qaq.savefig(output_filename_ag_vs_qaq, dpi=SAVE_DPI, bbox_inches='tight')
plt.show()
print(f"Figure saved to '{output_filename_ag_vs_qaq}'.")

corr_ag_vs_qaq_event_days.to_csv('spearman_correlation_all_ag_vs_qaq_event_days.csv')
print("Correlation matrix saved to CSV.")

In [ ]:
import numpy as np
import pandas as pd

# --- Gather and preprocess all relevant time series ---

# Initialize a dictionary to hold all preprocessed series
all_series = {}

# Meteorological data from previous cells (dc6477e6)
# Ensure all are timezone-naive and resampled to 5-min
# We'll use the existing resampled series where available for consistency

# Temperature
temp_data = ds_aq_clean['temp'].sel(location='QuantAQ ML').to_pandas()
if temp_data.index.tz is not None: temp_data.index = temp_data.index.tz_localize(None)
all_series['Temperature (°C)'] = temp_data.resample('5min').mean().ffill().bfill()

# Relative Humidity
rh_data = ds_aq_clean['rh'].sel(location='QuantAQ ML').to_pandas()
if rh_data.index.tz is not None: rh_data.index = rh_data.index.tz_localize(None)
all_series['RH (%)'] = rh_data.resample('5min').mean().ffill().bfill()

# River Stage 6hr Lag
river_stage_data = river['river_stage_lag6h'].copy()
if river_stage_data.index.tz is not None: river_stage_data.index = river_stage_data.index.tz_localize(None)
all_series['River Stage Lagged 6h (m)'] = river_stage_data.resample('5min').mean().ffill().bfill()

# MLH
mlh_data = mlh['mlh'].copy()
if mlh_data.index.tz is not None: mlh_data.index = mlh_data.index.tz_localize(None)
all_series['MLH (m)'] = mlh_data.resample('5min').mean().ffill().bfill()

# Shortwave Radiation
shortwave_data = mlh['shortwave'].copy()
if shortwave_data.index.tz is not None: shortwave_data.index = shortwave_data.index.tz_localize(None)
all_series['Shortwave Radiation (W m⁻²)'] = shortwave_data.resample('5min').mean().ffill().bfill()

# Wind Direction (using sine transform for linear correlation)
wind_dir_data = ds_aq_clean['wind_dir'].sel(location='QuantAQ ML').to_pandas()
if wind_dir_data.index.tz is not None: wind_dir_data.index = wind_dir_data.index.tz_localize(None)
wind_dir_resampled = wind_dir_data.resample('5min').mean().ffill().bfill()
all_series['Wind Direction Sine'] = np.sin(np.deg2rad(wind_dir_resampled))

# --- QuantAQ Sensors (PM1, PM2.5) ---
quantaq_sensors_to_include = [
    'QuantAQ ML',
    'QuantAQ IB City Hall',
    'QuantAQ Coast'
]

for loc in quantaq_sensors_to_include:
    # PM1
    pm1_data = ds_aq_clean['pm1'].sel(location=loc).to_pandas()
    if pm1_data.index.tz is not None: pm1_data.index = pm1_data.index.tz_localize(None)
    all_series[f'{loc} PM1 (µg m⁻³)'] = pm1_data.resample('5min').mean().ffill().bfill()

    # PM2.5
    pm25_data = ds_aq_clean['pm25'].sel(location=loc).to_pandas()
    if pm25_data.index.tz is not None: pm25_data.index = pm25_data.index.tz_localize(None)
    all_series[f'{loc} PM2.5 (µg m⁻³)'] = pm25_data.resample('5min').mean().ffill().bfill()

# --- AG Sensors (PM1, PM2.5, PM10) ---
ag_sensors_to_include = [
    'AG-19 (ML)', 'AG-15 (IB)', 'AG-7 (IB)', 'AG-17 (IB City Hall)',
    'AG-11 (IB Classroom)', 'AG-12 (IB)', 'AG-9 (SY)', 'AG-3 (SY)'
]

for loc in ag_sensors_to_include:
    # PM1
    pm1_data = ds_ag_filt['pm1_filtered'].sel(location=loc).to_pandas()
    if pm1_data.index.tz is not None: pm1_data.index = pm1_data.index.tz_localize(None)
    all_series[f'{loc} PM1 (µg m⁻³)'] = pm1_data.resample('5min').mean().ffill().bfill()

    # PM2.5
    # Using ds_ag_filt_pm25 which contains pm25_filtered
    pm25_data = ds_ag_filt_pm25['pm25_filtered'].sel(location=loc).to_pandas()
    if pm25_data.index.tz is not None: pm25_data.index = pm25_data.index.tz_localize(None)
    all_series[f'{loc} PM2.5 (µg m⁻³)'] = pm25_data.resample('5min').mean().ffill().bfill()

    # PM10
    # Using the raw pm10 data from ds_ag_clean since pm10_filtered is not defined
    pm10_data = ds_ag_clean['pm10'].sel(location=loc).to_pandas()
    if pm10_data.index.tz is not None: pm10_data.index = pm10_data.index.tz_localize(None)
    all_series[f'{loc} PM10 (µg m⁻³)'] = pm10_data.resample('5min').mean().ffill().bfill()


# Combine all preprocessed series into a single DataFrame
correlation_df_expanded = pd.DataFrame(all_series)

# Drop any rows with NaN values that might result from alignment or missing data points
correlation_df_expanded = correlation_df_expanded.dropna()

# Calculate the Pearson correlation matrix
spearman_corr_matrix_expanded = correlation_df_expanded.corr(method='pearson')

# Display the correlation matrix
print("Pearson Correlation Matrix for all requested variables:")
display(spearman_corr_matrix_expanded)

# Save the correlation matrix to a CSV file
output_filename_expanded = 'all_variables_pearson_correlation.csv'
spearman_corr_matrix_expanded.to_csv(output_filename_expanded)

print(f"\nExpanded Pearson correlation matrix saved to '{output_filename_expanded}'")

In [ ]:
def to_local_time(df_or_series):
    """Converts a DataFrame or Series index to timezone-naive PDT."""
    local_df = df_or_series.copy()
    if isinstance(local_df.index, pd.DatetimeIndex) and local_df.index.tz is None:
        # Assume UTC and convert to PDT (UTC-7)
        local_df.index = local_df.index - pd.Timedelta(hours=7)
    elif isinstance(local_df.index, pd.DatetimeIndex) and local_df.index.tz is not None:
        # Convert to PDT and then remove timezone info
        local_df.index = local_df.index.tz_convert('America/Los_Angeles').tz_localize(None)
    local_df.index.name = 'Time (PDT)'
    return local_df

print("Time conversion helper function defined.")

#### 1. AirGradient (AG) Data Spreadsheet

This spreadsheet will contain PM1, PM2.5, and PM10 data from AirGradient sensors, including raw, filtered, and strict filtered versions where available. All timestamps will be in local PDT.

In [ ]:
# Initialize an empty dictionary to store all AG series
ag_data_series = {}

# Loop through each AG sensor location
for loc in ds_ag_clean.location.values:
    # Raw PM data from ds_ag_clean
    ag_data_series[f'{loc}_PM1_Raw'] = ds_ag_clean['pm1'].sel(location=loc).to_pandas()
    ag_data_series[f'{loc}_PM25_Raw'] = ds_ag_clean['pm25'].sel(location=loc).to_pandas()
    ag_data_series[f'{loc}_PM10_Raw'] = ds_ag_clean['pm10'].sel(location=loc).to_pandas()

    # Filtered PM1 data from ds_ag_filt
    ag_data_series[f'{loc}_PM1_Filtered'] = ds_ag_filt['pm1_filtered'].sel(location=loc).to_pandas()
    ag_data_series[f'{loc}_PM1_Strict'] = ds_ag_filt['pm1_strict'].sel(location=loc).to_pandas()

    # Filtered PM2.5 data from ds_ag_filt_pm25
    ag_data_series[f'{loc}_PM25_Filtered'] = ds_ag_filt_pm25['pm25_filtered'].sel(location=loc).to_pandas()
    ag_data_series[f'{loc}_PM25_Strict'] = ds_ag_filt_pm25['pm25_strict'].sel(location=loc).to_pandas()

# Combine all series into a single DataFrame
ag_combined_df = pd.DataFrame(ag_data_series)

# Convert to local time
ag_combined_df_local = to_local_time(ag_combined_df)

# Save to CSV
ag_output_filename = 'ag_data_cleaned_time_corrected.csv'
ag_combined_df_local.to_csv(ag_output_filename)

print(f"AirGradient data saved to '{ag_output_filename}'.")
display(ag_combined_df_local.head())

#### 2. QuantAQ (AQ) Data Spreadsheet

This spreadsheet will contain PM1 and PM2.5 data from QuantAQ sensors in local PDT.

In [ ]:
# Initialize an empty dictionary to store all QuantAQ series
aq_data_series = {}

# Loop through each QuantAQ sensor location
for loc in ds_aq_clean.location.values:
    aq_data_series[f'{loc}_PM1'] = ds_aq_clean['pm1'].sel(location=loc).to_pandas()
    aq_data_series[f'{loc}_PM25'] = ds_aq_clean['pm25'].sel(location=loc).to_pandas()

# Combine all series into a single DataFrame
aq_combined_df = pd.DataFrame(aq_data_series)

# Convert to local time
aq_combined_df_local = to_local_time(aq_combined_df)

# Save to CSV
aq_output_filename = 'quantaq_data_cleaned_time_corrected.csv'
aq_combined_df_local.to_csv(aq_output_filename)

print(f"QuantAQ data saved to '{aq_output_filename}'.")
display(aq_combined_df_local.head())

#### 4. Atmospheric Parameters Spreadsheet

This spreadsheet will contain atmospheric parameters such as MLH, shortwave radiation, and river stage in local PDT.

In [ ]:
# MLH and river data are already converted to PDT in their respective load functions (Cell 4/5)
# Ensure they are timezone-naive consistently for export
mlh_local = to_local_time(mlh)
river_local = to_local_time(river)

# Atmospheric parameters — mlh and river already converted to PDT in their load functions
atmospheric_combined_df = pd.concat([mlh, river], axis=1).dropna(how='all')
atmospheric_combined_df.index.name = 'Time (PDT)'
atmospheric_combined_df.to_csv('atmospheric_parameters_time_corrected.csv')

display(atmospheric_combined_df.head())

In [ ]:
# ── FIGURE: Toxicity Potential using raw pm25_strict ─────────────────────────
# Tp = mean daily indoor PM2.5 (strict filtered) / WHO 15 µg/m³
# No decomposition — uses observed indoor PM2.5 directly

from scipy import stats as scipy_stats

WHO_PM25_TP = 15.0

# Sensors to include — same order as original figure
SENSOR_ORDER_TP = [l for l in ag_by_dist
                   if 'AG-6' not in l
                   and 'AG-9' not in l
                   and 'AG-3' not in l]

EVENT_CATS_TP = ['Clean', 'High H₂S + PM₁', 'All']
EVENT_COLS_TP  = {
    'Clean':          'steelblue',
    'High H₂S + PM₁': 'darkviolet',
    'All':            '#e06c00',
}
EVENT_MARKS_TP = {
    'Clean':           '^',
    'High H₂S + PM₁': 's',
    'All':             'o',
}
OFFSETS_TP = {
    'Clean':           -0.18,
    'High H₂S + PM₁':  0.18,
    'All':              0.0,
}

# ── Build pm25_strict time series per sensor ──────────────────────────────────
tp_work_records = []
for loc in SENSOR_ORDER_TP:
    if loc not in ds_ag_filt_pm25.location.values:
        continue
    pm25s = ds_ag_filt_pm25['pm25_filtered'].sel(location=loc).to_pandas()
    pm25s.index = pd.DatetimeIndex(pm25s.index).tz_localize(None)
    df_loc = pd.DataFrame({
        'dt':     pm25s.index,
        'sensor': loc,
        'pm25_filtered': pm25s.values,
    })
    df_loc['date'] = pd.to_datetime(df_loc['dt']).dt.date.astype(str)
    df_loc['event_type'] = df_loc['date'].map(lambda d: get_day_event(d))
    tp_work_records.append(df_loc)

tp_work = pd.concat(tp_work_records, ignore_index=True).dropna(subset=['pm25_filtered'])

# ── Daily mean indoor PM2.5 → Tp ─────────────────────────────────────────────
daily_indoor = (
    tp_work
    .groupby(['sensor', 'date', 'event_type'])['pm25_filtered']
    .mean()
    .reset_index()
)
daily_indoor['tp_daily'] = daily_indoor['pm25_filtered'] / WHO_PM25_TP

# ── Mean and SD of daily Tp across days, by sensor and event type ─────────────
tp_event = (
    daily_indoor[daily_indoor['event_type'].isin(['Clean', 'High H₂S + PM₁'])]
    .groupby(['sensor', 'event_type'])['tp_daily']
    .agg(['mean', 'std', 'count'])
    .reset_index()
)
tp_event.columns = ['sensor', 'event_type', 'tp_mean', 'tp_std', 'n_days']

# All days
tp_all = (
    daily_indoor
    .groupby(['sensor'])['tp_daily']
    .agg(['mean', 'std', 'count'])
    .reset_index()
)
tp_all.columns = ['sensor', 'tp_mean', 'tp_std', 'n_days']
tp_all['event_type'] = 'All'

tp_plot_data = pd.concat([tp_event, tp_all], ignore_index=True)
tp_plot_data['tp_std'] = tp_plot_data['tp_std'].fillna(0)
tp_plot_data = tp_plot_data[tp_plot_data['sensor'].isin(SENSOR_ORDER_TP)]

# Print summary
print("Daily mean Tp (pm25_strict / 15) by sensor and event type:")
pivot = tp_plot_data.pivot_table(index='sensor', columns='event_type', values='tp_mean')
available = [e for e in EVENT_CATS_TP if e in pivot.columns]
display(pivot[available].round(4))

# ── Build figure ──────────────────────────────────────────────────────────────
fig_tp, ax_tp = plt.subplots(figsize=(11, 6))
fig_tp.subplots_adjust(left=0.22, right=0.96, top=0.88, bottom=0.14)

y_pos = np.arange(len(SENSOR_ORDER_TP))

for et in EVENT_CATS_TP:
    col  = EVENT_COLS_TP[et]
    mark = EVENT_MARKS_TP[et]
    off  = OFFSETS_TP[et]

    for yi, sensor in enumerate(SENSOR_ORDER_TP):
        row = tp_plot_data[
            (tp_plot_data['sensor']     == sensor) &
            (tp_plot_data['event_type'] == et)
        ]
        if len(row) == 0:
            continue

        m = float(row['tp_mean'].values[0])
        s = float(row['tp_std'].values[0])
        y = y_pos[yi] + off

        ax_tp.plot([0, m], [y, y], color=col, lw=1.5, alpha=0.7, zorder=2)
        ax_tp.plot([max(0, m - s), m + s], [y, y],
                   color=col, lw=3.0, alpha=0.35,
                   solid_capstyle='round', zorder=3)
        ax_tp.scatter(m, y, color=col, s=65, marker=mark,
                      zorder=5, edgecolors='white', lw=0.7)

for yi in y_pos:
    ax_tp.axhline(yi, color='#eeeeee', lw=0.6, zorder=0)

ax_tp.set_yticks(y_pos)
ax_tp.set_yticklabels(SENSOR_ORDER_TP, fontsize=14)
ax_tp.set_ylim(-0.6, len(SENSOR_ORDER_TP) - 0.4)
ax_tp.set_ylabel('Sensor Location', fontsize=16)
ax_tp.set_xlabel('Toxicity Potential  (indoor PM₂.₅ / 15 µg m⁻³)', fontsize=16)
ax_tp.set_title(
    'Daily Mean Indoor PM₂.₅ Toxicity Potential by Event Type\n'
    '(pm25_strict — cooking-filtered observed indoor PM₂.₅)',
    fontsize=15
)

legend_labels = {
    'Clean':            'Clean',
    'High H₂S + PM₁':  'High H₂S + PM₁',
    'All':              'Entire Study Period',
}
leg_handles = [
    plt.scatter([], [], color=EVENT_COLS_TP[et], s=65,
                marker=EVENT_MARKS_TP[et], edgecolors='white',
                lw=0.5, label=legend_labels[et])
    for et in EVENT_CATS_TP
]
leg_handles.append(mpatches.Patch(
    facecolor='#888888', alpha=0.35,
    label='±1 SD across days in category'))
ax_tp.legend(handles=leg_handles, fontsize=14,
             loc='center right', frameon=True,
             framealpha=0.9, edgecolor='#ccc')

fig_tp.savefig('figure_tp_daily_pm25strict.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()
print("Tp figure (pm25_strict) saved.")

In [ ]:
# ── STATISTICAL TESTS: Mann-Whitney U and Welch's t-test ─────────────────────
# Replaces cells 78–82. All tests use:
#   • pm1_filtered  (consistent with cell 44 Mann-Whitney U and Figure 4 boxplots)
#   • PDT hours via (hour - 7) % 24  (consistent with cell 26 is_daytime definition)
#   • ag_by_dist_f4b sensor list  (excludes AG-9, AG-3, AG-6)
#   • event labels from get_day_event()  (consistent with all other cells)
#
# Three tests are run:
#   1. Mann-Whitney U (5-min observations, no day/night split)   — nonparametric
#   2. Welch's t-test on daily MEAN PM1 (daytime / nighttime)    — parametric
#   3. Welch's t-test on daily MAX PM1  (daytime / nighttime)    — parametric
#      Note: test 3 reuses bp_df_daily_max already built in cell 28

from scipy import stats as scipy_stats
import pandas as pd
import numpy as np

STAT_SENSORS = ag_by_dist_f4b   # consistent sensor list throughout

# ── Build aligned indoor PM1 DataFrame (pm1_filtered, PDT-indexed) ────────────
indoor_filt = pd.DataFrame({
    loc: ds_ag_filt['pm1_filtered'].sel(location=loc).to_pandas()
    for loc in ds_ag_filt.location.values
    if loc in STAT_SENSORS
})
indoor_filt.index = pd.DatetimeIndex(indoor_filt.index).tz_localize(None)

# Derive PDT hours and event labels directly from the index
# (index is already in PDT — build_ag_dataset subtracts 7h from UTC)
pdt_hours    = (indoor_filt.index.hour - 7) % 24   # matches cell 26 is_daytime logic
dates_str    = pd.Series(indoor_filt.index).dt.date.astype(str).values
event_labels = np.array([get_day_event(d) for d in dates_str])

daytime_mask  = (pdt_hours >= 6) & (pdt_hours < 20)
nighttime_mask = ~daytime_mask

# ── Helper: significance stars ────────────────────────────────────────────────
def sig_stars(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'

# ═══════════════════════════════════════════════════════════════════════════════
# TEST 1 — Mann-Whitney U (5-min observations, one-sided: event > clean)
# Matches cell 44 approach exactly; extended to day/night split
# ═══════════════════════════════════════════════════════════════════════════════
print("=" * 80)
print("TEST 1 — Mann-Whitney U: 5-min pm1_filtered  |  event > clean (one-sided)")
print("=" * 80)

mw_records = []
for period_label, mask in [('Full day',  np.ones(len(indoor_filt), dtype=bool)),
                             ('Daytime',   daytime_mask),
                             ('Nighttime', nighttime_mask)]:
    print(f"\n  {period_label}")
    print(f"  {'Sensor':<30} {'n_clean':>8} {'n_event':>8} "
          f"{'med_clean':>10} {'med_event':>10} {'U-stat':>10} {'p-value':>12} {'sig':>5}")
    print(f"  {'-'*90}")

    for loc in STAT_SENSORS:
        clean_vals = indoor_filt.loc[
            (event_labels == 'Clean') & mask, loc].dropna()
        event_vals = indoor_filt.loc[
            (event_labels == 'High H₂S + PM₁') & mask, loc].dropna()
        if len(clean_vals) < 10 or len(event_vals) < 10:
            continue
        u_stat, p_val = scipy_stats.mannwhitneyu(
            event_vals, clean_vals, alternative='greater')
        mw_records.append({'Sensor': loc, 'Period': period_label,
                            'n_clean': len(clean_vals), 'n_event': len(event_vals),
                            'med_clean': clean_vals.median(), 'med_event': event_vals.median(),
                            'U': u_stat, 'p': p_val, 'sig': sig_stars(p_val)})
        print(f"  {loc:<30} {len(clean_vals):>8} {len(event_vals):>8} "
              f"{clean_vals.median():>10.3f} {event_vals.median():>10.3f} "
              f"{u_stat:>10.0f} {p_val:>12.3e} {sig_stars(p_val):>5}")

mw_df = pd.DataFrame(mw_records)

# ═══════════════════════════════════════════════════════════════════════════════
# TEST 2 — Welch's t-test on daily MEAN PM1 (daytime / nighttime)
# Uses one daily mean per sensor per day — reduces autocorrelation vs 5-min obs
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("TEST 2 — Welch's t-test: daily MEAN pm1_filtered  |  event vs clean")
print("=" * 80)

# Build daily means with correct PDT day boundaries
daily_mean_records = []
for loc in STAT_SENSORS:
    ser = indoor_filt[loc].copy()
    for date_val in pd.date_range(
            pd.Timestamp(PLOT_START), pd.Timestamp(PLOT_END), freq='D'):
        date_str  = str(date_val.date())
        etype     = get_day_event(date_str)
        if etype is None: continue
        # Slice by calendar date (index is already PDT)
        day_data = ser[ser.index.date == date_val.date()]
        if day_data.empty: continue
        day_pdt_hours = (day_data.index.hour - 7) % 24
        for period, pmask in [('Daytime',  (day_pdt_hours >= 6) & (day_pdt_hours < 20)),
                               ('Nighttime', ~((day_pdt_hours >= 6) & (day_pdt_hours < 20)))]:
            vals = day_data[pmask].dropna()
            if len(vals) == 0: continue
            daily_mean_records.append({
                'sensor': loc, 'pm1': vals.mean(),
                'event_type': etype, 'period': period, 'date': date_str
            })

dm_df = pd.DataFrame(daily_mean_records)

t_records_mean = []
for period in ['Daytime', 'Nighttime']:
    print(f"\n  {period}")
    print(f"  {'Sensor':<30} {'n_clean':>7} {'n_event':>7} "
          f"{'mean_clean':>11} {'mean_event':>11} {'t-stat':>8} {'p-value':>12} {'sig':>5}")
    print(f"  {'-'*85}")
    sub = dm_df[dm_df['period'] == period]
    for loc in STAT_SENSORS:
        clean_d = sub[(sub['sensor']==loc) & (sub['event_type']=='Clean')]['pm1'].dropna()
        event_d = sub[(sub['sensor']==loc) & (sub['event_type']=='High H₂S + PM₁')]['pm1'].dropna()
        if len(clean_d) < 2 or len(event_d) < 2: continue
        t_stat, p_val = scipy_stats.ttest_ind(event_d, clean_d, equal_var=False)
        t_records_mean.append({'Sensor': loc, 'Period': period,
                                'n_clean': len(clean_d), 'n_event': len(event_d),
                                'mean_clean': clean_d.mean(), 'mean_event': event_d.mean(),
                                't': t_stat, 'p': p_val, 'sig': sig_stars(p_val)})
        print(f"  {loc:<30} {len(clean_d):>7} {len(event_d):>7} "
              f"{clean_d.mean():>11.3f} {event_d.mean():>11.3f} "
              f"{t_stat:>8.3f} {p_val:>12.3e} {sig_stars(p_val):>5}")

t_mean_df = pd.DataFrame(t_records_mean)

# ═══════════════════════════════════════════════════════════════════════════════
# TEST 3 — Welch's t-test on daily MAX PM1 (reuses bp_df_daily_max from cell 28)
# bp_df_daily_max already uses pm1_strict and correct is_daytime — no rebuild needed
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("TEST 3 — Welch's t-test: daily MAX pm1_strict (from bp_df_daily_max)")
print("         Reuses cell 28 data — is_daytime already PDT-correct")
print("=" * 80)

test_df_max = bp_df_daily_max[
    bp_df_daily_max['event_type'].isin(['Clean', 'High H₂S + PM₁'])].copy()

t_records_max = []
for period_label, tlabel in [('Daytime', 1), ('Nighttime', 0)]:
    print(f"\n  {period_label}")
    print(f"  {'Sensor':<30} {'n_clean':>7} {'n_event':>7} "
          f"{'mean_clean':>11} {'mean_event':>11} {'t-stat':>8} {'p-value':>12} {'sig':>5}")
    print(f"  {'-'*85}")
    for loc in STAT_SENSORS:
        clean_d = test_df_max[
            (test_df_max['sensor']==loc) &
            (test_df_max['event_type']=='Clean') &
            (test_df_max['is_daytime']==tlabel)]['pm1'].dropna()
        event_d = test_df_max[
            (test_df_max['sensor']==loc) &
            (test_df_max['event_type']=='High H₂S + PM₁') &
            (test_df_max['is_daytime']==tlabel)]['pm1'].dropna()
        if len(clean_d) < 2 or len(event_d) < 2: continue
        t_stat, p_val = scipy_stats.ttest_ind(event_d, clean_d, equal_var=False)
        t_records_max.append({'Sensor': loc, 'Period': period_label,
                               'n_clean': len(clean_d), 'n_event': len(event_d),
                               'mean_clean': clean_d.mean(), 'mean_event': event_d.mean(),
                               't': t_stat, 'p': p_val, 'sig': sig_stars(p_val)})
        print(f"  {loc:<30} {len(clean_d):>7} {len(event_d):>7} "
              f"{clean_d.mean():>11.3f} {event_d.mean():>11.3f} "
              f"{t_stat:>8.3f} {p_val:>12.3e} {sig_stars(p_val):>5}")

t_max_df = pd.DataFrame(t_records_max)

# ═══════════════════════════════════════════════════════════════════════════════
# TEST 4 — Mann-Whitney U on daily MAX PM1 (reuses bp_df_daily_max from cell 28)
# One-sided: event > clean  |  pm1_strict via bp_df_daily_max
# More appropriate than Welch's t-test for right-skewed daily max values
# with small sample sizes (n_clean=7, n_event=9)
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("TEST 4 — Mann-Whitney U: daily MAX pm1_strict  |  event > clean (one-sided)")
print("         Reuses bp_df_daily_max from cell 28 — is_daytime already PDT-correct")
print("=" * 80)

mw_max_records = []
for period_label, tlabel in [('Daytime', 1), ('Nighttime', 0)]:
    print(f"\n  {period_label}")
    print(f"  {'Sensor':<30} {'n_clean':>7} {'n_event':>7} "
          f"{'med_clean':>11} {'med_event':>11} {'U-stat':>8} {'p-value':>12} {'sig':>5}")
    print(f"  {'-'*85}")
    for loc in STAT_SENSORS:
        clean_d = test_df_max[
            (test_df_max['sensor']==loc) &
            (test_df_max['event_type']=='Clean') &
            (test_df_max['is_daytime']==tlabel)]['pm1'].dropna()
        event_d = test_df_max[
            (test_df_max['sensor']==loc) &
            (test_df_max['event_type']=='High H₂S + PM₁') &
            (test_df_max['is_daytime']==tlabel)]['pm1'].dropna()
        if len(clean_d) < 3 or len(event_d) < 3: continue
        u_stat, p_val = scipy_stats.mannwhitneyu(
            event_d, clean_d, alternative='greater')
        mw_max_records.append({
            'Sensor': loc, 'Period': period_label,
            'n_clean': len(clean_d), 'n_event': len(event_d),
            'med_clean': clean_d.median(), 'med_event': event_d.median(),
            'U': u_stat, 'p': p_val, 'sig': sig_stars(p_val)
        })
        print(f"  {loc:<30} {len(clean_d):>7} {len(event_d):>7} "
              f"{clean_d.median():>11.3f} {event_d.median():>11.3f} "
              f"{u_stat:>8.0f} {p_val:>12.3e} {sig_stars(p_val):>5}")

mw_max_df = pd.DataFrame(mw_max_records)

print(f"\n  mw_max_df: {len(mw_max_df)} rows (Mann-Whitney U, daily max, 2 periods × sensors)")
print("\nAll tests complete. Summary DataFrames: mw_df, t_mean_df, t_max_df, mw_max_df")

# ── Summary DataFrames ────────────────────────────────────────────────────────
print("\n\nSummary DataFrames available: mw_df, t_mean_df, t_max_df")
print(f"  mw_df:      {len(mw_df)} rows  (Mann-Whitney, 3 periods × sensors)")
print(f"  t_mean_df:  {len(t_mean_df)} rows  (Welch's t, daily mean, 2 periods × sensors)")
print(f"  t_max_df:   {len(t_max_df)} rows  (Welch's t, daily max,  2 periods × sensors)")

In [ ]:
# ── MET DATA COMPLETENESS CHECK + MANN-WHITNEY U (Clean vs. High H₂S + PM₁) ──
# Slots in after Cell 34 (met_dfs). Reuses CLEAN_DAYS_LIST (Cell 15, the
# authoritative full-study Clean-day list from classify_day()) and matches
# the sig_stars / print-table style used in Cell 78's stat tests.

from scipy import stats as scipy_stats

def sig_stars(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'

# ── NEW: restrict met_dfs to the actual study period before anything else ────
# met_dfs (Cell 34) was built from each variable's own raw date range, which
# can extend beyond PLOT_START/PLOT_END — filter here rather than editing Cell 34.
study_start = pd.Timestamp(PLOT_START)
study_end   = pd.Timestamp(PLOT_END)

met_dfs_bounded = {
    var: df[(pd.to_datetime(df['date']) >= study_start) & (pd.to_datetime(df['date']) <= study_end)].copy()
    for var, df in met_dfs.items()
}

# ── Part 1: data completeness — which classified Clean days does each
#    variable actually have, and which are missing? ──────────────────────
print("=" * 80)
print("MET DATA COMPLETENESS — Clean days per variable vs. full classification")
print("=" * 80)

all_classified_clean = set(CLEAN_DAYS_LIST)

for var_name, df in met_dfs_bounded.items():
    var_clean_dates = set(df[df['event_type'] == 'Clean']['date'].unique())
    missing = sorted(all_classified_clean - var_clean_dates)
    print(f"\n{var_name}: {len(var_clean_dates)}/{len(all_classified_clean)} Clean days present")
    if missing:
        print(f"  Missing: {missing}")

# ── Part 2: Mann-Whitney U — Clean vs. High H₂S+PM₁, per variable, per period ──
print("\n" + "=" * 80)
print("MANN-WHITNEY U — Meteorological variables, Clean vs. High H₂S+PM₁ (event < clean)")
print("=" * 80)

met_mw_records = []
for var_name, df in met_dfs_bounded.items():
    print(f"\n  {var_name}")
    print(f"  {'Period':<10} {'n_clean':>7} {'n_event':>7} "
          f"{'med_clean':>11} {'med_event':>11} {'U-stat':>8} {'p-value':>12} {'r':>7} {'sig':>5}")
    print(f"  {'-'*85}")

    for period_label, tlabel in [('Daytime', 1), ('Nighttime', 0)]:
        sub = df[df['is_daytime'] == tlabel]
        clean_vals = sub[sub['event_type'] == 'Clean']['value'].dropna()
        event_vals = sub[sub['event_type'] == 'High H₂S + PM₁']['value'].dropna()

        if len(clean_vals) < 3 or len(event_vals) < 3:
            continue

        u_stat, p_val = scipy_stats.mannwhitneyu(event_vals, clean_vals, alternative='less')
        n1, n2 = len(event_vals), len(clean_vals)
        effect_r = 1 - (2 * u_stat) / (n1 * n2)

        met_mw_records.append({
            'Variable': var_name, 'Period': period_label,
            'n_clean': n2, 'n_event': n1,
            'med_clean': clean_vals.median(), 'med_event': event_vals.median(),
            'U': u_stat, 'p': p_val, 'r': round(effect_r, 3), 'sig': sig_stars(p_val)
        })
        print(f"  {period_label:<10} {n2:>7} {n1:>7} "
              f"{clean_vals.median():>11.3f} {event_vals.median():>11.3f} "
              f"{u_stat:>8.0f} {p_val:>12.3e} {effect_r:>7.3f} {sig_stars(p_val):>5}")

met_mw_df = pd.DataFrame(met_mw_records)

print(f"\n  met_mw_df: {len(met_mw_df)} rows (Mann-Whitney U, meteo variables, 2 periods × 3 variables)")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── DATA PREP: Daily MAX PM2.5 per sensor, day/night split (mirrors Cell 27) ──
records_pm25_max = []

for date_val in pd.to_datetime(ds_ag_filt_pm25.time.values).normalize().unique():
    date_str = str(date_val.date())
    event_type = get_day_event(date_str)
    if event_type is None:
        continue

    for loc in ds_ag_filt_pm25.location.values:
        pm25_data_for_day = ds_ag_filt_pm25['pm25_strict'].sel(
            location=loc,
            time=slice(date_val, date_val + pd.Timedelta(days=1) - pd.Timedelta(minutes=5))
        ).to_pandas()

        if pm25_data_for_day.empty or pm25_data_for_day.isnull().all():
            continue

        daytime_vals   = pm25_data_for_day[(pm25_data_for_day.index.hour >= 6) & (pm25_data_for_day.index.hour < 20)]
        nighttime_vals = pm25_data_for_day[~((pm25_data_for_day.index.hour >= 6) & (pm25_data_for_day.index.hour < 20))]

        daytime_max   = daytime_vals.max()
        nighttime_max = nighttime_vals.max()

        if pd.notna(daytime_max):
            records_pm25_max.append({'sensor': loc, 'pm25': daytime_max, 'event_type': event_type, 'is_daytime': 1, 'date': date_str})
            records_pm25_max.append({'sensor': loc, 'pm25': daytime_max, 'event_type': 'All',        'is_daytime': 1, 'date': date_str})
        if pd.notna(nighttime_max):
            records_pm25_max.append({'sensor': loc, 'pm25': nighttime_max, 'event_type': event_type, 'is_daytime': 0, 'date': date_str})
            records_pm25_max.append({'sensor': loc, 'pm25': nighttime_max, 'event_type': 'All',        'is_daytime': 0, 'date': date_str})

bp_df_pm25_max = pd.DataFrame(records_pm25_max)

# ── PLOTTING: same layout as Figure 4b ───────────────────────────────────
EVENT_ORDER_PM25 = ['Clean', 'High H₂S + PM₁', 'All']  # confirm labels match get_day_event()

fig_pm25, axes_pm25 = plt.subplots(2, 1, figsize=(14, 7), sharey=True)
fig_pm25.subplots_adjust(left=0.09, right=0.97, top=0.91, bottom=0.10, wspace=0.06)

for ai, (ax, period_label, tlabel) in enumerate(zip(
    axes_pm25, ['Daytime (06:00–19:59)', 'Nighttime (20:00–05:59)'], [1, 0]
)):
    sub = bp_df_pm25_max[bp_df_pm25_max['is_daytime'] == tlabel]
    n_events = len(EVENT_ORDER_PM25)
    n_sensors = len(ag_by_dist_f4c)
    GW = n_sensors + 1.5

    for gi, et in enumerate(EVENT_ORDER_PM25):
        x_base = gi * GW
        if gi > 0:
            ax.axvline(x_base - 0.75, color='#cccccc', lw=0.7, zorder=0)
        for si, loc in enumerate(ag_by_dist_f4c):
            x = x_base + si
            d = sub[(sub['sensor'] == loc) & (sub['event_type'] == et)]['pm25'].dropna().values
            if len(d) > 0:
                q9, q25, med, q75, q91 = np.percentile(d, [9, 25, 50, 75, 91])
                iqr = q75 - q25
                bw = 0.52
                col = SENSOR_COLORS_F4C[loc]
                rect = mpatches.Rectangle((x - bw/2, q25), bw, iqr, facecolor=col, alpha=0.25,
                                           edgecolor=col, lw=1.0, zorder=3)
                ax.add_patch(rect)
                ax.plot([x - bw*0.3, x + bw*0.3], [med, med], color=col, lw=2.0, zorder=5)
                ax.plot([x, x], [q9, q25], color=col, lw=1.0, alpha=0.8, zorder=3)
                ax.plot([x, x], [q75, q91], color=col, lw=1.0, alpha=0.8, zorder=3)
                for cy in [q9, q91]:
                    ax.plot([x - bw*0.2, x + bw*0.2], [cy, cy], color=col, lw=1.0, alpha=0.8)

    tick_x = [gi * GW + n_sensors / 2 - 0.5 for gi in range(n_events)]
    ax.set_xticks(tick_x)
    ax.set_xticklabels(EVENT_ORDER_PM25 if ai == 1 else [], fontsize=18)
    ax.set_xlim(-0.8, n_events * GW - 0.6)
    ax.set_title(period_label, fontsize=16, pad=4 if ai == 0 else -5)
    ax.grid(True, axis='y', alpha=0.2)
    ax.text(0.005, 0.97, f'({("ab")[ai]})', transform=ax.transAxes, fontsize=16, va='center')

fig_pm25.supylabel('Daily Max Indoor PM₂.₅ (µg m⁻³)', fontsize=18, x=0.04)

sensor_handles = [mpatches.Patch(color=SENSOR_COLORS_F4C[l], alpha=0.7, label=short_label(l)) for l in ag_by_dist_f4c]
legend = fig_pm25.legend(handles=sensor_handles, fontsize=16, ncol=1, loc='center left',
                          bbox_to_anchor=(1, 0.5), frameon=True, framealpha=0.9, edgecolor='#ccc', title='AG Sensors')
legend.get_title().set_fontsize(18)

fig_pm25.savefig('figure_pm25_boxplots_daily_max.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()

# ── STATISTICAL TEST: Clean vs. High H2S+PM1, per sensor, per period ─────
print("PM2.5 daily max — Clean vs. High H₂S+PM₁ comparison\n")
print(f"{'Sensor':<25}{'Period':<10}{'t-test p':>12}{'Mann-Whitney p':>18}")

for loc in ag_by_dist_f4c:
    for period_label, tlabel in zip(['Daytime', 'Nighttime'], [1, 0]):
        sub = bp_df_pm25_max[bp_df_pm25_max['is_daytime'] == tlabel]
        clean_vals = sub[(sub['sensor'] == loc) & (sub['event_type'] == 'Clean')]['pm25'].dropna().values
        high_vals  = sub[(sub['sensor'] == loc) & (sub['event_type'] == 'High H₂S + PM₁')]['pm25'].dropna().values

        if len(clean_vals) < 3 or len(high_vals) < 3:
            print(f"{short_label(loc):<25}{period_label:<10}{'insufficient data':>30}")
            continue

        t_p = stats.ttest_ind(clean_vals, high_vals, equal_var=False).pvalue
        u_p = stats.mannwhitneyu(clean_vals, high_vals, alternative='two-sided').pvalue

        sig_t = '*' if t_p < 0.05 else ''
        sig_u = '*' if u_p < 0.05 else ''
        print(f"{short_label(loc):<25}{period_label:<10}{t_p:>11.4f}{sig_t}{u_p:>16.4f}{sig_u}")

# New F and I/O

In [ ]:
# ── INFILTRATION FACTOR (PM2.5) — CO-LOCATED OUTDOOR, HOURLY, NO THRESHOLD ───
# Final methodology: hourly resolution, no outdoor-concentration gate (event-
# day classification handles that role instead), each sensor regressed
# against its OWN co-located outdoor QuantAQ station. Restricted to the 4
# sensors that have a true co-located outdoor pairing.

COLOCATED_MAP = {
    'AG-12 (IB)':           'QuantAQ Coast',
    'AG-13 (TR Estuary)':   'QuantAQ TR Estuary',
    'AG-17 (IB City Hall)': 'QuantAQ IB City Hall',
    'AG-19 (ML)':           'QuantAQ ML',   # AG-19 IS co-located with the trailer
}

out_hourly_by_loc = {}
for aq_loc in set(COLOCATED_MAP.values()):
    s = ds_aq_clean['pm25'].sel(location=aq_loc).to_pandas()
    s.index = pd.DatetimeIndex(s.index).tz_localize(None)
    out_hourly_by_loc[aq_loc] = s.resample('h').mean()

INFILTRATION_FACTORS_COLOCATED_PM25 = {}
print("\nComputing infiltration factors (PM2.5, co-located outdoor, hourly, no threshold):")
print(f"{'Sensor':<20} {'Outdoor ref':<22} {'F':>7} {'SE':>7} {'R²':>7} {'n':>6}")
print('-'*70)
for ag_loc, aq_loc in COLOCATED_MAP.items():
    bg = SENSOR_BACKGROUND_PM25[ag_loc]

    ind = ds_ag_filt_pm25['pm25_strict'].sel(location=ag_loc).to_pandas()
    ind.index = pd.DatetimeIndex(ind.index).tz_localize(None)
    ind_h = ind.resample('h').mean()

    out_h = out_hourly_by_loc[aq_loc]

    df = pd.concat([ind_h, out_h], axis=1, keys=['indoor', 'outdoor']).dropna()
    # NOTE: no OUT_PM25_THRESH gate — removed per the no-threshold methodology;
    # event-day classification (Clean/H2S/PM1/both) is what stratifies the
    # data instead, matching Wallace & Ott (2023).

    y = (df['indoor'] - bg).values
    x = df['outdoor'].values
    valid = np.isfinite(x) & np.isfinite(y) & (y >= 0)

    if valid.sum() < 30:
        INFILTRATION_FACTORS_COLOCATED_PM25[ag_loc] = 0.0
        print(f"  {ag_loc:<20} {aq_loc:<22}: insufficient data (n={valid.sum()})")
        continue

    slope, _, r, _, se = stats.linregress(x[valid], y[valid])
    INFILTRATION_FACTORS_COLOCATED_PM25[ag_loc] = max(0.0, round(float(slope), 3))
    print(f"  {ag_loc:<20} {aq_loc:<22} {slope:>7.3f} {se:>7.3f} {r**2:>7.3f} {valid.sum():>6}")

In [ ]:
# ── SELF-CONTAINED: I/O RATIO + CUMULATIVE EXPOSURE + TABLE 1 (PM2.5 only) ──
# Requires: ds_ag_filt_pm25, ds_aq_clean, get_day_event, CLEAN_DAYS_LIST
# already defined (Cells 9/10/12/15). Everything below is computed fresh.

from scipy import stats as scipy_stats

WHO_PM25       = 15.0
CLEAN_DAYS_BG  = CLEAN_DAYS_LIST
DT_HOURS       = 5 / 60
HOME_OCCUPANCY = 0.69

COLOCATED_MAP = {
    'AG-12 (IB)':           'QuantAQ Coast',
    'AG-13 (TR Estuary)':   'QuantAQ TR Estuary',
    'AG-17 (IB City Hall)': 'QuantAQ IB City Hall',
    'AG-19 (ML)':           'QuantAQ ML',
}
EVENT_CATS = ['All', 'Clean', 'High H₂S', 'High H₂S + PM₁']

# ============================================================
# I/O RATIO — PM2.5, co-located outdoor, hourly, no threshold
# ============================================================
io_records = []
for ag_loc, aq_loc in COLOCATED_MAP.items():
    ind = ds_ag_filt_pm25['pm25_strict'].sel(location=ag_loc).to_pandas()
    ind.index = pd.DatetimeIndex(ind.index).tz_localize(None)
    ind_h = ind.resample('h').mean()

    out = ds_aq_clean['pm25'].sel(location=aq_loc).to_pandas()
    out.index = pd.DatetimeIndex(out.index).tz_localize(None)
    out_h = out.resample('h').mean()

    df = pd.concat([ind_h, out_h], axis=1, keys=['indoor', 'outdoor']).dropna()
    df = df[df['outdoor'] > 0]
    if df.empty:
        continue

    dates_h       = pd.Series(df.index).dt.date.astype(str).values
    event_labels  = np.array([get_day_event(d) for d in dates_h])
    local_hours_h = (df.index.hour - 7) % 24
    daytime_mask  = (local_hours_h >= 6) & (local_hours_h < 20)
    io_ratio      = (df['indoor'] / df['outdoor']).values

    for et in EVENT_CATS:
        emask = (event_labels == et) if et != 'All' else np.ones(len(df), dtype=bool)
        for period_name, pmask in [
            ('Full day',  np.ones(len(df), dtype=bool)),
            ('Daytime',   daytime_mask),
            ('Nighttime', ~daytime_mask),
        ]:
            vals = io_ratio[emask & pmask]
            vals = vals[np.isfinite(vals)]
            if len(vals) >= 10:
                io_records.append({
                    'sensor': ag_loc, 'event_type': et, 'period': period_name,
                    'io_mean': round(float(np.mean(vals)), 3),
                    'io_median': round(float(np.median(vals)), 3),
                    'io_std': round(float(np.std(vals)), 3),
                    'n': int(len(vals)),
                })

io_df = pd.DataFrame(io_records)   # <-- freshly rebuilt HERE, right before use below

# ============================================================
# CUMULATIVE EXPOSURE — PM2.5, real measured data
# ============================================================
indoor_pm25 = pd.DataFrame({
    loc: ds_ag_filt_pm25['pm25_strict'].sel(location=loc).to_pandas()
    for loc in ds_ag_filt_pm25.location.values
})
indoor_pm25.index = pd.DatetimeIndex(indoor_pm25.index).tz_localize(None)

dates_5m        = pd.Series(indoor_pm25.index).dt.date.astype(str).values
local_hours_5m  = (indoor_pm25.index.hour - 7) % 24
daytime_mask_5m = (local_hours_5m >= 6) & (local_hours_5m < 20)

C_background = {}
for loc in indoor_pm25.columns:
    bg_mask = np.isin(dates_5m, CLEAN_DAYS_BG)
    bg_vals = indoor_pm25.loc[bg_mask, loc].dropna()
    C_background[loc] = float(bg_vals.mean()) if len(bg_vals) > 10 else 0.0

STAT_SENSORS = [l for l in indoor_pm25.columns if 'AG-6' not in l]

exp_records = []
for loc in STAT_SENSORS:
    bg = C_background[loc]
    excess = np.clip(indoor_pm25[loc].values - bg, 0, None)
    for period_name, pmask in [
        ('Full day',  np.ones(len(indoor_pm25), dtype=bool)),
        ('Daytime',   daytime_mask_5m),
        ('Nighttime', ~daytime_mask_5m),
    ]:
        vals = excess[pmask]
        vals = vals[np.isfinite(vals)]
        if len(vals) < 10:
            continue
        n_hours  = len(vals) * DT_HOURS
        cum_full = float(np.nansum(vals)) * DT_HOURS
        exp_records.append({
            'sensor': loc, 'period': period_name,
            'cum_exp_ug_m3_h': round(cum_full, 2),
            'cum_exp_weighted': round(cum_full * HOME_OCCUPANCY, 2),
            'who_guideline_days': round(cum_full / (WHO_PM25 * 24), 3),
            'cum_normalized': round(cum_full / (WHO_PM25 * n_hours), 3),
            'n_hours': round(n_hours, 1),
            'C_background': round(bg, 3),
        })

exp_df = pd.DataFrame(exp_records)   # <-- freshly rebuilt HERE too

# ============================================================
# TABLE 1
# ============================================================
SENSOR_LABELS = {
    'AG-19 (ML)':           'R1 (0.72 km)',
    'AG-17 (IB City Hall)': 'I1 (2.9 km)',
    'AG-13 (TR Estuary)':   'I3 (3.61 km)',
    'AG-12 (IB)':           'R4 (4.43 km)',
}
ROW_ORDER = ['AG-19 (ML)', 'AG-17 (IB City Hall)', 'AG-13 (TR Estuary)', 'AG-12 (IB)']

table1_rows = []
for ag_loc in ROW_ORDER:
    io_sub = io_df[(io_df['sensor'] == ag_loc) & (io_df['period'] == 'Full day')]
    def io_val(event_type):
        m = io_sub[io_sub['event_type'] == event_type]
        return float(m['io_mean'].iloc[0]) if len(m) else float('nan')

    exp_sub = exp_df[(exp_df['sensor'] == ag_loc) & (exp_df['period'] == 'Full day')]
    unweighted = float(exp_sub['who_guideline_days'].iloc[0]) if len(exp_sub) else float('nan')
    weighted   = unweighted * HOME_OCCUPANCY

    table1_rows.append({
        'Sensor': SENSOR_LABELS[ag_loc],
        'I/O Ratio: All Days': round(io_val('All'), 3),
        'I/O Ratio: High H2S+PM1': round(io_val('High H₂S + PM₁'), 3),
        'I/O Ratio: Clean Days': round(io_val('Clean'), 3),
        'WHO Guideline-Days: Unweighted': round(unweighted, 3),
        'WHO Guideline-Days: Weighted': round(weighted, 3),
    })

table1_df = pd.DataFrame(table1_rows)
table1_df.to_csv('table1_final.csv', index=False)
display(table1_df)

In [ ]:
# ── STATS TEST: I/O RATIO — Clean vs. High H2S+PM1 ──────────────────────────
# Compares the distribution of hourly I/O ratio (PM2.5, co-located outdoor,
# no threshold) between Clean days and High H2S+PM1 days, per sensor.
# Mann-Whitney U (nonparametric, robust to skew/non-normality) is the primary
# test given how skewed these ratio distributions are; Welch's t-test is
# reported alongside for comparison, consistent with your other stats cells.

from scipy import stats as scipy_stats

COLOCATED_MAP = {
    'AG-12 (IB)':           'QuantAQ Coast',
    'AG-13 (TR Estuary)':   'QuantAQ TR Estuary',
    'AG-17 (IB City Hall)': 'QuantAQ IB City Hall',
    'AG-19 (ML)':           'QuantAQ ML',
}

io_test_records = []

for ag_loc, aq_loc in COLOCATED_MAP.items():
    ind = ds_ag_filt_pm25['pm25_strict'].sel(location=ag_loc).to_pandas()
    ind.index = pd.DatetimeIndex(ind.index).tz_localize(None)
    ind_h = ind.resample('h').mean()

    out = ds_aq_clean['pm25'].sel(location=aq_loc).to_pandas()
    out.index = pd.DatetimeIndex(out.index).tz_localize(None)
    out_h = out.resample('h').mean()

    df = pd.concat([ind_h, out_h], axis=1, keys=['indoor', 'outdoor']).dropna()
    df = df[df['outdoor'] > 0]
    if df.empty:
        continue

    dates_h      = pd.Series(df.index).dt.date.astype(str).values
    event_labels = np.array([get_day_event(d) for d in dates_h])
    io_ratio     = (df['indoor'] / df['outdoor']).values

    clean_vals = io_ratio[event_labels == 'Clean']
    high_vals  = io_ratio[event_labels == 'High H₂S + PM₁']
    clean_vals = clean_vals[np.isfinite(clean_vals)]
    high_vals  = high_vals[np.isfinite(high_vals)]

    if len(clean_vals) < 5 or len(high_vals) < 5:
        print(f"{ag_loc}: insufficient data (n_clean={len(clean_vals)}, n_high={len(high_vals)})")
        continue

    u_stat, p_mw     = scipy_stats.mannwhitneyu(clean_vals, high_vals, alternative='two-sided')
    t_stat, p_welch  = scipy_stats.ttest_ind(clean_vals, high_vals, equal_var=False)

    io_test_records.append({
        'sensor':          ag_loc,
        'n_clean':         len(clean_vals),
        'n_high_h2s_pm1':  len(high_vals),
        'median_clean':    round(float(np.median(clean_vals)), 3),
        'median_high':     round(float(np.median(high_vals)), 3),
        'mean_clean':      round(float(np.mean(clean_vals)), 3),
        'mean_high':       round(float(np.mean(high_vals)), 3),
        'mannwhitney_U':   round(float(u_stat), 1),
        'p_mannwhitney':   float(p_mw),
        'welch_t':         round(float(t_stat), 3),
        'p_welch':         float(p_welch),
        'sig_mw_0.05':     p_mw < 0.05,
    })

io_test_df = pd.DataFrame(io_test_records)
io_test_df.to_csv('io_ratio_clean_vs_high_h2s_pm1_test.csv', index=False)

print("I/O ratio: Clean vs. High H2S+PM1 (hourly, PM2.5, co-located, no threshold)")
display(io_test_df)